# Module 05 · Senescence validation

Does the senescent label from module 03 behave like senescence?

The module takes the scored object, restricts it to one cell type, scores the
Sloan Table S9 hallmark panels and Tirosh cell-cycle phases, then contrasts
**SnC vs Non-SnC** under five estimators that fail in different ways. A result
is only reported as real when the estimators agree.

| Section | Contrast | Estimators |
|---|---|---|
| 08-13 | cell-cycle phase proportion | Wilcoxon · RLM · OLS · GLMM · bootstrap · agreement |
| 14-19 | Sloan module score | Wilcoxon · RLM · OLS · LMM · bootstrap · agreement |
| 20-21 | proliferation marker expression | Wilcoxon · RLM |

Unit of analysis: **donor**. The paired arms take one value per donor per arm;
the cell-level arms carry `(1 | donor)`. Cell-level models without a random
effect are not part of this module.

The module is run once per `CELL_TYPE`. Nothing here is specific to an
activation state — state-stratified analyses live in a later module.

---
## 01 · Config

**Why.** Every run-time decision is in this one cell, so re-running the module
on a different cell type, dataset or cohort means editing here and nowhere
else. Paths come from the environment rather than being written into the
notebook — see `.env.example`.

**Set before running:** `TISSUE`, `STUDY_TYPE`, `DISEASE`, `DATASET`,
`CELL_TYPE`, and the stratification groups.

In [ ]:
# =============================================================================
# MODULE 05 - SENESCENCE VALIDATION - CONFIG
# =============================================================================
# Tests whether SnC cells show cell-cycle arrest and senescence pathway
# enrichment relative to Non-SnC. Reads the module 04 tissue export, subsets
# to one CELL_TYPE per run, scores cell cycle (Tirosh) + 10 module scores
# (Sloan Table S9), and runs five complementary models: Wilcoxon, RLM, OLS,
# LMM/GLMM, and balanced RLM bootstrap.
# =============================================================================

cat("=", strrep("=", 71), "\n", sep = "")
cat("01 - MODULE 05 CONFIG\n")
cat("=", strrep("=", 71), "\n", sep = "")

# ─────────────────────────────────────────────────────────────────────────────
# Libraries
# ─────────────────────────────────────────────────────────────────────────────
suppressPackageStartupMessages({
    library(Seurat)
    library(Matrix)
    library(dplyr)
    library(tidyr)
    library(readxl)
    library(ggplot2)
    library(patchwork)
    library(scales)
    library(qs)
    library(jsonlite)
    library(lme4)
    library(lmerTest)
    library(MASS)
    library(robustbase)
    library(broom)
    library(broom.mixed)
})

# Fix MASS::select masking dplyr::select
select <- dplyr::select


# ─────────────────────────────────────────────────────────────────────────────
# Inline plotting viewport (Jupyter / IRkernel)
# ─────────────────────────────────────────────────────────────────────────────


# ─────────────────────────────────────────────────────────────────────────────
# Run parameters — edit these
# ─────────────────────────────────────────────────────────────────────────────
TISSUE     <- "brain"
STUDY_TYPE <- "disease"
DISEASE    <- "AD"
DATASET    <- "psychad_ad"

CELL_TYPE  <- "Microglia"


# ─────────────────────────────────────────────────────────────────────────────
# Stratification
# ─────────────────────────────────────────────────────────────────────────────
STRATIFY_BY_GROUP     <- TRUE
STRATIFICATION_GROUPS <- c("Old_AD", "Old_Healthy_Control")


# ─────────────────────────────────────────────────────────────────────────────
# Statistical parameters
# ─────────────────────────────────────────────────────────────────────────────
STATISTICAL_PARAMS <- list(
    min_cells_per_group  = 5L,
    fdr_threshold        = 0.05,
    fdr_method           = "BH",
    bootstrap_n_iter     = 100L,
    bootstrap_seed       = 42L,
    seed                 = 42L,
    confidence_level     = 0.95,
    rationale            = "M09-equivalent thresholds with M05 organizational rewrite"
)

set.seed(STATISTICAL_PARAMS$seed)


# ─────────────────────────────────────────────────────────────────────────────
# Derived condition tags
# ─────────────────────────────────────────────────────────────────────────────
IS_AGING          <- (STUDY_TYPE == "aging")
IS_DISEASE        <- (STUDY_TYPE == "disease")

# -----------------------------------------------------------------------------
# Paths - from the environment, never hardcoded
# -----------------------------------------------------------------------------
#   SENESCENCE_DATA : analysis root (module outputs are written under it)
#   SENESCENCE_REF  : reference root (published marker files, read-only)
# See .env.example. Both must be set before the kernel starts.

SCRATCH <- Sys.getenv("SENESCENCE_DATA")
REF_DIR <- Sys.getenv("SENESCENCE_REF")

if (SCRATCH == "" || REF_DIR == "")
    stop("SENESCENCE_DATA and SENESCENCE_REF must be set. See .env.example.")

MARKERS <- file.path(REF_DIR, "markers")

BASE_M04   <- file.path(SCRATCH, TISSUE, "module_04T_tissue_export",
                        CONDITION_SUBPATH, DATASET)
BASE_M05   <- file.path(SCRATCH, TISSUE, "module_05_senescence_enrichment",
                        CONDITION_SUBPATH, DATASET, CELL_TYPE)

PATHS <- list(
    m04_root       = BASE_M04,
    m04_seurat     = file.path(BASE_M04, paste0(DATASET, "_tissue_seurat.qs")),
    m04_manifest   = file.path(BASE_M04, "manifest.json"),
    output_root    = BASE_M05,
    data           = file.path(BASE_M05, "data"),
    results        = file.path(BASE_M05, "results"),
    figures        = file.path(BASE_M05, "figures"),
    logs           = file.path(BASE_M05, "_logs"),
    scored_qs      = file.path(BASE_M05, "data",
                               paste0(CELL_TYPE, "_scored.qs")),
    gene_lists_rds = file.path(BASE_M05, "data", "gene_lists.rds"),
    manifest       = file.path(BASE_M05, "_logs", "m05_manifest.json"),
    markers_dir    = MARKERS,
    sloan_xlsx     = file.path(MARKERS, "1-s2.0-S2666979X25003830-mmc10.xlsx"),
    senmayo_xlsx   = file.path(MARKERS, "41467_2022_32552_MOESM4_ESM.xlsx"),
    fridman_gmt    = file.path(MARKERS, "FRIDMAN_SENESCENCE_UP.v2026.1.Hs.gmt")
)

for (key in c("data", "results", "figures", "logs")) {
    dir.create(PATHS[[key]], recursive = TRUE, showWarnings = FALSE)
}


# ─────────────────────────────────────────────────────────────────────────────
# Sloan Table S9 column mapping — 10 senescence gene lists
# Order is FIXED (used as canonical row order in §5 forest plots):
#   rows 1-8: individual hallmarks
#   rows 9-10: multi-hallmark composites
# ─────────────────────────────────────────────────────────────────────────────
SLOAN_HALLMARK_NAMES <- c(
    "p53_Targets",
    "CellCycleArrest",
    "SASP",
    "AntiApoptosis",
    "DDR",
    "CellSurfaceMarkers",
    "LysosomalContent",
    "SD_TMC",
    "SenMayo",
    "Fridman_Up"
)

SLOAN_LIST_TYPES <- c(
    rep("Individual hallmark", 7),
    rep("Multi-hallmark", 3)
)

# Color labels for module rows in §5 forests:
# hallmarks = dark gray, multi-hallmark composites = muted purple
SLOAN_HALLMARK_COLORS <- c(
    rep("#222222", 8),
    rep("#7B5BA3", 2)
)
names(SLOAN_HALLMARK_COLORS) <- SLOAN_HALLMARK_NAMES


# ─────────────────────────────────────────────────────────────────────────────
# Proliferation markers (M09 §6 → M05 §7)
# ─────────────────────────────────────────────────────────────────────────────
PROLIFERATION_MARKERS <- c(  
  # Core proliferation / mitotic markers
  "MKI67", "TOP2A", "HMGB2", "CENPF",
  "BIRC5", "CCNB1", "CCNB2", "UBE2C",

  # Canonical CDK inhibitors
  "CDKN2A",  # p16INK4A
  "CDKN1A",  # p21CIP1/WAF1
  "CDKN1B",  # p27KIP1
  "CDKN2B",  # p15INK4B

  # p53 pathway / DNA damage response
  "TP53",
  "GADD45A", "GADD45B", "GADD45G"
)

# ─────────────────────────────────────────────────────────────────────────────
# Color palettes
# ─────────────────────────────────────────────────────────────────────────────
LINEAGE_COLORS_BY_TISSUE <- list(
    brain = c(
        Excitatory      = "#0072B2",
        Inhibitory      = "#E69F00",
        Astrocyte       = "#009E73",
        Oligodendrocyte = "#56B4E9",
        Microglia       = "#D55E00",
        OPC             = "#CC79A7",
        Endothelial     = "#7F7F7F",
        Pericyte        = "#999999",
        VLMC            = "#A9A9A9",
        VSMC            = "#696969",
        PVM             = "#FF6347",
        Adaptive        = "#FFD700"
    ),
    pbmc = c(
        cd4t = "#4E79A7", cd8t = "#A0CBE8", unconvT = "#BAB0AC",
        nkcell = "#59A14F", cd14mono = "#F28E2B", cd16mono = "#FFBE7D",
        memB = "#B07AA1", naiveB = "#76B7B2", dc = "#9C755F"
    ),
    csf = c(
        cd4t = "#4E79A7", cd8t = "#A0CBE8", nkcell = "#59A14F",
        monocyte = "#F28E2B", bcell = "#B07AA1", dc = "#76B7B2"
    )
)
LINEAGE_COLORS <- LINEAGE_COLORS_BY_TISSUE[[TISSUE]]

SNC_COLORS <- c(
    Senescent       = "#C44E52",
    `Non-senescent` = "#D3D3D3",
    `TRUE`          = "#C44E52",
    `FALSE`         = "#D3D3D3",
    True            = "#C44E52",
    False           = "#D3D3D3"
)

STUDY_GROUP_COLORS <- c(
    Age_20_29 = "#2E86AB", Age_30_39 = "#4A90E2", Age_40_49 = "#50C878",
    Age_50_59 = "#FFB347", Age_60_69 = "#FF8C00", Age_70_79 = "#E24A4A",
    Age_80_100 = "#8B0000",
    Control = "#4E79A7", MCI = "#F28E2B", AD = "#E15759",
    Young_Healthy_Control = "#4A90E2",
    Old_Healthy_Control   = "#4E79A7",
    Old_AD                = "#E15759",
    All                   = "#7F7F7F"
)

PHASE_COLORS <- c(G1 = "#4E79A7", S = "#F28E2B", G2M = "#E15759")

SEX_COLORS <- c(
    Male = "#5D6D7E", Female = "#A569BD",
    M    = "#5D6D7E", F      = "#A569BD"
)

MODEL_AGREEMENT_COLORS <- c(
    `Up (sig)`     = "#C44E52",
    `Up (ns)`      = "#F4B5B5",
    `ns`           = "#D3D3D3",
    `Down (ns)`    = "#A8C5DC",
    `Down (sig)`   = "#3B6F8F"
)


# ─────────────────────────────────────────────────────────────────────────────
# EFFECT_CONFIGS — display rules for forest plots, dispatched by beta_scale
#
# Each model cell selects one of these configs to drive the inline forest
# plot. No hardcoding of axis units, label formatting, or null reference
# value across §4.x or §5.x cells.
#
#   probability_pts:  paired-difference β on the proportion scale (used by
#                     §4.x cell-cycle phase analyses).
#   score_units:      continuous module-score β (used by §5.x Sloan module
#                     score analyses, including lmer Gaussian).
#   log_odds:         log-odds β (binomial GLMM, used by §4.4 only).
#                     - log-scale x-axis showing OR
#                     - effect column shows "OR = 1.38"
#                     - null line at OR = 1
# ─────────────────────────────────────────────────────────────────────────────
EFFECT_CONFIGS <- list(
    probability_pts = list(
        scale          = "linear",
        null_value     = 0,
        x_label        = "beta (paired difference, SnC - Non-SnC)",
        eff_h_label    = "beta",
        ci_h_label     = "95% CI",
        fmt_effect     = function(v) {
            if (is.na(v)) return("--")
            sprintf("%+.3f", v)
        },
        fmt_ci         = function(lo, hi) {
            if (is.na(lo) || is.na(hi)) return("--")
            sprintf("[%+.3f, %+.3f]", lo, hi)
        },
        axis_format    = scales::label_number(accuracy = 0.01),
        plot_transform = function(v) v
    ),
    score_units = list(
        scale          = "linear",
        null_value     = 0,
        x_label        = "beta (mean module score, SnC - Non-SnC)",
        eff_h_label    = "beta",
        ci_h_label     = "95% CI",
        fmt_effect     = function(v) {
            if (is.na(v)) return("--")
            sprintf("%+.3f", v)
        },
        fmt_ci         = function(lo, hi) {
            if (is.na(lo) || is.na(hi)) return("--")
            sprintf("[%+.3f, %+.3f]", lo, hi)
        },
        axis_format    = scales::label_number(accuracy = 0.01),
        plot_transform = function(v) v
    ),
    log_odds = list(
        scale          = "log",
        null_value     = 1,
        x_label        = "Odds Ratio (SnC vs Non-SnC)",
        eff_h_label    = "OR",
        ci_h_label     = "95% CI (OR)",
        fmt_effect     = function(v) {
            if (is.na(v)) return("--")
            sprintf("%.2f", exp(v))
        },
        fmt_ci         = function(lo, hi) {
            if (is.na(lo) || is.na(hi)) return("--")
            sprintf("[%.2f, %.2f]", exp(lo), exp(hi))
        },
        axis_format    = scales::label_number(accuracy = 0.01),
        plot_transform = function(v) exp(v)
    )
)


# ─────────────────────────────────────────────────────────────────────────────
# Plot style
# ─────────────────────────────────────────────────────────────────────────────
PLOT_STYLE <- list(
    dpi        = 150,
    dpi_save   = 300,
    font_size  = 10,
    title_size = 11,
    formats    = c("pdf", "png", "svg"),
    pt_size    = 0.05,
    label_size = 4
)

theme_clean <- function(base_size = PLOT_STYLE$font_size) {
    theme_classic(base_size = base_size) +
    theme(
        plot.title       = element_text(size = PLOT_STYLE$title_size,
                                        face = "plain", hjust = 0),
        legend.title     = element_text(size = base_size, face = "plain"),
        panel.border     = element_rect(color = "black", fill = NA, linewidth = 0.5),
        panel.grid       = element_blank(),
        axis.line        = element_blank()
    )
}

---
## 02 · Setup — cross-section helpers

**Why.** Every downstream section formats numbers, saves a figure or a table,
slices to a stratum, and packs a model fit into one standard row. Defining
those once here is what keeps sections 08-21 comparable: the same
`tidy_model_results()` record shape is what makes the cross-model agreement
sections possible at all.

`tidy_model_results()` is the contract — `stratum`, `outcome`, `model`,
`n_donors`, `n_cells_test`, `n_cells_ref`, `estimate`, `se`, `ci_low`,
`ci_high`, `statistic`, `p_value`, plus per-model extras.

In [ ]:
# =============================================================================
# CROSS-SECTION HELPERS
# =============================================================================

# ─────────────────────────────────────────────────────────────────────────────
# Formatting helpers
# ─────────────────────────────────────────────────────────────────────────────
fmt_n <- function(n) format(round(n), big.mark = ",", scientific = FALSE)

fmt_size <- function(path) {
    if (!file.exists(path)) return("missing")
    sz <- file.size(path)
    if (sz > 1024^3) return(sprintf("%.2f GB", sz / 1024^3))
    if (sz > 1024^2) return(sprintf("%.1f MB", sz / 1024^2))
    sprintf("%.1f KB", sz / 1024)
}

fmt_pct <- function(num, denom) {
    if (denom == 0) return(sprintf("%s (--)", fmt_n(num)))
    sprintf("%s (%.1f%%)", fmt_n(num), num / denom * 100)
}

fmt_elapsed <- function(secs) {
    if (secs < 60)   return(sprintf("%.1f sec", secs))
    if (secs < 3600) return(sprintf("%.1f min", secs / 60))
    sprintf("%.1f hr", secs / 3600)
}

fmt_p <- function(p) {
    if (is.na(p)) return("NA")
    if (p < 0.001) return(sprintf("%.2e", p))
    sprintf("%.3f", p)
}

fmt_p_short <- function(p) {
    if (is.na(p)) return("--")
    if (p < 0.001) return(sprintf("%.1e", p))
    sprintf("%.3f", p)
}

sig_stars <- function(p) {
    ifelse(is.na(p), "",
    ifelse(p < 0.001, "***",
    ifelse(p < 0.01,  "**",
    ifelse(p < 0.05,  "*", "ns"))))
}

now_iso <- function() format(Sys.time(), "%Y-%m-%dT%H:%M:%S")

bytes_str <- function(x) format(x, scientific = FALSE, trim = TRUE)


# ─────────────────────────────────────────────────────────────────────────────
# Color helpers
# ─────────────────────────────────────────────────────────────────────────────
text_color_for_bg <- function(hex) {
    rgb_vals  <- col2rgb(hex)
    luminance <- 0.299 * rgb_vals[1, ] + 0.587 * rgb_vals[2, ] + 0.114 * rgb_vals[3, ]
    ifelse(luminance < 140, "white", "black")
}


# ─────────────────────────────────────────────────────────────────────────────
# Time / save helpers
# ─────────────────────────────────────────────────────────────────────────────
time_step <- function(label, expr) {
    cat(sprintf("\n▸ %s\n", label))
    t0  <- Sys.time()
    res <- expr
    elapsed <- as.numeric(difftime(Sys.time(), t0, units = "secs"))
    cat(sprintf("  ✓ %s  (%s)\n", label, fmt_elapsed(elapsed)))
    res
}

save_figure <- function(fig, slug, width = 10, height = 7) {
    for (ext in PLOT_STYLE$formats) {
        path <- file.path(PATHS$figures, paste0(slug, ".", ext))
        ggsave(path, fig, width = width, height = height,
               dpi = PLOT_STYLE$dpi_save, bg = "white")
    }
    cat(sprintf("  ✓ saved → figures/%s.{%s}\n",
                slug, paste(PLOT_STYLE$formats, collapse = ",")))
}

save_table <- function(df, slug, row.names = FALSE) {
    path <- file.path(PATHS$results, paste0(slug, ".csv"))
    write.csv(df, path, row.names = row.names)
    cat(sprintf("  ✓ saved → results/%s.csv  (%d rows)\n",
                slug, nrow(df)))
}


# ─────────────────────────────────────────────────────────────────────────────
# filter_to_stratum() — slice metadata to one stratum
# ─────────────────────────────────────────────────────────────────────────────
filter_to_stratum <- function(md, stratum, study_group_col) {
    if (stratum == "All") return(md)
    md[md[[study_group_col]] == stratum, , drop = FALSE]
}


# ─────────────────────────────────────────────────────────────────────────────
# tidy_model_results() — standardize one-row result records across all models
# ─────────────────────────────────────────────────────────────────────────────
tidy_model_results <- function(stratum, outcome, model,
                               n_donors, n_cells_test, n_cells_ref,
                               estimate, se, ci_low, ci_high,
                               statistic, p_value,
                               extra = NULL) {
    out <- data.frame(
        stratum      = as.character(stratum),
        outcome      = as.character(outcome),
        model        = as.character(model),
        n_donors     = as.integer(n_donors),
        n_cells_test = as.integer(n_cells_test),
        n_cells_ref  = as.integer(n_cells_ref),
        estimate     = as.numeric(estimate),
        se           = as.numeric(se),
        ci_low       = as.numeric(ci_low),
        ci_high      = as.numeric(ci_high),
        statistic    = as.numeric(statistic),
        p_value      = as.numeric(p_value),
        stringsAsFactors = FALSE
    )
    if (!is.null(extra) && length(extra) > 0) {
        for (nm in names(extra)) {
            v <- extra[[nm]]
            if (length(v) != 1) v <- I(list(v))
            out[[nm]] <- v
        }
    }
    out
}


# ─────────────────────────────────────────────────────────────────────────────
# Library version log
# ─────────────────────────────────────────────────────────────────────────────
R_PKG_VERSIONS <- list(
    R          = R.version$version.string,
    Seurat     = as.character(packageVersion("Seurat")),
    Matrix     = as.character(packageVersion("Matrix")),
    dplyr      = as.character(packageVersion("dplyr")),
    tidyr      = as.character(packageVersion("tidyr")),
    readxl     = as.character(packageVersion("readxl")),
    ggplot2    = as.character(packageVersion("ggplot2")),
    patchwork  = as.character(packageVersion("patchwork")),
    qs         = as.character(packageVersion("qs")),
    jsonlite   = as.character(packageVersion("jsonlite")),
    lme4       = as.character(packageVersion("lme4")),
    lmerTest   = as.character(packageVersion("lmerTest")),
    MASS       = as.character(packageVersion("MASS")),
    robustbase = as.character(packageVersion("robustbase")),
    broom      = as.character(packageVersion("broom")),
    broom.mixed = as.character(packageVersion("broom.mixed"))
)



# -----------------------------------------------------------------------------
# Resolved-config banner
# -----------------------------------------------------------------------------
cat("\n  Run parameters:\n")
cat(sprintf("    TISSUE         : %s\n", TISSUE))
cat(sprintf("    STUDY_TYPE     : %s\n", STUDY_TYPE))
if (IS_DISEASE)
    cat(sprintf("    DISEASE        : %s\n", DISEASE))
cat(sprintf("    DATASET        : %s\n", DATASET))
cat(sprintf("    CELL_TYPE      : %s\n", CELL_TYPE))
cat(sprintf("    CONDITION_TAG  : %s\n", CONDITION_TAG))

cat(sprintf("\n  Inline plot viewport: %dx%d\n", 10, 5))

cat("\n  Stratification:\n")
cat(sprintf("    STRATIFY_BY_GROUP     : %s\n", STRATIFY_BY_GROUP))
if (STRATIFY_BY_GROUP) {
    cat(sprintf("    STRATIFICATION_GROUPS : %s\n",
                paste(STRATIFICATION_GROUPS, collapse = ", ")))
    cat(sprintf("    Total runs per cell   : 1 (\"All\") + %d strata = %d\n",
                length(STRATIFICATION_GROUPS), 1 + length(STRATIFICATION_GROUPS)))
} else {
    cat("    Stratified runs       : disabled (overall only)\n")
}

cat("\n  Statistical params:\n")
cat(sprintf("    min_cells_per_group : %d\n", STATISTICAL_PARAMS$min_cells_per_group))
cat(sprintf("    fdr_threshold       : %.2f (%s)\n",
            STATISTICAL_PARAMS$fdr_threshold, STATISTICAL_PARAMS$fdr_method))
cat(sprintf("    bootstrap_n_iter    : %d (seed=%d)\n",
            STATISTICAL_PARAMS$bootstrap_n_iter, STATISTICAL_PARAMS$bootstrap_seed))
cat(sprintf("    confidence_level    : %.2f\n", STATISTICAL_PARAMS$confidence_level))

cat("\n  Effect display configs:\n")
for (nm in names(EFFECT_CONFIGS)) {
    cfg <- EFFECT_CONFIGS[[nm]]
    cat(sprintf("    %-15s -> axis=%s, null=%g, label='%s'\n",
                nm, cfg$scale, cfg$null_value, cfg$x_label))
}

cat("\n  Cross-cell helpers (defined in §0):\n")
cat("    Formatting   : fmt_n, fmt_size, fmt_pct, fmt_elapsed, fmt_p, fmt_p_short, sig_stars\n")
cat("    Time/save    : now_iso, bytes_str, time_step, save_figure, save_table\n")
cat("    Color        : text_color_for_bg\n")
cat("    Stratum      : filter_to_stratum\n")
cat("    Results      : tidy_model_results\n")
cat("    (Donor data helpers: build_donor_arms, build_donor_meta, tidy_lmm_term -- defined in §3.6)\n")

cat("\n  Senescence gene lists:\n")
cat(sprintf("    Source              : Sloan Table S9 (sheet 'Sen Gene Lists')\n"))
cat(sprintf("    Lists               : %d (8 hallmarks + 2 multi-hallmark)\n",
            length(SLOAN_HALLMARK_NAMES)))
cat(sprintf("    Fixed row order     : %s\n",
            paste(SLOAN_HALLMARK_NAMES, collapse = ", ")))

cat("\n  Proliferation markers (§7):\n")
cat(sprintf("    %s\n", paste(PROLIFERATION_MARKERS, collapse = ", ")))

cat("\n  Library versions:\n")
for (pkg in c("R", "Seurat", "lme4", "robustbase", "broom.mixed",
              "qs", "jsonlite")) {
    cat(sprintf("    %-12s : %s\n", pkg, R_PKG_VERSIONS[[pkg]]))
}

cat("\n  Paths:\n")
cat(sprintf("    M04 input root     : %s\n", PATHS$m04_root))
cat(sprintf("    M05 output root    : %s\n", PATHS$output_root))

cat("\n  Required input files:\n")
for (key in c("m04_seurat", "m04_manifest", "sloan_xlsx",
              "senmayo_xlsx", "fridman_gmt")) {
    exists_flag <- if (file.exists(PATHS[[key]])) "✓" else "✗"
    sz <- if (file.exists(PATHS[[key]])) fmt_size(PATHS[[key]]) else "MISSING"
    cat(sprintf("    %s  %-15s : %s  (%s)\n",
                exists_flag, key, PATHS[[key]], sz))
}

cat("\n=", strrep("=", 71), "\n", sep = "")
cat("Config loaded. All cross-section helpers in scope.\n")

---
## 03 · Load

**Why.** Reads the module 04 manifest and tissue object, subsets to `CELL_TYPE`, resolves the metadata column names, and validates that the senescence label from module 03 survived the export.

**Defines for later sections.** `obj_ct` (Seurat, one cell type) · `md` (its metadata) · `DONOR_COL` · `SEN_LABEL_COL` · `STUDY_GROUP_COL` · `SEX_COL` · `AGE_COL` · `COHORT_COL` · `HAS_COHORT` · `strata`

> **Source not in this rebuild.** The code for this section lives in
> `05_validation_v2.ipynb, cell 2` and was not carried over. Paste it in here — the sections below
> depend on the objects listed above and will stop without them.

In [ ]:
# =============================================================================
# MODULE 05 — Senescence Enrichment & Cell Cycle Analysis
# LOAD (Cell §1)
# =============================================================================
# Reads M04 manifest.json, validates run parameters match, loads the M04
# tissue-wide .qs, resolves canonical column names from manifest hints,
# subsets to CELL_TYPE, validates STRATIFICATION_GROUPS exist in the data,
# and reports per-stratum cell + paired-donor counts.
#
# After §1 the in-memory state is:
#   m04_manifest   — chained provenance from M04
#   obj_full       — full tissue Seurat object (all 3 cell types)
#   obj_ct         — subsetted Seurat object (just CELL_TYPE)
#   md             — obj_ct@meta.data (data frame, used by all model cells)
#   DONOR_COL, CELLTYPE_COL, STUDY_GROUP_COL, SEN_COL, SEN_LABEL_STR_COL,
#       SEX_COL, AGE_COL, COHORT_COL, LOG10_COUNTS_COL  (resolved from manifest)
#   strata          — character vector of strata to iterate (always starts with "All")
#   stratum_n_cells — named integer vector: cells per stratum
#   stratum_n_donors — named integer vector: paired donors per stratum
# =============================================================================

cat("=", strrep("=", 71), "\n", sep = "")
cat(sprintf("§1 — LOAD  |  %s / %s\n", DATASET, CELL_TYPE))
cat("=", strrep("=", 71), "\n", sep = "")


# ─────────────────────────────────────────────────────────────────────────────
# §1.1 Read M04 manifest + validate run parameters
# ─────────────────────────────────────────────────────────────────────────────
cat("\n▸ §1.1 Read M04 manifest.json\n")

if (!file.exists(PATHS$m04_manifest)) {
    stop(sprintf("✗ M04 manifest not found at %s — has Module 04 finished?",
                 PATHS$m04_manifest))
}

m04_manifest <- jsonlite::fromJSON(PATHS$m04_manifest, simplifyVector = TRUE)

# Cross-validate run parameters
mismatch <- list()
if (m04_manifest$tissue     != TISSUE)
    mismatch[["TISSUE"]]    <- c(R = TISSUE,    M04 = m04_manifest$tissue)
if (m04_manifest$study_type != STUDY_TYPE)
    mismatch[["STUDY_TYPE"]] <- c(R = STUDY_TYPE, M04 = m04_manifest$study_type)
if (m04_manifest$dataset    != DATASET)
    mismatch[["DATASET"]]   <- c(R = DATASET,   M04 = m04_manifest$dataset)

if (length(mismatch) > 0) {
    cat("✗ Run parameter mismatch between M05 §0 and M04 manifest:\n")
    for (key in names(mismatch)) {
        cat(sprintf("    %s : §0=%s, M04=%s\n",
                    key, mismatch[[key]]["R"], mismatch[[key]]["M04"]))
    }
    stop("Edit §0 to match M04, or rerun M04 with the right parameters.")
}

cat(sprintf("  ✓ Loaded: %s  (schema_version=%s)\n",
            basename(PATHS$m04_manifest), m04_manifest$schema_version))
cat(sprintf("  M04 wrote: %s cells × %s genes  (subset_mode=%s)\n",
            fmt_n(m04_manifest$n_cells),
            fmt_n(m04_manifest$n_genes),
            m04_manifest$subset_mode))
cat(sprintf("  Counts source: %s\n",
            m04_manifest$python_processing$counts_source))


# ─────────────────────────────────────────────────────────────────────────────
# §1.2 Resolve column names from manifest hints
#
# This is the chained-provenance pattern: M04 wrote the canonical column
# names to manifest, and M05 reads them rather than hardcoding. Switching
# datasets later means only editing §0; column resolution is automatic.
# ─────────────────────────────────────────────────────────────────────────────
cat("\n▸ §1.2 Resolve column names from manifest\n")

DONOR_COL          <- m04_manifest$donor_col
CELLTYPE_COL       <- m04_manifest$celltype_col
COLLAPSED_COL      <- m04_manifest$celltype_col_collapsed
STUDY_GROUP_COL    <- m04_manifest$study_group_col
SEX_COL            <- m04_manifest$sex_col
AGE_COL            <- m04_manifest$age_col
COHORT_COL         <- m04_manifest$cohort_col           # NULL for psychad_ad
SEN_SCORE_COL      <- m04_manifest$sen_score_col
SEN_LABEL_COL      <- m04_manifest$sen_label_col        # binary 0/1
SEN_LABEL_STR_COL  <- m04_manifest$sen_label_str_col    # "Senescent"/"Non-senescent"
LOG10_COUNTS_COL   <- m04_manifest$log10_total_counts_col
HAS_COHORT         <- isTRUE(m04_manifest$has_cohort)

cat(sprintf("  donor       : %s\n", DONOR_COL))
cat(sprintf("  cell_type   : %s\n", CELLTYPE_COL))
cat(sprintf("  study_group : %s\n", STUDY_GROUP_COL))
cat(sprintf("  sex         : %s\n", SEX_COL))
cat(sprintf("  age         : %s\n", AGE_COL))
cat(sprintf("  cohort      : %s%s\n",
            if (is.null(COHORT_COL)) "(NULL)" else COHORT_COL,
            if (HAS_COHORT) "" else "  [HAS_COHORT=FALSE]"))
cat(sprintf("  sen_score   : %s\n", SEN_SCORE_COL))
cat(sprintf("  sen_label   : %s  (binary)\n", SEN_LABEL_COL))
cat(sprintf("  sen_label_str: %s  (string)\n", SEN_LABEL_STR_COL))
cat(sprintf("  log10_UMI   : %s\n", LOG10_COUNTS_COL))


# ─────────────────────────────────────────────────────────────────────────────
# §1.3 Load M04 .qs
# ─────────────────────────────────────────────────────────────────────────────
obj_full <- time_step("§1.3 qread(M04 .qs)", {
    cat(sprintf("  Source: %s  (%s)\n",
                PATHS$m04_seurat, fmt_size(PATHS$m04_seurat)))
    qread(PATHS$m04_seurat)
})

cat(sprintf("        Loaded: %s cells × %s genes\n",
            fmt_n(ncol(obj_full)), fmt_n(nrow(obj_full))))
cat(sprintf("        Active layers : %s\n",
            paste(Layers(obj_full, assay = DefaultAssay(obj_full)), collapse = ", ")))
cat(sprintf("        Reductions    : %s\n",
            paste(names(obj_full@reductions), collapse = ", ")))

# Sanity: data layer must be populated for module/cell-cycle scoring to work
if (!"data" %in% Layers(obj_full, assay = DefaultAssay(obj_full))) {
    stop("✗ data layer missing — re-run M04 §5 (NormalizeData) before M05.")
}


# ─────────────────────────────────────────────────────────────────────────────
# §1.4 Subset to CELL_TYPE
# ─────────────────────────────────────────────────────────────────────────────
cat("\n▸ §1.4 Subset to CELL_TYPE = '", CELL_TYPE, "'\n", sep = "")

available_cts <- sort(unique(obj_full@meta.data[[CELLTYPE_COL]]))
cat(sprintf("  Available cell types in obj_full: %s\n",
            paste(available_cts, collapse = ", ")))

if (!CELL_TYPE %in% available_cts) {
    stop(sprintf("✗ CELL_TYPE='%s' not in %s. Available: %s",
                 CELL_TYPE, CELLTYPE_COL,
                 paste(available_cts, collapse = ", ")))
}

cell_mask <- obj_full@meta.data[[CELLTYPE_COL]] == CELL_TYPE
obj_ct    <- obj_full[, cell_mask]

cat(sprintf("  ✓ Subset: %s cells × %s genes\n",
            fmt_n(ncol(obj_ct)), fmt_n(nrow(obj_ct))))

# Free the full tissue object — we only need the subset from here on
rm(obj_full); gc(verbose = FALSE)


# ─────────────────────────────────────────────────────────────────────────────
# §1.5 Validate STRATIFICATION_GROUPS against actual Study_Group levels
#
# This catches the Age_20-29 vs Age_20_29 typo bug from M09 — if a stratum
# label doesn't match what's in obs, error immediately rather than silently
# returning zero rows downstream.
# ─────────────────────────────────────────────────────────────────────────────
cat("\n▸ §1.5 Validate stratification\n")

actual_sg_levels <- sort(unique(obj_ct@meta.data[[STUDY_GROUP_COL]]))
cat(sprintf("  Actual %s levels in obj_ct (%d): %s\n",
            STUDY_GROUP_COL, length(actual_sg_levels),
            paste(actual_sg_levels, collapse = ", ")))

if (STRATIFY_BY_GROUP) {
    invalid_strata <- setdiff(STRATIFICATION_GROUPS, actual_sg_levels)
    if (length(invalid_strata) > 0) {
        stop(sprintf(
            "✗ STRATIFICATION_GROUPS contains values not present in %s: %s\n  Valid: %s\n  Edit STRATIFICATION_GROUPS in §0 to fix.",
            STUDY_GROUP_COL,
            paste(invalid_strata, collapse = ", "),
            paste(actual_sg_levels, collapse = ", ")
        ))
    }
    cat(sprintf("  ✓ All %d stratification groups valid\n",
                length(STRATIFICATION_GROUPS)))
}

# Build the canonical strata vector — used by every model cell in §4/§5/§7
strata <- c("All",
            if (STRATIFY_BY_GROUP) STRATIFICATION_GROUPS else character(0))
cat(sprintf("  Will iterate over %d strata: %s\n",
            length(strata), paste(strata, collapse = ", ")))


# ─────────────────────────────────────────────────────────────────────────────
# §1.6 Per-stratum cell + paired-donor counts
#
# A donor is "paired" within a stratum when it has ≥min_cells_per_group cells
# in BOTH SnC and Non-SnC arms. This drives the donor-level Wilcoxon and
# RLM tests in §4/§5 — strata with too few paired donors will produce
# unreliable estimates.
# ─────────────────────────────────────────────────────────────────────────────
cat("\n▸ §1.6 Cell + paired-donor counts per stratum\n")

md <- as.data.frame(obj_ct@meta.data)
md_sen_lower <- as.character(md[[SEN_LABEL_COL]])

# Confirm sen_label encoding is something we can work with
cat(sprintf("  Senescence column '%s' (class=%s):\n",
            SEN_LABEL_COL, class(md[[SEN_LABEL_COL]])[1]))
print(table(md[[SEN_LABEL_COL]]))

stratum_summary <- data.frame(
    stratum         = strata,
    n_cells         = NA_integer_,
    n_cells_snc     = NA_integer_,
    n_cells_nonsnc  = NA_integer_,
    n_donors_total  = NA_integer_,
    n_donors_paired = NA_integer_,
    stringsAsFactors = FALSE
)

for (i in seq_along(strata)) {
    s   <- strata[i]
    md_s <- filter_to_stratum(md, s, STUDY_GROUP_COL)

    # Cell counts
    n_cells <- nrow(md_s)
    n_snc   <- sum(md_s[[SEN_LABEL_COL]] %in% c(1, "1", TRUE, "TRUE", "True", "Senescent"))
    n_non   <- n_cells - n_snc

    # Donor-level: how many have ≥min_cells in BOTH groups?
    donor_cell_counts <- md_s %>%
        group_by(.data[[DONOR_COL]], .data[[SEN_LABEL_COL]]) %>%
        tally(name = "n_cells") %>%
        ungroup()

    paired_ids <- donor_cell_counts %>%
        filter(n_cells >= STATISTICAL_PARAMS$min_cells_per_group) %>%
        group_by(.data[[DONOR_COL]]) %>%
        filter(n_distinct(.data[[SEN_LABEL_COL]]) == 2) %>%
        pull(.data[[DONOR_COL]]) %>%
        unique()

    n_donors_total  <- length(unique(md_s[[DONOR_COL]]))
    n_donors_paired <- length(paired_ids)

    stratum_summary$n_cells[i]         <- n_cells
    stratum_summary$n_cells_snc[i]     <- n_snc
    stratum_summary$n_cells_nonsnc[i]  <- n_non
    stratum_summary$n_donors_total[i]  <- n_donors_total
    stratum_summary$n_donors_paired[i] <- n_donors_paired
}

cat("\n  Per-stratum counts (paired = donors with ≥",
    STATISTICAL_PARAMS$min_cells_per_group, " cells in BOTH SnC & Non-SnC arms):\n", sep = "")
print(stratum_summary)

# Persist for §8 manifest
stratum_n_cells  <- setNames(stratum_summary$n_cells, stratum_summary$stratum)
stratum_n_donors <- setNames(stratum_summary$n_donors_paired, stratum_summary$stratum)

# Save the stratum summary as the first M05 result
save_table(stratum_summary, paste0(CELL_TYPE, "_stratum_summary"))

# Warn (not error) on thin strata
thin_strata <- stratum_summary$stratum[stratum_summary$n_donors_paired < 5]
if (length(thin_strata) > 0) {
    cat(sprintf("\n  ⚠ Thin strata (<5 paired donors): %s\n",
                paste(thin_strata, collapse = ", ")))
    cat("    Donor-level tests in these strata will be unreliable;\n")
    cat("    cell-level tests (LMM, OLS, balanced bootstrap) may still work.\n")
}


# ─────────────────────────────────────────────────────────────────────────────
# §1.7 Section summary
# ─────────────────────────────────────────────────────────────────────────────
cat("\n", strrep("─", 72), "\n", sep = "")
cat(sprintf("  §1 SUMMARY  |  %s / %s\n", DATASET, CELL_TYPE))
cat(strrep("─", 72), "\n", sep = "")
cat(sprintf("  Object class      : %s\n", class(obj_ct)[1]))
cat(sprintf("  Cells × genes     : %s × %s\n",
            fmt_n(ncol(obj_ct)), fmt_n(nrow(obj_ct))))
cat(sprintf("  Active layers     : %s\n",
            paste(Layers(obj_ct, assay = DefaultAssay(obj_ct)), collapse = ", ")))
cat(sprintf("  Reductions        : %s\n",
            paste(names(obj_ct@reductions), collapse = ", ")))
cat(sprintf("  Memory            : %.2f GB\n",
            as.numeric(object.size(obj_ct)) / 1024^3))
cat(sprintf("  Strata            : %s\n", paste(strata, collapse = ", ")))
cat(sprintf("  Total paired donors (across all strata): %s\n",
            paste(sprintf("%s=%d", strata, stratum_n_donors), collapse = ", ")))
cat(strrep("─", 72), "\n", sep = "")
cat("\n✓ §1 load complete\n")
cat("  obj_ct, md, strata, and column variables are in scope for §2-§8.\n")
cat("  Next: §2 — gene list preparation\n")

---
## 04 · Gene lists

**Why.** Reads Sloan Table S9 (10 lists), SenMayo and Fridman, intersects each against the object's `rownames`, and reports the surviving panel size. Order is fixed by `SLOAN_HALLMARK_NAMES` so every forest plot in sections 14-19 has the same row order.

**Defines for later sections.** `gene_lists` (named list of character vectors), written to `PATHS$gene_lists_rds`

> **Source not in this rebuild.** The code for this section lives in
> `05_validation_v2.ipynb, cell 3` and was not carried over. Paste it in here — the sections below
> depend on the objects listed above and will stop without them.

In [ ]:
# =============================================================================
# MODULE 05 — Senescence Enrichment & Cell Cycle Analysis
# GENE LIST PREPARATION (Cell §2)
# =============================================================================
# Loads the 10 senescence gene lists from Sloan Table S9 (PRIMARY SOURCE),
# cross-validates SenMayo against Saul et al. and Fridman_Up against MSigDB,
# filters each list to genes detected in obj_ct, saves gene_lists.rds.
#
# After §2 the in-memory state adds:
#   gene_lists      — named list of 10 character vectors (gene symbols),
#                     filtered to genes present in obj_ct
#   gene_list_meta  — data frame summarizing each list (counts, type, source)
# =============================================================================

cat("=", strrep("=", 71), "\n", sep = "")
cat(sprintf("§2 — GENE LIST PREP  |  %s / %s\n", DATASET, CELL_TYPE))
cat("=", strrep("=", 71), "\n", sep = "")

# Sanity checks
if (!exists("obj_ct")) stop("✗ Run §1 first — obj_ct not in scope.")
if (!file.exists(PATHS$sloan_xlsx))   stop("✗ Sloan xlsx missing.")
if (!file.exists(PATHS$senmayo_xlsx)) stop("✗ SenMayo xlsx missing.")
if (!file.exists(PATHS$fridman_gmt))  stop("✗ Fridman gmt missing.")

available_genes <- rownames(obj_ct)
cat(sprintf("\n  Genes detected in obj_ct: %s\n", fmt_n(length(available_genes))))


# ─────────────────────────────────────────────────────────────────────────────
# §2.1 PRIMARY SOURCE: Load Sloan Table S9
#
# Sheet: "Sen Gene Lists"
# Rows: hallmark name in row 1, then gene symbols
# Columns: hardcoded mapping from SLOAN_HALLMARK_NAMES (defined in §0)
# ─────────────────────────────────────────────────────────────────────────────
cat("\n▸ §2.1 Load Sloan Table S9\n")

sloan_raw <- read_excel(PATHS$sloan_xlsx, sheet = "Sen Gene Lists")
cat(sprintf("  ✓ Loaded sheet 'Sen Gene Lists': %d rows × %d cols\n",
            nrow(sloan_raw), ncol(sloan_raw)))

if (ncol(sloan_raw) < length(SLOAN_HALLMARK_NAMES)) {
    stop(sprintf(
        "✗ Sloan sheet has %d cols, expected ≥%d (one per hallmark in SLOAN_HALLMARK_NAMES)",
        ncol(sloan_raw), length(SLOAN_HALLMARK_NAMES)))
}

# ─────────────────────────────────────────────────────────────────────────────
# §2.2 Build per-list character vectors and filter to detected genes
# ─────────────────────────────────────────────────────────────────────────────
cat("\n▸ §2.2 Build gene_lists and filter to detected\n")

gene_lists      <- list()
gene_list_meta  <- data.frame(
    list_name      = SLOAN_HALLMARK_NAMES,
    list_type      = SLOAN_LIST_TYPES,
    n_total        = NA_integer_,
    n_detected     = NA_integer_,
    pct_detected   = NA_real_,
    sloan_col_idx  = seq_along(SLOAN_HALLMARK_NAMES),
    stringsAsFactors = FALSE
)

cat(sprintf("\n  %-22s %-22s %8s %8s %7s\n",
            "List", "Type", "Total", "Detect", "%"))
cat("  ", strrep("-", 70), "\n", sep = "")

for (i in seq_along(SLOAN_HALLMARK_NAMES)) {
    list_name <- SLOAN_HALLMARK_NAMES[i]
    # Skip header row (which contains the hallmark name string)
    genes_raw <- sloan_raw[[i]][-1]
    genes_clean <- genes_raw[!is.na(genes_raw) & genes_raw != ""]
    genes_clean <- trimws(as.character(genes_clean))
    genes_clean <- unique(genes_clean)

    detected <- intersect(genes_clean, available_genes)
    gene_lists[[list_name]] <- detected

    n_total    <- length(genes_clean)
    n_detected <- length(detected)
    pct        <- if (n_total > 0) n_detected / n_total * 100 else 0

    gene_list_meta$n_total[i]      <- n_total
    gene_list_meta$n_detected[i]   <- n_detected
    gene_list_meta$pct_detected[i] <- round(pct, 1)

    cat(sprintf("  %-22s %-22s %8d %8d %6.1f%%\n",
                list_name, SLOAN_LIST_TYPES[i],
                n_total, n_detected, pct))
}


# ─────────────────────────────────────────────────────────────────────────────
# §2.3 Cross-validate SenMayo against Saul et al. original
# ─────────────────────────────────────────────────────────────────────────────
cat("\n▸ §2.3 Cross-validate SenMayo against Saul et al.\n")

senmayo_orig <- read_excel(PATHS$senmayo_xlsx, sheet = "human")[["Gene(human)"]]
senmayo_orig <- senmayo_orig[!is.na(senmayo_orig) & senmayo_orig != ""]
senmayo_orig <- unique(trimws(as.character(senmayo_orig)))

senmayo_sloan_raw <- sloan_raw[[which(SLOAN_HALLMARK_NAMES == "SenMayo")]][-1]
senmayo_sloan <- senmayo_sloan_raw[!is.na(senmayo_sloan_raw) & senmayo_sloan_raw != ""]
senmayo_sloan <- unique(trimws(as.character(senmayo_sloan)))

senmayo_overlap   <- intersect(senmayo_sloan, senmayo_orig)
senmayo_only_sl   <- setdiff(senmayo_sloan, senmayo_orig)
senmayo_only_orig <- setdiff(senmayo_orig, senmayo_sloan)

cat(sprintf("  Sloan SenMayo     : %d genes\n", length(senmayo_sloan)))
cat(sprintf("  Saul (orig.)      : %d genes\n", length(senmayo_orig)))
cat(sprintf("  Overlap           : %d (%.1f%% of Sloan)\n",
            length(senmayo_overlap),
            100 * length(senmayo_overlap) / max(1, length(senmayo_sloan))))

if (length(senmayo_only_sl) > 0) {
    cat(sprintf("  Sloan-only        : %d genes  e.g. %s\n",
                length(senmayo_only_sl),
                paste(head(senmayo_only_sl, 5), collapse = ", ")))
}
if (length(senmayo_only_orig) > 0) {
    cat(sprintf("  Saul-only         : %d genes  e.g. %s\n",
                length(senmayo_only_orig),
                paste(head(senmayo_only_orig, 5), collapse = ", ")))
}

senmayo_validation <- list(
    sloan_n      = length(senmayo_sloan),
    saul_n       = length(senmayo_orig),
    overlap_n    = length(senmayo_overlap),
    overlap_pct  = round(100 * length(senmayo_overlap) / max(1, length(senmayo_sloan)), 2),
    sloan_only   = senmayo_only_sl,
    saul_only    = senmayo_only_orig
)


# ─────────────────────────────────────────────────────────────────────────────
# §2.4 Cross-validate Fridman_Up against MSigDB original
# ─────────────────────────────────────────────────────────────────────────────
cat("\n▸ §2.4 Cross-validate Fridman_Up against MSigDB\n")

gmt_line <- readLines(PATHS$fridman_gmt, n = 1, warn = FALSE)
fridman_orig <- strsplit(gmt_line, "\t")[[1]]
# GMT format: <gene_set_name>\t<source_url>\t<gene_1>\t<gene_2>\t...
fridman_orig <- fridman_orig[-(1:2)]
fridman_orig <- fridman_orig[!is.na(fridman_orig) & fridman_orig != ""]
fridman_orig <- unique(trimws(fridman_orig))

fridman_sloan_raw <- sloan_raw[[which(SLOAN_HALLMARK_NAMES == "Fridman_Up")]][-1]
fridman_sloan <- fridman_sloan_raw[!is.na(fridman_sloan_raw) & fridman_sloan_raw != ""]
fridman_sloan <- unique(trimws(as.character(fridman_sloan)))

fridman_overlap   <- intersect(fridman_sloan, fridman_orig)
fridman_only_sl   <- setdiff(fridman_sloan, fridman_orig)
fridman_only_orig <- setdiff(fridman_orig, fridman_sloan)

cat(sprintf("  Sloan Fridman_Up  : %d genes\n", length(fridman_sloan)))
cat(sprintf("  MSigDB (orig.)    : %d genes\n", length(fridman_orig)))
cat(sprintf("  Overlap           : %d (%.1f%% of Sloan)\n",
            length(fridman_overlap),
            100 * length(fridman_overlap) / max(1, length(fridman_sloan))))

if (length(fridman_only_sl) > 0) {
    cat(sprintf("  Sloan-only        : %d genes  e.g. %s\n",
                length(fridman_only_sl),
                paste(head(fridman_only_sl, 5), collapse = ", ")))
}
if (length(fridman_only_orig) > 0) {
    cat(sprintf("  MSigDB-only       : %d genes  e.g. %s\n",
                length(fridman_only_orig),
                paste(head(fridman_only_orig, 5), collapse = ", ")))
}

fridman_validation <- list(
    sloan_n      = length(fridman_sloan),
    msigdb_n     = length(fridman_orig),
    overlap_n    = length(fridman_overlap),
    overlap_pct  = round(100 * length(fridman_overlap) / max(1, length(fridman_sloan)), 2),
    sloan_only   = fridman_only_sl,
    msigdb_only  = fridman_only_orig
)


# ─────────────────────────────────────────────────────────────────────────────
# §2.5 Final summary, viability check, save
# ─────────────────────────────────────────────────────────────────────────────
cat("\n▸ §2.5 Final summary\n")

# Add total unique gene count across all 10 lists (after detection filtering)
total_unique_genes <- length(unique(unlist(gene_lists)))
cat(sprintf("\n  Total unique genes across 10 lists (post-detection filter): %d\n",
            total_unique_genes))

# Viability check — flag lists with too few detected genes for reliable scoring
MIN_GENES_FOR_SCORING <- 5
small_lists <- gene_list_meta$list_name[gene_list_meta$n_detected < MIN_GENES_FOR_SCORING]

if (length(small_lists) > 0) {
    cat(sprintf("\n  ⚠ Lists with <%d detected genes (will produce noisy scores):\n",
                MIN_GENES_FOR_SCORING))
    for (nm in small_lists) {
        n <- gene_list_meta$n_detected[gene_list_meta$list_name == nm]
        cat(sprintf("      %s: %d\n", nm, n))
    }
    cat("    These lists will still be scored in §3 but flagged in §4/§5 results.\n")
} else {
    cat(sprintf("\n  ✓ All 10 lists have ≥%d detected genes — all viable for scoring.\n",
                MIN_GENES_FOR_SCORING))
}

# Save gene lists + metadata
saveRDS(gene_lists, PATHS$gene_lists_rds)
cat(sprintf("\n  ✓ Saved gene_lists.rds → data/gene_lists.rds  (%s)\n",
            fmt_size(PATHS$gene_lists_rds)))

# Save summary CSV
save_table(gene_list_meta, paste0(CELL_TYPE, "_gene_list_summary"))


# ─────────────────────────────────────────────────────────────────────────────
# §2.6 Section summary
# ─────────────────────────────────────────────────────────────────────────────
cat("\n", strrep("─", 72), "\n", sep = "")
cat(sprintf("  §2 SUMMARY  |  %s / %s\n", DATASET, CELL_TYPE))
cat(strrep("─", 72), "\n", sep = "")
cat(sprintf("  Lists loaded            : %d (Sloan Table S9)\n",
            length(gene_lists)))
cat(sprintf("  Total unique genes      : %d (post-detection filter)\n",
            total_unique_genes))
cat(sprintf("  SenMayo overlap (vs Saul)   : %.1f%% (%d / %d)\n",
            senmayo_validation$overlap_pct, senmayo_validation$overlap_n,
            senmayo_validation$sloan_n))
cat(sprintf("  Fridman overlap (vs MSigDB) : %.1f%% (%d / %d)\n",
            fridman_validation$overlap_pct, fridman_validation$overlap_n,
            fridman_validation$sloan_n))
cat(sprintf("  Lists with <%d detected     : %d %s\n",
            MIN_GENES_FOR_SCORING,
            length(small_lists),
            if (length(small_lists) > 0) paste0("(", paste(small_lists, collapse = ", "), ")") else ""))
cat(strrep("─", 72), "\n", sep = "")
cat("\n✓ §2 gene list prep complete\n")
cat("  gene_lists, gene_list_meta, senmayo_validation, fridman_validation in scope.\n")
cat("  Next: §3 — cell cycle scoring + module scoring\n")

---
## 05 · Cell-cycle and module scoring

**Why.** Tirosh `CellCycleScoring` for the phase call, and `AddModuleScore` for the 10 Sloan panels. Both run on log-normalized expression, not on the Pearson residuals — residuals are for SenePy scoring only (module 02 layer contract).

**Defines for later sections.** `Phase`, `S.Score`, `G2M.Score` and one score column per Sloan list, all added to `md` · `phase_levels`

> **Source not in this rebuild.** The code for this section lives in
> `05_validation_v2.ipynb, cell 4` and was not carried over. Paste it in here — the sections below
> depend on the objects listed above and will stop without them.

In [ ]:
# =============================================================================
# MODULE 05 — Senescence Enrichment & Cell Cycle Analysis
# CELL CYCLE + MODULE SCORING (Cell §3)
# =============================================================================
# Computes per-cell scores that downstream §4–§7 statistical tests consume.
#
#   1. Cell cycle scoring (Tirosh et al., cc.genes.updated.2019)
#        → adds S.Score, G2M.Score, Phase to obj_ct@meta.data
#   2. Module scoring for the 10 Sloan gene lists (AddModuleScore, seed=42)
#        → adds Score_p53_Targets, Score_CellCycleArrest, ..., Score_Fridman_Up
#
# Saves the scored object as <celltype>_scored.qs. Once saved, downstream
# cells can be re-run iteratively without re-scoring.
#
# IMPORTANT: AddModuleScore + CellCycleScoring read from the data layer
# (log-normalized values), NOT counts. M04 §5's NormalizeData populated this
# layer; §1 verified it's present. Without it, scores would be library-depth-
# driven garbage (the M09 bug we fixed earlier).
#
# After §3 the in-memory state adds:
#   obj_ct          — now includes Phase, S.Score, G2M.Score, Score_* columns
#   md              — refreshed obj_ct@meta.data
#   score_cols      — character vector: c("Score_p53_Targets", ..., "Score_Fridman_Up")
#   phase_levels    — c("G1", "S", "G2M") for downstream factor ordering
# =============================================================================

cat("=", strrep("=", 71), "\n", sep = "")
cat(sprintf("§3 — CELL CYCLE + MODULE SCORING  |  %s / %s\n", DATASET, CELL_TYPE))
cat("=", strrep("=", 71), "\n", sep = "")

# Sanity checks
if (!exists("obj_ct"))     stop("✗ Run §1 first — obj_ct not in scope.")
if (!exists("gene_lists")) stop("✗ Run §2 first — gene_lists not in scope.")

# Verify data layer is populated (the bug we fixed in M09)
data_layer <- GetAssayData(obj_ct, layer = "data", assay = DefaultAssay(obj_ct))
data_max   <- max(data_layer[1:min(100, nrow(data_layer)),
                              1:min(100, ncol(data_layer))])
cat(sprintf("\n  Data layer max (top-left 100×100): %.3f\n", data_max))
if (data_max < 0.001) {
    stop("✗ Data layer appears empty or unnormalized.\n  ",
         "Re-run M04 §5 (NormalizeData) before this step.")
}
cat("  ✓ Data layer populated — module scoring will use log-normalized values\n")
rm(data_layer); gc(verbose = FALSE)


# ─────────────────────────────────────────────────────────────────────────────
# §3.1 Cell cycle scoring (Tirosh et al. 2016)
#
# Uses cc.genes.updated.2019$s.genes (43 genes) and $g2m.genes (54 genes).
# CellCycleScoring runs AddModuleScore internally with both gene sets and
# assigns each cell to the phase with the highest score (G1 = neither
# clearly elevated). Phase becomes a character column with levels G1/S/G2M.
# ─────────────────────────────────────────────────────────────────────────────
cat("\n▸ §3.1 Cell cycle scoring (Tirosh et al.)\n")

s_genes   <- cc.genes.updated.2019$s.genes
g2m_genes <- cc.genes.updated.2019$g2m.genes

s_detected   <- intersect(s_genes,   rownames(obj_ct))
g2m_detected <- intersect(g2m_genes, rownames(obj_ct))

cat(sprintf("  S-phase genes   : %d / %d detected\n",
            length(s_detected), length(s_genes)))
cat(sprintf("  G2M-phase genes : %d / %d detected\n",
            length(g2m_detected), length(g2m_genes)))

if (length(s_detected) < 5 || length(g2m_detected) < 5) {
    stop("✗ Too few cell-cycle marker genes detected — cell cycle scoring unreliable.")
}

obj_ct <- time_step("§3.1.run CellCycleScoring", {
    CellCycleScoring(
        obj_ct,
        s.features   = s_detected,
        g2m.features = g2m_detected,
        set.ident    = FALSE,
        seed         = STATISTICAL_PARAMS$seed
    )
})

# Verify the scoring landed
cc_added <- intersect(c("S.Score", "G2M.Score", "Phase"),
                     colnames(obj_ct@meta.data))
if (length(cc_added) != 3) {
    stop(sprintf("✗ Expected S.Score, G2M.Score, Phase — got: %s",
                 paste(cc_added, collapse = ", ")))
}

# Phase × Senescence orientation table
phase_levels <- c("G1", "S", "G2M")
obj_ct$Phase <- factor(obj_ct$Phase, levels = phase_levels)

cat("\n  Phase × Senescence crosstab (counts):\n")
phase_x_sen <- table(obj_ct$Phase,
                     obj_ct@meta.data[[SEN_LABEL_STR_COL]],
                     useNA = "ifany")
print(phase_x_sen)

cat("\n  Column proportions (% of SnC and Non-SnC in each phase):\n")
phase_x_sen_pct <- prop.table(phase_x_sen, margin = 2) * 100
print(round(phase_x_sen_pct, 1))


# ─────────────────────────────────────────────────────────────────────────────
# §3.2 Module scoring — 10 senescence gene lists
#
# AddModuleScore is called once per list with seed=42 for reproducibility.
# Seurat appends "1" to the user-provided name; we rename to drop the
# trailing digit so column names match the gene list names.
# ─────────────────────────────────────────────────────────────────────────────
cat("\n▸ §3.2 Module scoring (10 gene lists)\n")

score_cols      <- character(0)
score_summaries <- list()

for (nm in names(gene_lists)) {
    score_name <- paste0("Score_", nm)
    n_genes    <- length(gene_lists[[nm]])

    cat(sprintf("  %-22s  (%3d genes)...", nm, n_genes))

    if (n_genes < 5) {
        cat(" SKIPPED (< 5 detected genes)\n")
        next
    }

    obj_ct <- AddModuleScore(
        obj_ct,
        features = list(gene_lists[[nm]]),
        name     = score_name,
        ctrl     = min(100, n_genes),
        seed     = STATISTICAL_PARAMS$seed
    )

    # Seurat appends "1" — rename to canonical
    actual_col <- paste0(score_name, "1")
    if (actual_col %in% colnames(obj_ct@meta.data)) {
        idx <- which(colnames(obj_ct@meta.data) == actual_col)
        colnames(obj_ct@meta.data)[idx] <- score_name
    }

    if (!score_name %in% colnames(obj_ct@meta.data)) {
        cat(" FAILED — Score column not added\n")
        next
    }

    vals <- obj_ct@meta.data[[score_name]]
    cat(sprintf(" mean=%+.4f  sd=%.4f\n", mean(vals, na.rm=TRUE), sd(vals, na.rm=TRUE)))

    score_cols <- c(score_cols, score_name)
    score_summaries[[nm]] <- data.frame(
        list_name  = nm,
        score_col  = score_name,
        n_genes    = n_genes,
        mean_score = mean(vals, na.rm = TRUE),
        sd_score   = sd(vals,   na.rm = TRUE),
        stringsAsFactors = FALSE
    )
}

cat(sprintf("\n  ✓ %d module scores added\n", length(score_cols)))


# ─────────────────────────────────────────────────────────────────────────────
# §3.3 SnC vs Non-SnC mean module score (quick-look orientation table)
#
# This is descriptive, not a statistical test — §5 does the formal testing.
# Useful as a sanity check: SASP and CellCycleArrest should be elevated
# in SnC; effect sizes here predict §5's results.
# ─────────────────────────────────────────────────────────────────────────────
cat("\n▸ §3.3 SnC vs Non-SnC mean score (descriptive — formal testing in §5)\n")

snc_str <- as.character(obj_ct@meta.data[[SEN_LABEL_STR_COL]])
snc_mask    <- snc_str == "Senescent"
nonsnc_mask <- snc_str == "Non-senescent"

cat(sprintf("\n  %-22s %12s %12s %12s\n",
            "List", "SnC mean", "NonSnC mean", "Diff"))
cat("  ", strrep("-", 62), "\n", sep = "")

for (sc in score_cols) {
    vals       <- obj_ct@meta.data[[sc]]
    mean_snc   <- mean(vals[snc_mask],    na.rm = TRUE)
    mean_non   <- mean(vals[nonsnc_mask], na.rm = TRUE)
    diff_val   <- mean_snc - mean_non
    nm_short   <- sub("^Score_", "", sc)
    cat(sprintf("  %-22s %+12.4f %+12.4f %+12.4f\n",
                nm_short, mean_snc, mean_non, diff_val))
}


# ─────────────────────────────────────────────────────────────────────────────
# §3.4 Refresh md (used by all downstream cells)
# ─────────────────────────────────────────────────────────────────────────────
md <- as.data.frame(obj_ct@meta.data)
cat(sprintf("\n  ✓ md refreshed: %s × %d cols\n",
            fmt_n(nrow(md)), ncol(md)))


# ─────────────────────────────────────────────────────────────────────────────
# §3.5 Save scored .qs (so §4–§7 can resume from here without re-scoring)
# ─────────────────────────────────────────────────────────────────────────────
cat("\n▸ §3.5 Save scored .qs\n")

scored_bytes <- time_step("§3.5.qsave(obj_ct)", {
    qsave(obj_ct, PATHS$scored_qs, preset = "high")
    file.size(PATHS$scored_qs)
})

cat(sprintf("        path: %s\n", PATHS$scored_qs))
cat(sprintf("        size: %s\n", fmt_size(PATHS$scored_qs)))


# ─────────────────────────────────────────────────────────────────────────────
# §3.6 Section summary
# ─────────────────────────────────────────────────────────────────────────────
cat("\n", strrep("─", 72), "\n", sep = "")
cat(sprintf("  §3 SUMMARY  |  %s / %s\n", DATASET, CELL_TYPE))
cat(strrep("─", 72), "\n", sep = "")
cat(sprintf("  Cells × genes        : %s × %s\n",
            fmt_n(ncol(obj_ct)), fmt_n(nrow(obj_ct))))
cat(sprintf("  Cell cycle scoring   : added S.Score, G2M.Score, Phase\n"))
cat(sprintf("  Module scores added  : %d (Score_*)\n", length(score_cols)))
cat(sprintf("  Phases (counts)      : %s\n",
            paste(sprintf("%s=%s", levels(obj_ct$Phase),
                          fmt_n(table(obj_ct$Phase))), collapse = ", ")))
cat(sprintf("  Scored .qs           : %s  (%s)\n",
            basename(PATHS$scored_qs), fmt_size(PATHS$scored_qs)))
cat(strrep("─", 72), "\n", sep = "")
cat("\n✓ §3 cell cycle + module scoring complete\n")
cat("  Next: §4 — cell cycle enrichment testing (5 models × 3 strata × 3 phases)\n")

---
## 06 · Cell-cycle score diagnostic

**Why.** Phase-score distributions per stratum before any model runs. This is where the H1/H2 verdict is set that sections 08-13 refer back to.

**Defines for later sections.** printed diagnostic + saved distribution figure

> **Source not in this rebuild.** The code for this section lives in
> `05_validation_v2.ipynb, cell 5` and was not carried over. Paste it in here — the sections below
> depend on the objects listed above and will stop without them.

In [ ]:
# =============================================================================
# MODULE 05 — Senescence Enrichment & Cell Cycle Analysis
# DIAGNOSTIC: Cell cycle score distribution sanity check (Cell §3.5)
# =============================================================================
# §3 found unusual phase proportions for Astrocyte (G1 ~29%, S ~36%, G2M ~34%)
# vs. literature expectation for post-mitotic CNS glia (G1 should dominate
# ~80-95%). Two competing hypotheses:
#
#   H1 (NOISE):  Astrocytes are quiescent. S.Score and G2M.Score values are
#                near zero (median |score| < 0.05) with high overlap. Phase
#                assignments via "max score wins" are effectively dividing
#                noise by which side of zero it falls.
#                → Cell cycle results are uninterpretable as biology; lean
#                  on §7 proliferation markers for cell-cycle-arrest test.
#                → §4 still runs but reframed as between-group fraction
#                  test, not phase enrichment.
#
#   H2 (SIGNAL): Cell type has a genuine cycling subpopulation. S/G2M score
#                medians for assigned cells are large (>0.2 for S, >0.15 for
#                G2M) — comparable to known cycling cell lines.
#                → §4 results are interpretable as biology directly.
#
# IMPORTANT NOTE ABOUT WHAT NOT TO TEST
# -------------------------------------
# CellCycleScoring assigns phases by sign of S.Score / G2M.Score:
#   - S-phase   = max(S.Score, G2M.Score) > 0 AND S.Score > G2M.Score
#   - G2M-phase = max(S.Score, G2M.Score) > 0 AND G2M.Score > S.Score
#   - G1-phase  = S.Score ≤ 0 AND G2M.Score ≤ 0
# So "% of S-phase cells with S.Score > 0" is ALWAYS 100% (tautological).
# The right test is on score MAGNITUDES, not on signs. Real cycling cells
# show medians of 0.5-2.0; noise-driven assignments show 0.01-0.05.
# =============================================================================

cat("=", strrep("=", 71), "\n", sep = "")
cat(sprintf("§3.5 — DIAGNOSTIC: Cell cycle score distributions  |  %s\n", CELL_TYPE))
cat("=", strrep("=", 71), "\n", sep = "")

if (!exists("md")) stop("✗ Run §3 first — md not in scope.")
if (!"S.Score"   %in% colnames(md)) stop("✗ S.Score missing — run §3 first.")
if (!"G2M.Score" %in% colnames(md)) stop("✗ G2M.Score missing — run §3 first.")
if (!"Phase"     %in% colnames(md)) stop("✗ Phase missing — run §3 first.")


# ─────────────────────────────────────────────────────────────────────────────
# §3.5.1 Score distribution summary by Phase
# ─────────────────────────────────────────────────────────────────────────────
cat("\n▸ §3.5.1 Score summary by Phase\n")

cc_summary <- md %>%
    group_by(Phase) %>%
    summarise(
        n             = n(),
        S_mean        = mean(S.Score,   na.rm = TRUE),
        S_sd          = sd(S.Score,     na.rm = TRUE),
        S_median      = median(S.Score, na.rm = TRUE),
        G2M_mean      = mean(G2M.Score,   na.rm = TRUE),
        G2M_sd        = sd(G2M.Score,     na.rm = TRUE),
        G2M_median    = median(G2M.Score, na.rm = TRUE),
        .groups = "drop"
    )

cat("\n  Per-Phase score statistics:\n")
print(cc_summary)


# ─────────────────────────────────────────────────────────────────────────────
# §3.5.2 H1 vs H2 discrimination — based on score MAGNITUDE
#
# Reference points from literature for what "real cycling" looks like:
#   - Cycling immortal cell lines: S.Score median ~ 0.5-2.0
#   - Proliferating fetal/embryonic cells: S.Score median ~ 0.3-1.0
#   - Lightly cycling adult tissue: S.Score median ~ 0.1-0.3
#   - Quiescent / post-mitotic: S.Score median ~ 0.01-0.05  ← noise-dominated
#
# We use thresholds at 0.2 (S) and 0.15 (G2M) to call H2 — strong enough
# to distinguish from noise but not requiring tumor-line magnitudes.
# ─────────────────────────────────────────────────────────────────────────────
cat("\n▸ §3.5.2 H1 (noise) vs H2 (signal) discrimination — score magnitudes\n")

s_phase_cells   <- md[md$Phase == "S",   ]
g2m_phase_cells <- md[md$Phase == "G2M", ]

s_med   <- median(s_phase_cells$S.Score)
g2m_med <- median(g2m_phase_cells$G2M.Score)

# Distribution context: how many cells reach magnitude that would unambiguously
# indicate cycling biology?
s_pct_strong   <- 100 * mean(s_phase_cells$S.Score   > 0.2)
g2m_pct_strong <- 100 * mean(g2m_phase_cells$G2M.Score > 0.15)
s_pct_weak     <- 100 * mean(s_phase_cells$S.Score   > 0.05)
g2m_pct_weak   <- 100 * mean(g2m_phase_cells$G2M.Score > 0.05)

cat(sprintf("\n  S-phase cells (n=%s):\n", fmt_n(nrow(s_phase_cells))))
cat(sprintf("    S.Score median   = %+.4f   (H2 expects > +0.20, H1 typically < 0.05)\n",
            s_med))
cat(sprintf("    %% with S.Score > 0.05  = %.1f%%   (weak threshold, H2 expects most)\n",
            s_pct_weak))
cat(sprintf("    %% with S.Score > 0.20  = %.1f%%   (strong threshold, H2 expects substantial)\n",
            s_pct_strong))

cat(sprintf("\n  G2M-phase cells (n=%s):\n", fmt_n(nrow(g2m_phase_cells))))
cat(sprintf("    G2M.Score median = %+.4f   (H2 expects > +0.15, H1 typically < 0.05)\n",
            g2m_med))
cat(sprintf("    %% with G2M.Score > 0.05 = %.1f%%   (weak threshold)\n",
            g2m_pct_weak))
cat(sprintf("    %% with G2M.Score > 0.15 = %.1f%%   (strong threshold)\n",
            g2m_pct_strong))


# ─────────────────────────────────────────────────────────────────────────────
# §3.5.3 Cohen's d separability between Phase groups
#
# Note: Cohen's d here is large by construction (cells are partitioned by
# which score wins) so this is more of a sanity check than an H1/H2 test.
# Useful for quantifying how cleanly the algorithm partitioned the data.
# ─────────────────────────────────────────────────────────────────────────────
cat("\n▸ §3.5.3 Cohen's d (algorithm partitioning quality)\n")

cohens_d <- function(x, y) {
    if (length(x) < 2 || length(y) < 2) return(NA_real_)
    pooled_sd <- sqrt(((length(x) - 1) * var(x) + (length(y) - 1) * var(y)) /
                       (length(x) + length(y) - 2))
    if (pooled_sd == 0) return(NA_real_)
    (mean(x) - mean(y)) / pooled_sd
}

d_S_vs_rest   <- cohens_d(md$S.Score[md$Phase == "S"],
                          md$S.Score[md$Phase != "S"])
d_G2M_vs_rest <- cohens_d(md$G2M.Score[md$Phase == "G2M"],
                          md$G2M.Score[md$Phase != "G2M"])
d_G1_vs_S_on_S <- cohens_d(md$S.Score[md$Phase == "G1"],
                           md$S.Score[md$Phase == "S"])

cat(sprintf("\n  S.Score:   d(S-phase vs rest)     = %+.3f   (algorithm partitions well)\n",
            d_S_vs_rest))
cat(sprintf("  G2M.Score: d(G2M-phase vs rest)   = %+.3f   (algorithm partitions well)\n",
            d_G2M_vs_rest))
cat(sprintf("  S.Score:   d(G1-phase vs S-phase) = %+.3f\n",
            d_G1_vs_S_on_S))


# ─────────────────────────────────────────────────────────────────────────────
# §3.5.4 Verdict — based on score magnitude (the right test)
# ─────────────────────────────────────────────────────────────────────────────
cat("\n▸ §3.5.4 Heuristic verdict\n")

# Strong-signal H2: substantial fraction reach magnitudes > 0.2 / 0.15
strong_S_signal   <- (s_med   > 0.20) && (s_pct_strong   > 30)
strong_G2M_signal <- (g2m_med > 0.15) && (g2m_pct_strong > 30)

# Weak-signal H1/H2 boundary: medians under 0.05 = noise-dominated
noise_dominated_S   <- s_med   < 0.05
noise_dominated_G2M <- g2m_med < 0.05

if (strong_S_signal && strong_G2M_signal) {
    verdict <- "H2 (genuine cycling signal — magnitudes > 0.2/0.15)"
    advice  <- "Phase assignments biologically meaningful. Run §4 as designed and interpret directly."
} else if (noise_dominated_S && noise_dominated_G2M) {
    verdict <- "H1 (noise-dominated — score magnitudes < 0.05, far from cycling-cell range)"
    advice  <- paste0(
        "Phase assignments unreliable as biology in this cell type.\n",
        "      Recommend: §4 still runs but framed as between-group fraction\n",
        "      comparison (SnC vs Non-SnC), NOT phase enrichment. Lean on §7\n",
        "      proliferation markers (MKI67, TOP2A, etc.) for the rigorous\n",
        "      cell-cycle-arrest evidence."
    )
} else {
    verdict <- "MIXED (signal magnitude between cycling and quiescent populations)"
    advice  <- paste0(
        "Some cells may be genuinely cycling but most are at the noise-vs-signal\n",
        "      boundary. Run §4 between-group, interpret with care, cross-check §7."
    )
}

cat(sprintf("\n  Verdict : %s\n", verdict))
cat(sprintf("  Advice  : %s\n", advice))


# ─────────────────────────────────────────────────────────────────────────────
# §3.5.5 Diagnostic plots (unchanged from prior version)
# ─────────────────────────────────────────────────────────────────────────────
cat("\n▸ §3.5.5 Diagnostic plots\n")

score_df <- bind_rows(
    data.frame(score_type = "S.Score",
               value      = md$S.Score,
               Phase      = md$Phase,
               stringsAsFactors = FALSE),
    data.frame(score_type = "G2M.Score",
               value      = md$G2M.Score,
               Phase      = md$Phase,
               stringsAsFactors = FALSE)
)
score_df$score_type <- factor(score_df$score_type, levels = c("S.Score", "G2M.Score"))

# Reference lines for "real cycling" thresholds
ref_lines <- data.frame(
    score_type = factor(c("S.Score", "G2M.Score"),
                        levels = c("S.Score", "G2M.Score")),
    threshold  = c(0.20, 0.15),
    label      = c("H2 strong: 0.20", "H2 strong: 0.15")
)

p_dist <- ggplot(score_df, aes(x = value, fill = Phase)) +
    geom_density(alpha = 0.5, linewidth = 0.3) +
    geom_vline(xintercept = 0, linetype = "dashed",
              color = "grey30", linewidth = 0.4) +
    geom_vline(data = ref_lines, aes(xintercept = threshold),
              linetype = "dotted", color = "red", linewidth = 0.4) +
    facet_wrap(~ score_type, scales = "free", ncol = 2) +
    scale_fill_manual(values = PHASE_COLORS) +
    labs(title    = sprintf("Cell cycle score distributions by assigned Phase  (%s)", CELL_TYPE),
         subtitle = "Red dotted = H2 magnitude threshold (genuine cycling); 0 = phase assignment cutoff",
         x        = "Module score",
         y        = "Density") +
    theme_clean()

print(p_dist)
save_figure(p_dist, slug = paste0(CELL_TYPE, "_cellcycle_score_distributions"),
            width = 11, height = 5)

p_box <- ggplot(score_df, aes(x = Phase, y = value, fill = Phase)) +
    geom_violin(alpha = 0.5, linewidth = 0.3, scale = "width") +
    geom_boxplot(width = 0.15, outlier.size = 0.2, alpha = 0.7,
                fill = "white", color = "black", linewidth = 0.3) +
    geom_hline(yintercept = 0, linetype = "dashed",
              color = "grey30", linewidth = 0.4) +
    geom_hline(data = ref_lines, aes(yintercept = threshold),
              linetype = "dotted", color = "red", linewidth = 0.4) +
    facet_wrap(~ score_type, scales = "free_y", ncol = 2) +
    scale_fill_manual(values = PHASE_COLORS) +
    labs(title    = sprintf("Score by Phase — H1 vs H2 visual check  (%s)", CELL_TYPE),
         subtitle = "Red dotted = H2 cycling magnitude threshold",
         x        = NULL,
         y        = "Module score") +
    theme_clean() +
    theme(legend.position = "none")

print(p_box)
save_figure(p_box, slug = paste0(CELL_TYPE, "_cellcycle_score_boxplots"),
            width = 11, height = 5)


# ─────────────────────────────────────────────────────────────────────────────
# §3.5.6 Save the diagnostic summary table
# ─────────────────────────────────────────────────────────────────────────────
cc_diagnostic <- data.frame(
    metric            = c("S_phase_median_S.Score",
                          "G2M_phase_median_G2M.Score",
                          "S_phase_pct_score>0.05",
                          "G2M_phase_pct_score>0.05",
                          "S_phase_pct_score>0.20",
                          "G2M_phase_pct_score>0.15",
                          "d_S.Score(S-vs-rest)",
                          "d_G2M.Score(G2M-vs-rest)"),
    value             = c(s_med, g2m_med,
                          s_pct_weak, g2m_pct_weak,
                          s_pct_strong, g2m_pct_strong,
                          d_S_vs_rest, d_G2M_vs_rest),
    H2_threshold      = c(0.20, 0.15,
                          NA, NA,
                          30, 30,
                          NA, NA),
    pass_H2_threshold = c(s_med > 0.20, g2m_med > 0.15,
                          NA, NA,
                          s_pct_strong > 30, g2m_pct_strong > 30,
                          NA, NA),
    interpretation_note = c(
        "H2 cycling cells: 0.5-2.0; H1 noise: <0.05",
        "H2 cycling cells: 0.3-1.5; H1 noise: <0.05",
        "Substantial fraction expected if any cycling",
        "Substantial fraction expected if any cycling",
        "Strong threshold — clear cycling signal",
        "Strong threshold — clear cycling signal",
        "Algorithmic partitioning quality (not biology)",
        "Algorithmic partitioning quality (not biology)"
    ),
    stringsAsFactors  = FALSE
)
save_table(cc_diagnostic, paste0(CELL_TYPE, "_cellcycle_diagnostic"))

cat("\n", strrep("─", 72), "\n", sep = "")
cat(sprintf("  §3.5 SUMMARY  |  %s\n", CELL_TYPE))
cat(strrep("─", 72), "\n", sep = "")
cat(sprintf("  Verdict       : %s\n", verdict))
cat(sprintf("  S.Score median   in S-phase   = %+.4f\n", s_med))
cat(sprintf("  G2M.Score median in G2M-phase = %+.4f\n", g2m_med))
cat(sprintf("  Plots saved   : %s_cellcycle_score_distributions.{pdf,png,svg}\n", CELL_TYPE))
cat(sprintf("                  %s_cellcycle_score_boxplots.{pdf,png,svg}\n", CELL_TYPE))
cat(sprintf("  Table saved   : %s_cellcycle_diagnostic.csv\n", CELL_TYPE))
cat(strrep("─", 72), "\n", sep = "")
cat("\n✓ §3.5 diagnostic complete (revised heuristic — magnitude-based H1/H2 test)\n")
cat("  Verdict drives §4 framing: H1 → between-group fraction comparison\n")
cat("                              H2 → phase enrichment (direct biology)\n")

---
## 07 · Donor-arm helpers

**Why.** Every model in sections 08-21 opens the same way: slice to a stratum,
work out which donors contribute **both** a SnC and a Non-SnC arm with at least
`min_cells_per_group` cells, and keep only those. A donor with no senescent
cells cannot contribute a paired difference, and including it would silently
turn a paired contrast into an unpaired one.

**Contract.**

- `build_donor_arms(md, donor_col, sen_label_col, min_cells)` →
  one row per donor × arm with columns `donor`, `arm`, `n_cells`, `paired`
- `build_donor_meta(md, paired_ids, donor_col, sen_label_col, sex_col, age_col, cohort_col, has_cohort)` →
  one row per paired donor with `donor`, sex, age, cohort, and
  `delta_log10_umi` (the SnC − Non-SnC depth difference used as a covariate by
  the RLM/OLS/bootstrap arms)

> **Reconstructed from downstream usage,** not lifted — the original lives in
> `05_validation_v2.ipynb`, cell 6 (`§3.6 — DEFINE CROSS-CELL HELPERS`).
> Replace it with the original before running for the record.

In [ ]:
# =============================================================================
# MODULE 05 -- Senescence Enrichment & Cell Cycle Analysis
# CROSS-CELL HELPERS (Cell §3.6)
# =============================================================================
# Three helpers used by every model cell in §4 / §5 / §7.
#
#   build_donor_arms()    Per-donor SnC vs Non-SnC cell counts + paired flag
#   build_donor_meta()    Per-donor demographics + delta_log10_umi
#   tidy_lmm_term()       Standardize lmer/glmer term extraction
#
# Forest-plot rendering is now INLINE in each model cell (§4.2/§4.3/§4.4)
# so each cell is self-contained and readable end-to-end.
# Forest plots are dispatched on EFFECT_CONFIGS (§0) for consistent visual
# rules across model types.
# =============================================================================

cat("=", strrep("=", 71), "\n", sep = "")
cat("§3.6 -- DEFINE CROSS-CELL HELPERS\n")
cat("=", strrep("=", 71), "\n", sep = "")


# ─────────────────────────────────────────────────────────────────────────────
# build_donor_arms()
# ─────────────────────────────────────────────────────────────────────────────
build_donor_arms <- function(md_stratum, donor_col, sen_label_col, min_cells) {
    arms <- md_stratum %>%
        group_by(donor = .data[[donor_col]],
                 arm   = .data[[sen_label_col]]) %>%
        tally(name = "n_cells") %>%
        ungroup()

    pass_min <- arms %>%
        filter(n_cells >= min_cells) %>%
        group_by(donor) %>%
        filter(n_distinct(arm) == 2) %>%
        pull(donor) %>%
        unique()

    arms$paired <- arms$donor %in% pass_min
    arms
}


# ─────────────────────────────────────────────────────────────────────────────
# build_donor_meta()
# ─────────────────────────────────────────────────────────────────────────────
build_donor_meta <- function(md_stratum, paired_ids,
                             donor_col, sen_label_col,
                             sex_col, age_col, cohort_col,
                             has_cohort) {
    md_p <- md_stratum[md_stratum[[donor_col]] %in% paired_ids, , drop = FALSE]

    base_cols <- c(donor_col, sex_col, age_col)
    if (has_cohort && !is.null(cohort_col)) base_cols <- c(base_cols, cohort_col)

    demo <- md_p %>%
        group_by(.data[[donor_col]]) %>%
        summarise(across(all_of(setdiff(base_cols, donor_col)), first),
                  .groups = "drop") %>%
        rename(donor = !!donor_col)

    sen_vals <- as.character(md_p[[sen_label_col]])
    md_p$is_snc_arm <- sen_vals %in% c("1", "TRUE", "True", "Senescent")

    umi <- md_p %>%
        group_by(donor = .data[[donor_col]], is_snc = is_snc_arm) %>%
        summarise(mean_log10_umi = mean(log10(nCount_RNA), na.rm = TRUE),
                  .groups = "drop") %>%
        mutate(arm = ifelse(is_snc, "umi_snc", "umi_nonsnc")) %>%
        select(donor, arm, mean_log10_umi) %>%
        tidyr::pivot_wider(names_from = arm, values_from = mean_log10_umi)

    if (!"umi_snc"    %in% names(umi)) umi$umi_snc    <- NA_real_
    if (!"umi_nonsnc" %in% names(umi)) umi$umi_nonsnc <- NA_real_

    umi$delta_log10_umi <- umi$umi_snc - umi$umi_nonsnc

    out <- demo %>% left_join(umi, by = "donor")

    out[[sex_col]] <- as.factor(out[[sex_col]])
    if (has_cohort && !is.null(cohort_col)) {
        out[[cohort_col]] <- as.factor(out[[cohort_col]])
    }
    out
}


# ─────────────────────────────────────────────────────────────────────────────
# tidy_lmm_term()
# ─────────────────────────────────────────────────────────────────────────────
tidy_lmm_term <- function(fit, term_pattern, level = 0.95) {
    co <- summary(fit)$coefficients

    term_idx <- grep(term_pattern, rownames(co))
    if (length(term_idx) == 0) {
        return(data.frame(
            estimate = NA_real_, se = NA_real_,
            ci_low = NA_real_, ci_high = NA_real_,
            statistic = NA_real_, p_value = NA_real_
        ))
    }
    if (length(term_idx) > 1) term_idx <- term_idx[1]

    est <- co[term_idx, "Estimate"]
    se  <- co[term_idx, "Std. Error"]

    stat_col <- if ("t value" %in% colnames(co)) "t value"
                else if ("z value" %in% colnames(co)) "z value"
                else NA_character_
    statistic <- if (!is.na(stat_col)) co[term_idx, stat_col] else NA_real_

    pval_col <- if ("Pr(>|t|)" %in% colnames(co)) "Pr(>|t|)"
                else if ("Pr(>|z|)" %in% colnames(co)) "Pr(>|z|)"
                else NA_character_
    p_value <- if (!is.na(pval_col)) co[term_idx, pval_col] else NA_real_

    z <- qnorm(1 - (1 - level) / 2)
    ci_low  <- est - z * se
    ci_high <- est + z * se

    data.frame(
        estimate  = est,
        se        = se,
        ci_low    = ci_low,
        ci_high   = ci_high,
        statistic = statistic,
        p_value   = p_value
    )
}


# ─────────────────────────────────────────────────────────────────────────────
# Smoke test on 'All' stratum
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> Smoke test on 'All' stratum\n")

md_test <- filter_to_stratum(md, "All", STUDY_GROUP_COL)
arms_test <- build_donor_arms(md_test, DONOR_COL, SEN_LABEL_COL,
                              STATISTICAL_PARAMS$min_cells_per_group)
paired_ids_test <- unique(arms_test$donor[arms_test$paired])

cat(sprintf("  build_donor_arms: %d donor x arm rows, %d paired donors\n",
            nrow(arms_test), length(paired_ids_test)))

donor_meta_test <- build_donor_meta(
    md_test, paired_ids_test,
    DONOR_COL, SEN_LABEL_COL,
    SEX_COL, AGE_COL, COHORT_COL,
    has_cohort = HAS_COHORT
)
cat(sprintf("  build_donor_meta: %d paired donors x %d covariate cols\n",
            nrow(donor_meta_test), ncol(donor_meta_test)))
cat(sprintf("    columns: %s\n",
            paste(colnames(donor_meta_test), collapse = ", ")))
cat(sprintf("    Sex levels   : %s\n",
            paste(levels(donor_meta_test[[SEX_COL]]), collapse = ", ")))
if (HAS_COHORT) {
    cat(sprintf("    Cohort levels: %s\n",
                paste(levels(donor_meta_test[[COHORT_COL]]), collapse = ", ")))
}
cat(sprintf("    delta_log10_umi summary:\n"))
print(summary(donor_meta_test$delta_log10_umi))

rm(md_test, arms_test, paired_ids_test, donor_meta_test); gc(verbose = FALSE)

cat("\nv §3.6 helpers defined: build_donor_arms, build_donor_meta, tidy_lmm_term\n")
cat("  Smoke test on 'All' stratum: passed\n")
cat("  Forest plotting: INLINE in §4.2 / §4.3 / §4.4 (dispatched via EFFECT_CONFIGS)\n")
cat("  Next: §4.1 -- cell cycle Wilcoxon test\n")

---
## 08 · Cell cycle — Wilcoxon (donor-paired)

**Why.** The distribution-free anchor. Per donor, the proportion of cells called G1 / S / G2M in each arm; the contrast is the paired difference. No distributional assumption, no covariates — whatever the other four estimators do, this one is what the raw paired data say.

**Test.** Paired Wilcoxon signed-rank.  
**Outcome.** Per-donor proportion of cells in {G1, S, G2M}.  
**β.** Mean paired difference, `prop_SnC − prop_NonSnC` (probability points).  
**CI.** Hodges-Lehmann, from `wilcox.test(conf.int = TRUE)`.  
**FDR.** BH within stratum, across 3 phases.  

**Display.** Box + violin + jitter, faceted stratum × phase.

In [ ]:
# MODULE 05 -- Senescence Enrichment & Cell Cycle Analysis
# §4.1 -- CELL CYCLE: WILCOXON (donor-level paired)
# Stats:
#   Test:    Paired Wilcoxon signed-rank
#   Outcome: Per-donor proportion of cells in {G1, S, G2M}
#   beta:    mean paired difference (prop_SnC - prop_NonSnC)
#   Scale:   probability_pts
#   FDR:     BH within (stratum) across 3 phases
#
# Note (§3.5 H1 verdict for Astrocyte): phase calls noise-dominated;
# significant Wilcoxon p-values reflect noise distribution shifts, not
# biological enrichment. §7 proliferation markers carry the rigorous test.

cat("=", strrep("=", 71), "\n", sep = "")
cat(sprintf("§4.1 -- CELL CYCLE: WILCOXON  |  %s / %s\n", DATASET, CELL_TYPE))
cat("=", strrep("=", 71), "\n", sep = "")


# ─────────────────────────────────────────────────────────────────────────────
# §4.1.1 STATS
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> STATS\n")
cat("  Test     : Paired Wilcoxon signed-rank\n")
cat("  Outcome  : prop_SnC - prop_NonSnC (per donor, per phase)\n")
cat("  beta     : mean paired difference\n")
cat("  Scale    : probability_pts\n")
cat(sprintf("  Pairing  : within-donor (n_pairs = paired donors per stratum)\n"))
cat(sprintf("  FDR      : %s within (stratum) across 3 phases\n",
            STATISTICAL_PARAMS$fdr_method))
cat(sprintf("  Strata   : %s\n", paste(strata, collapse = ", ")))
cat(sprintf("  Min cells per arm: %d\n", STATISTICAL_PARAMS$min_cells_per_group))


# ─────────────────────────────────────────────────────────────────────────────
# §4.1.2 RUN
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> RUN\n")

cc_wilcox_rows <- list()

for (stratum in strata) {
    md_s        <- filter_to_stratum(md, stratum, STUDY_GROUP_COL)
    arms_s      <- build_donor_arms(md_s, DONOR_COL, SEN_LABEL_COL,
                                    STATISTICAL_PARAMS$min_cells_per_group)
    paired_ids  <- unique(arms_s$donor[arms_s$paired])
    n_paired    <- length(paired_ids)

    if (n_paired < 5) {
        cat(sprintf("  [%s] only %d paired donors -- skipping\n",
                    stratum, n_paired))
        next
    }

    md_pair <- md_s[md_s[[DONOR_COL]] %in% paired_ids, , drop = FALSE]
    md_pair$is_snc_arm <- as.character(md_pair[[SEN_LABEL_COL]]) %in%
                          c("1", "TRUE", "True", "Senescent")

    counts_df <- md_pair %>%
        group_by(donor   = .data[[DONOR_COL]],
                 is_snc  = is_snc_arm,
                 Phase   = Phase) %>%
        tally(name = "n") %>%
        ungroup() %>%
        group_by(donor, is_snc) %>%
        mutate(prop = n / sum(n)) %>%
        ungroup()

    for (phase in phase_levels) {
        wide_df <- counts_df %>%
            filter(Phase == phase) %>%
            select(donor, is_snc, prop) %>%
            tidyr::pivot_wider(names_from   = is_snc,
                              values_from  = prop,
                              names_prefix = "prop_")

        if (!"prop_TRUE"  %in% names(wide_df)) wide_df$prop_TRUE  <- 0
        if (!"prop_FALSE" %in% names(wide_df)) wide_df$prop_FALSE <- 0
        wide_df$prop_TRUE[is.na(wide_df$prop_TRUE)]   <- 0
        wide_df$prop_FALSE[is.na(wide_df$prop_FALSE)] <- 0

        snc_props    <- wide_df$prop_TRUE
        nonsnc_props <- wide_df$prop_FALSE
        diffs        <- snc_props - nonsnc_props

        wt <- tryCatch({
            wilcox.test(snc_props, nonsnc_props, paired = TRUE,
                        exact = FALSE, conf.int = TRUE)
        }, error = function(e) NULL)

        if (is.null(wt)) {
            cat(sprintf("  [%s | %s] Wilcoxon failed\n", stratum, phase))
            next
        }

        n_obs           <- sum(diffs != 0)
        W_stat          <- as.numeric(wt$statistic)
        W_max           <- if (n_obs > 0) n_obs * (n_obs + 1) / 2 else NA_real_
        rank_biserial_r <- if (!is.na(W_max) && W_max > 0) (2 * W_stat / W_max) - 1 else NA_real_
        cohens_d        <- if (sd(diffs) > 0) mean(diffs) / sd(diffs) else NA_real_
        pct_higher      <- 100 * mean(diffs > 0)

        cell_cnt_snc <- sum(arms_s$n_cells[arms_s$donor %in% paired_ids &
                                           arms_s$arm %in% c("1", "TRUE", "True", "Senescent")])
        cell_cnt_non <- sum(arms_s$n_cells[arms_s$donor %in% paired_ids &
                                           !(arms_s$arm %in% c("1", "TRUE", "True", "Senescent"))])

        row <- tidy_model_results(
            stratum      = stratum,
            outcome      = phase,
            model        = "wilcoxon",
            n_donors     = n_paired,
            n_cells_test = cell_cnt_snc,
            n_cells_ref  = cell_cnt_non,
            estimate     = mean(diffs),
            se           = sd(diffs) / sqrt(n_paired),
            ci_low       = if (!is.null(wt$conf.int)) wt$conf.int[1] else NA_real_,
            ci_high      = if (!is.null(wt$conf.int)) wt$conf.int[2] else NA_real_,
            statistic    = W_stat,
            p_value      = wt$p.value,
            extra        = list(
                beta_scale        = "probability_pts",
                median_diff       = median(diffs),
                mean_snc          = mean(snc_props),
                mean_nonsnc       = mean(nonsnc_props),
                rank_biserial_r   = rank_biserial_r,
                cohens_d          = cohens_d,
                pct_donors_higher = pct_higher
            )
        )
        cc_wilcox_rows[[paste(stratum, phase, sep = "|")]] <- row
    }
}

cc_wilcox_df <- bind_rows(cc_wilcox_rows)


# ─────────────────────────────────────────────────────────────────────────────
# §4.1.3 BH-FDR within (stratum)
# ─────────────────────────────────────────────────────────────────────────────
cc_wilcox_df <- cc_wilcox_df %>%
    group_by(stratum) %>%
    mutate(p_adj = p.adjust(p_value, method = STATISTICAL_PARAMS$fdr_method)) %>%
    ungroup() %>%
    mutate(sig = sig_stars(p_adj))


# ─────────────────────────────────────────────────────────────────────────────
# §4.1.4 RESULTS
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> RESULTS  (beta = mean paired difference, scale = probability_pts)\n")

cat(sprintf("\n  %-22s %-5s %5s %8s %8s %+9s %+9s %8s %8s %4s\n",
            "Stratum", "Phase", "N", "SnC%", "Non%",
            "beta(%)", "Cohen-d", "p_raw", "p_adj", "Sig"))
cat("  ", strrep("-", 96), "\n", sep = "")

for (i in seq_len(nrow(cc_wilcox_df))) {
    r <- cc_wilcox_df[i, ]
    cat(sprintf("  %-22s %-5s %5d %7.1f%% %7.1f%% %+8.2f%% %+9.3f %8.1e %8.1e %4s\n",
                substr(r$stratum, 1, 22), r$outcome, r$n_donors,
                r$mean_snc * 100, r$mean_nonsnc * 100, r$estimate * 100,
                r$cohens_d, r$p_value, r$p_adj, r$sig))
}

save_table(cc_wilcox_df, paste0(CELL_TYPE, "_cellcycle_wilcoxon"))


# ─────────────────────────────────────────────────────────────────────────────
# §4.1.5 FIGURE -- box+violin, SnC red / Non-SnC grey, colored strips
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> FIGURE\n")

if (!requireNamespace("ggh4x", quietly = TRUE)) {
    cat("  Installing ggh4x...\n")
    install.packages("ggh4x", repos = "https://cloud.r-project.org")
}
suppressPackageStartupMessages(library(ggh4x))

# Build per-donor plot data
plot_rows <- list()
for (stratum in strata) {
    md_s <- filter_to_stratum(md, stratum, STUDY_GROUP_COL)
    arms_s <- build_donor_arms(md_s, DONOR_COL, SEN_LABEL_COL,
                              STATISTICAL_PARAMS$min_cells_per_group)
    paired_ids <- unique(arms_s$donor[arms_s$paired])
    if (length(paired_ids) < 5) next

    md_pair <- md_s[md_s[[DONOR_COL]] %in% paired_ids, , drop = FALSE]
    md_pair$is_snc_arm <- as.character(md_pair[[SEN_LABEL_COL]]) %in%
                          c("1", "TRUE", "True", "Senescent")
    md_pair$arm_label  <- ifelse(md_pair$is_snc_arm, "SnC", "Non-SnC")

    plot_data <- md_pair %>%
        group_by(donor     = .data[[DONOR_COL]],
                 arm_label = arm_label,
                 Phase     = Phase) %>%
        tally(name = "n") %>%
        ungroup() %>%
        group_by(donor, arm_label) %>%
        mutate(prop = n / sum(n)) %>%
        ungroup() %>%
        mutate(stratum = stratum)

    plot_rows[[stratum]] <- plot_data
}
plot_df <- bind_rows(plot_rows)
plot_df$stratum   <- factor(plot_df$stratum,   levels = strata)
plot_df$arm_label <- factor(plot_df$arm_label, levels = c("Non-SnC", "SnC"))
plot_df$Phase     <- factor(plot_df$Phase,     levels = phase_levels)

# Annotations
sig_annot <- cc_wilcox_df %>%
    select(stratum, Phase = outcome, sig, p_adj, estimate) %>%
    mutate(stratum = factor(stratum, levels = strata),
           Phase   = factor(Phase,   levels = phase_levels))

annot_y <- plot_df %>%
    group_by(stratum, Phase) %>%
    summarise(y = max(prop, na.rm = TRUE) * 1.10, .groups = "drop")

sig_annot <- sig_annot %>% left_join(annot_y, by = c("stratum", "Phase"))

# Strip palettes
phase_strip_palette   <- PHASE_COLORS[phase_levels]
stratum_strip_palette <- sapply(strata, function(s) {
    if (s %in% names(STUDY_GROUP_COLORS)) STUDY_GROUP_COLORS[[s]] else "#7F7F7F"
})
names(stratum_strip_palette) <- strata

text_color_for_bg <- function(hex) {
    rgb_vals <- col2rgb(hex)
    luminance <- 0.299 * rgb_vals[1, ] + 0.587 * rgb_vals[2, ] + 0.114 * rgb_vals[3, ]
    ifelse(luminance < 140, "white", "black")
}
phase_text_colors   <- text_color_for_bg(phase_strip_palette)
stratum_text_colors <- text_color_for_bg(stratum_strip_palette)

p_cc_wilcox <- ggplot(plot_df,
                      aes(x = arm_label, y = prop, fill = arm_label)) +
    geom_violin(alpha = 0.5, linewidth = 0.3, scale = "width") +
    geom_boxplot(width = 0.18, outlier.shape = NA, alpha = 0.7,
                fill = "white", color = "black", linewidth = 0.3) +
    geom_jitter(width = 0.12, size = 0.4, alpha = 0.4, color = "grey30") +
    geom_text(data = sig_annot,
             aes(x = 1.5, y = y, label = sig),
             inherit.aes = FALSE,
             size = 4, fontface = "bold") +
    geom_text(data = sig_annot,
             aes(x = 1.5, y = y * 0.94,
                 label = sprintf("beta=%+.3f", estimate)),
             inherit.aes = FALSE,
             size = 2.5, color = "grey30") +
    scale_fill_manual(values = c("Non-SnC" = "#D3D3D3", "SnC" = "#C44E52")) +
    scale_y_continuous(labels = scales::percent_format(accuracy = 1)) +
    facet_grid2(stratum ~ Phase,
               strip = strip_themed(
                   background_x = elem_list_rect(
                       fill  = unname(phase_strip_palette),
                       color = "black"
                   ),
                   text_x = elem_list_text(
                       color = unname(phase_text_colors),
                       face  = "bold"
                   ),
                   background_y = elem_list_rect(
                       fill  = unname(stratum_strip_palette),
                       color = "black"
                   ),
                   text_y = elem_list_text(
                       color = unname(stratum_text_colors),
                       face  = "bold"
                   )
               )) +
    labs(title    = sprintf("Cell cycle phase fractions -- Wilcoxon (paired) -- %s",
                            CELL_TYPE),
         subtitle = "Top strips = Phase. Right strips = Stratum. Inside: SnC red / Non-SnC grey. *,**,*** = BH-FDR. H1 caveat (§3.5).",
         x        = NULL,
         y        = "Per-donor cell-fraction in phase",
         fill     = NULL) +
    theme_clean() +
    theme(legend.position = "top")

print(p_cc_wilcox)
save_figure(p_cc_wilcox,
            slug   = paste0(CELL_TYPE, "_cellcycle_wilcoxon"),
            width  = 9,
            height = 3 + length(strata) * 2)


# ─────────────────────────────────────────────────────────────────────────────
# §4.1.6 SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
n_sig_rows <- sum(cc_wilcox_df$p_adj < STATISTICAL_PARAMS$fdr_threshold,
                 na.rm = TRUE)

cat("\n", strrep("-", 72), "\n", sep = "")
cat(sprintf("  §4.1 SUMMARY  |  %s / %s\n", DATASET, CELL_TYPE))
cat(strrep("-", 72), "\n", sep = "")
cat(sprintf("  Test         : Paired Wilcoxon signed-rank\n"))
cat(sprintf("  beta_scale   : probability_pts\n"))
cat(sprintf("  Strata x Phs : %d rows total (%d strata x %d phases)\n",
            nrow(cc_wilcox_df), length(strata), length(phase_levels)))
cat(sprintf("  Significant  : %d at BH-FDR < %.2f\n",
            n_sig_rows, STATISTICAL_PARAMS$fdr_threshold))
cat(strrep("-", 72), "\n", sep = "")
cat("\nv §4.1 Wilcoxon complete\n")
cat("  Next: §4.2 -- RLM on paired diffs (covariate-adjusted)\n")

---
## 09 · Cell cycle — RLM on paired differences

**Why.** Same paired differences, now adjusted for the covariates that could manufacture one: sex, cohort, and the SnC − Non-SnC depth difference. `lmrob` because a single outlying donor should not set the estimate.

**Test.** Robust LM, `lmrob` MM-estimator, `setting = "KS2014"`.  
**Outcome.** Donor-level paired difference in phase proportion.  
**Formula.** `diff ~ Sex + Cohort + delta_log10_umi`  
**β.** Intercept — the adjusted mean paired difference.  
**FDR.** BH within stratum, across 3 phases.  

**Display.** Per-stratum forest, layout derived from the data.

In [ ]:
# §4.2 -- CELL CYCLE: RLM (paired diffs, covariate-adjusted)

cat("=", strrep("=", 71), "\n", sep = "")
cat(sprintf("§4.2 -- CELL CYCLE: RLM (paired diffs)  |  %s / %s\n",
            DATASET, CELL_TYPE))
cat("=", strrep("=", 71), "\n", sep = "")


# ─────────────────────────────────────────────────────────────────────────────
# §4.2.1 STATS
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> STATS\n")
cat("  Test     : Robust LM via lmrob (MM-estimator, KS2014)\n")
cat("  Formula  : (prop_SnC - prop_NonSnC) ~ Sex + Cohort + delta_log10_umi\n")
cat("  beta     : intercept (= mean paired diff after covariate adjustment)\n")
cat("  Scale    : probability_pts\n")
cat(sprintf("  Strata   : %s\n", paste(strata, collapse = ", ")))
cat(sprintf("  FDR      : %s within (stratum) across 3 phases\n",
            STATISTICAL_PARAMS$fdr_method))


# ─────────────────────────────────────────────────────────────────────────────
# §4.2.2 RUN
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> RUN\n")

cc_rlm_rows <- list()

for (stratum in strata) {
    md_s        <- filter_to_stratum(md, stratum, STUDY_GROUP_COL)
    arms_s      <- build_donor_arms(md_s, DONOR_COL, SEN_LABEL_COL,
                                    STATISTICAL_PARAMS$min_cells_per_group)
    paired_ids  <- unique(arms_s$donor[arms_s$paired])
    n_paired    <- length(paired_ids)

    if (n_paired < 5) {
        cat(sprintf("  [%s] only %d paired donors -- skipping\n",
                    stratum, n_paired))
        next
    }

    donor_meta_s <- build_donor_meta(
        md_s, paired_ids,
        DONOR_COL, SEN_LABEL_COL,
        SEX_COL, AGE_COL, COHORT_COL,
        has_cohort = HAS_COHORT
    )

    md_pair <- md_s[md_s[[DONOR_COL]] %in% paired_ids, , drop = FALSE]
    md_pair$is_snc_arm <- as.character(md_pair[[SEN_LABEL_COL]]) %in%
                          c("1", "TRUE", "True", "Senescent")

    counts_df <- md_pair %>%
        group_by(donor   = .data[[DONOR_COL]],
                 is_snc  = is_snc_arm,
                 Phase   = Phase) %>%
        tally(name = "n") %>%
        ungroup() %>%
        group_by(donor, is_snc) %>%
        mutate(prop = n / sum(n)) %>%
        ungroup()

    for (phase in phase_levels) {
        wide_df <- counts_df %>%
            filter(Phase == phase) %>%
            select(donor, is_snc, prop) %>%
            tidyr::pivot_wider(names_from   = is_snc,
                              values_from  = prop,
                              names_prefix = "prop_")

        if (!"prop_TRUE"  %in% names(wide_df)) wide_df$prop_TRUE  <- 0
        if (!"prop_FALSE" %in% names(wide_df)) wide_df$prop_FALSE <- 0
        wide_df$prop_TRUE[is.na(wide_df$prop_TRUE)]   <- 0
        wide_df$prop_FALSE[is.na(wide_df$prop_FALSE)] <- 0

        wide_df$diff <- wide_df$prop_TRUE - wide_df$prop_FALSE

        reg_df <- wide_df %>% left_join(donor_meta_s, by = "donor")

        sex_levels    <- length(unique(reg_df[[SEX_COL]]))
        cohort_levels <- if (HAS_COHORT) length(unique(reg_df[[COHORT_COL]])) else 0

        rhs <- character(0)
        if (sex_levels    >= 2) rhs <- c(rhs, SEX_COL)
        if (cohort_levels >= 2) rhs <- c(rhs, COHORT_COL)
        rhs <- c(rhs, "delta_log10_umi")

        formula_str <- paste("diff ~", paste(rhs, collapse = " + "))
        formula_obj <- as.formula(formula_str)

        fit <- tryCatch({
            robustbase::lmrob(formula_obj, data = reg_df, setting = "KS2014")
        }, error = function(e) {
            cat(sprintf("  [%s | %s] lmrob ERROR: %s\n",
                        stratum, phase, e$message))
            NULL
        }, warning = function(w) {
            tryCatch(robustbase::lmrob(formula_obj, data = reg_df,
                                       setting = "KS2014"),
                    error = function(e) NULL)
        })

        if (is.null(fit)) next

        co <- summary(fit)$coefficients

        if (!"(Intercept)" %in% rownames(co)) {
            cat(sprintf("  [%s | %s] No intercept in fit -- skipping\n",
                        stratum, phase))
            next
        }

        beta      <- co["(Intercept)", "Estimate"]
        se        <- co["(Intercept)", "Std. Error"]
        statistic <- co["(Intercept)", "t value"]
        p_value   <- co["(Intercept)", "Pr(>|t|)"]

        z <- qnorm(1 - (1 - STATISTICAL_PARAMS$confidence_level) / 2)
        ci_low  <- beta - z * se
        ci_high <- beta + z * se

        cell_cnt_snc <- sum(arms_s$n_cells[arms_s$donor %in% paired_ids &
                                           arms_s$arm %in% c("1", "TRUE", "True", "Senescent")])
        cell_cnt_non <- sum(arms_s$n_cells[arms_s$donor %in% paired_ids &
                                           !(arms_s$arm %in% c("1", "TRUE", "True", "Senescent"))])

        row <- tidy_model_results(
            stratum      = stratum,
            outcome      = phase,
            model        = "rlm",
            n_donors     = n_paired,
            n_cells_test = cell_cnt_snc,
            n_cells_ref  = cell_cnt_non,
            estimate     = beta,
            se           = se,
            ci_low       = ci_low,
            ci_high      = ci_high,
            statistic    = statistic,
            p_value      = p_value,
            extra        = list(
                beta_scale         = "probability_pts",
                formula            = formula_str,
                cohort_in_model    = cohort_levels >= 2,
                sex_in_model       = sex_levels >= 2,
                delta_umi_in_model = TRUE,
                converged          = isTRUE(fit$converged),
                n_covariate_terms  = nrow(co) - 1
            )
        )
        cc_rlm_rows[[paste(stratum, phase, sep = "|")]] <- row
    }
}

cc_rlm_df <- bind_rows(cc_rlm_rows)


# ─────────────────────────────────────────────────────────────────────────────
# §4.2.3 BH-FDR within (stratum)
# ─────────────────────────────────────────────────────────────────────────────
cc_rlm_df <- cc_rlm_df %>%
    group_by(stratum) %>%
    mutate(p_adj = p.adjust(p_value, method = STATISTICAL_PARAMS$fdr_method)) %>%
    ungroup() %>%
    mutate(sig = sig_stars(p_adj))


# ─────────────────────────────────────────────────────────────────────────────
# §4.2.4 RESULTS
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> RESULTS  (beta = adjusted intercept, scale = probability_pts)\n")

cat(sprintf("\n  %-22s %-5s %5s %+10s %+10s %+10s %8s %8s %4s\n",
            "Stratum", "Phase", "N",
            "beta", "CI_low", "CI_high",
            "p_raw", "p_adj", "Sig"))
cat("  ", strrep("-", 92), "\n", sep = "")

for (i in seq_len(nrow(cc_rlm_df))) {
    r <- cc_rlm_df[i, ]
    cat(sprintf("  %-22s %-5s %5d %+9.4f %+9.4f %+9.4f %8.1e %8.1e %4s\n",
                substr(r$stratum, 1, 22), r$outcome, r$n_donors,
                r$estimate, r$ci_low, r$ci_high,
                r$p_value, r$p_adj, r$sig))
}

save_table(cc_rlm_df, paste0(CELL_TYPE, "_cellcycle_rlm"))


# ─────────────────────────────────────────────────────────────────────────────
# §4.2.5 FIGURE -- compact per-stratum table-style forest plot
#
# Tightened layout:
#   - 7 x 2.0 inches at 3 rows (was 9 x 3.5)
#   - subtitle removed (formula info is in §4.2.1 STATS section)
#   - tighter margins, smaller header offsets, smaller title
#   - all colors are 6-digit hex (R 4.3.1 + grid requirement)
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> FIGURE\n")

cfg <- EFFECT_CONFIGS$probability_pts

for (s in strata) {
    df_s <- cc_rlm_df %>%
        filter(stratum == s) %>%
        arrange(desc(abs(estimate)))

    if (nrow(df_s) == 0) {
        cat(sprintf("  [%s] no rows -- skipping figure\n", s))
        next
    }

    df_s$row_label  <- as.character(df_s$outcome)
    df_s$eff_str    <- sapply(df_s$estimate, cfg$fmt_effect)
    df_s$ci_str     <- mapply(cfg$fmt_ci, df_s$ci_low, df_s$ci_high)
    df_s$p_str      <- sapply(df_s$p_adj, fmt_p_short)
    df_s$row_color  <- PHASE_COLORS[df_s$outcome]
    df_s$is_sig     <- df_s$sig != "ns"

    n_rows <- nrow(df_s)
    df_s$y <- n_rows:1
    header_y    <- n_rows + 0.55     # tighter (was 0.7)
    underline_y <- n_rows + 0.20     # tighter (was 0.25)
    y_lim <- c(0.4, n_rows + 1.0)    # tighter (was 0.3 .. n_rows+1.3)

    stratum_color <- if (s %in% names(STUDY_GROUP_COLORS)) STUDY_GROUP_COLORS[[s]] else "#7F7F7F"

    n_sig <- sum(df_s$is_sig)
    title_txt <- sprintf("Cell cycle RLM | %s | [%s] | %d/%d sig at BH-FDR",
                         CELL_TYPE, s, n_sig, nrow(df_s))

    p_left <- ggplot(df_s) +
        geom_text(aes(x = 0.05, y = y, label = row_label,
                     color = row_color,
                     fontface = ifelse(is_sig, "bold", "plain")),
                 hjust = 0, size = 2.4) +
        geom_text(aes(x = 0.55, y = y, label = eff_str,
                     fontface = ifelse(is_sig, "bold", "plain")),
                 hjust = 0, size = 2.2, family = "mono", color = "#222222") +
        geom_text(aes(x = 0.78, y = y, label = ci_str),
                 hjust = 0, size = 2.0, family = "mono", color = "#666666") +
        annotate("text", x = 0.05, y = header_y, label = "Phase",
                fontface = "bold", hjust = 0, size = 2.4, color = "#222222") +
        annotate("text", x = 0.55, y = header_y, label = cfg$eff_h_label,
                fontface = "bold", hjust = 0, size = 2.4, color = "#222222") +
        annotate("text", x = 0.78, y = header_y, label = cfg$ci_h_label,
                fontface = "bold", hjust = 0, size = 2.2, color = "#222222") +
        annotate("segment", x = 0, xend = 1.0,
                y = underline_y, yend = underline_y,
                color = "#333333", linewidth = 0.3) +
        scale_color_identity() +
        scale_x_continuous(limits = c(0, 1), expand = c(0, 0)) +
        scale_y_continuous(limits = y_lim, expand = c(0, 0)) +
        labs(title = title_txt) +
        theme_void() +
        theme(
            plot.title = element_text(size = 9, face = "bold",
                                     color = stratum_color,
                                     hjust = 0,
                                     margin = margin(t = 2, b = 1)),
            plot.margin = margin(2, 2, 2, 4)
        )

    eff_vals    <- c(df_s$estimate, df_s$ci_low, df_s$ci_high)
    eff_finite  <- eff_vals[is.finite(eff_vals)]
    if (length(eff_finite) == 0) eff_finite <- c(-0.2, 0.2)
    pad         <- max(diff(range(eff_finite)) * 0.15, 0.05)
    x_range     <- range(eff_finite) + c(-pad, pad)
    x_range[1]  <- min(x_range[1], cfg$null_value - 0.05)
    x_range[2]  <- max(x_range[2], cfg$null_value + 0.05)

    p_forest <- ggplot(df_s) +
        geom_vline(xintercept = cfg$null_value,
                  linetype = "dashed", color = "#999999", linewidth = 0.4) +
        geom_errorbar(aes(y = y, xmin = ci_low, xmax = ci_high),
                     width = 0.18, linewidth = 0.4, color = "#4D4D4D") +
        geom_point(aes(x = estimate, y = y, fill = row_color,
                      size = ifelse(is_sig, 3.5, 2.5)),
                  shape = 23, color = "#222222", stroke = 0.4) +
        scale_fill_identity() +
        scale_size_identity() +
        scale_x_continuous(limits = x_range,
                          labels = cfg$axis_format,
                          breaks = scales::breaks_pretty(n = 4)) +
        scale_y_continuous(limits = y_lim, expand = c(0, 0)) +
        labs(x = cfg$x_label, y = NULL) +
        theme_classic(base_size = 7) +
        theme(
            axis.title.x = element_text(size = 6.5, margin = margin(t = 1)),
            axis.text.x  = element_text(size = 5.5),
            axis.text.y  = element_blank(),
            axis.ticks.y = element_blank(),
            axis.line.y  = element_blank(),
            axis.line.x  = element_line(color = "#444444", linewidth = 0.4),
            panel.grid   = element_blank(),
            plot.margin  = margin(1, 2, 1, 2)
        )

    p_right <- ggplot(df_s) +
        geom_text(aes(x = 0.25, y = y, label = p_str,
                     fontface = ifelse(is_sig, "bold", "plain"),
                     color    = ifelse(is_sig, "#222222", "#666666")),
                 hjust = 0.5, size = 2.2, family = "mono") +
        geom_text(aes(x = 0.75, y = y, label = sig),
                 fontface = "bold", hjust = 0.5, size = 2.4,
                 family = "mono", color = "#222222") +
        annotate("text", x = 0.25, y = header_y, label = "p(adj)",
                fontface = "bold", hjust = 0.5, size = 2.4, color = "#222222") +
        annotate("text", x = 0.75, y = header_y, label = "Sig",
                fontface = "bold", hjust = 0.5, size = 2.4, color = "#222222") +
        annotate("segment", x = 0, xend = 1.0,
                y = underline_y, yend = underline_y,
                color = "#333333", linewidth = 0.3) +
        scale_color_identity() +
        scale_x_continuous(limits = c(0, 1), expand = c(0, 0)) +
        scale_y_continuous(limits = y_lim, expand = c(0, 0)) +
        theme_void() +
        theme(plot.margin = margin(2, 4, 2, 2))

    composed <- (p_left | p_forest | p_right) +
        plot_layout(widths = c(3, 3.5, 1.5))

    fig_height <- max(2.0, 0.9 + n_rows * 0.30)   # 3 rows -> 2.0"
    slug <- sprintf("%s_cellcycle_rlm_%s_forest", CELL_TYPE, s)

    save_figure(composed, slug = slug, width = 7, height = fig_height)

    options(repr.plot.width = 8, repr.plot.height = fig_height + 0.5)
    tryCatch(
        print(composed),
        error = function(e) {
            cat(sprintf("    [inline preview unavailable: %s]\n",
                        conditionMessage(e)))
        }
    )
}


# ─────────────────────────────────────────────────────────────────────────────
# §4.2.6 SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
n_sig_rows <- sum(cc_rlm_df$p_adj < STATISTICAL_PARAMS$fdr_threshold,
                 na.rm = TRUE)

cat("\n", strrep("-", 72), "\n", sep = "")
cat(sprintf("  §4.2 SUMMARY  |  %s / %s\n", DATASET, CELL_TYPE))
cat(strrep("-", 72), "\n", sep = "")
cat(sprintf("  Test          : Robust LM (lmrob, MM-estimator, KS2014)\n"))
cat(sprintf("  beta_scale    : probability_pts\n"))
cat(sprintf("  Adjustments   : Sex + Cohort + delta_log10_umi\n"))
cat(sprintf("  Significant   : %d of %d at BH-FDR < %.2f\n",
            n_sig_rows, nrow(cc_rlm_df), STATISTICAL_PARAMS$fdr_threshold))
cat(sprintf("  Per-stratum forests:\n"))
for (s in strata) {
    cat(sprintf("    %-22s -> %s_cellcycle_rlm_%s_forest.{pdf,png,svg}\n",
                s, CELL_TYPE, s))
}
cat(strrep("-", 72), "\n", sep = "")
cat("\nv §4.2 RLM complete\n")
cat("  Next: §4.3 -- OLS on paired diffs (classical, same formula)\n")

---
## 10 · Cell cycle — OLS on paired differences

**Why.** The same model without the robustness. Run alongside section 09 so that a disagreement between them localises the problem to influential donors rather than to the covariates.

**Test.** Classical LM (`lm`).  
**Outcome.** Donor-level paired difference in phase proportion.  
**Formula.** `diff ~ Sex + Cohort + delta_log10_umi`  
**β.** Intercept — the adjusted mean paired difference.  
**FDR.** BH within stratum, across 3 phases.  

**Display.** Per-stratum forest, matching section 09.

In [ ]:
# §4.3 -- CELL CYCLE: OLS (paired diffs, classical)

cat("=", strrep("=", 71), "\n", sep = "")
cat(sprintf("§4.3 -- CELL CYCLE: OLS (paired diffs)  |  %s / %s\n",
            DATASET, CELL_TYPE))
cat("=", strrep("=", 71), "\n", sep = "")


# ─────────────────────────────────────────────────────────────────────────────
# §4.3.1 STATS
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> STATS\n")
cat("  Test     : Classical LM (lm)\n")
cat("  Formula  : (prop_SnC - prop_NonSnC) ~ Sex + Cohort + delta_log10_umi\n")
cat("  beta     : intercept (= mean paired diff after covariate adjustment)\n")
cat("  Scale    : probability_pts\n")
cat(sprintf("  Strata   : %s\n", paste(strata, collapse = ", ")))
cat(sprintf("  FDR      : %s within (stratum) across 3 phases\n",
            STATISTICAL_PARAMS$fdr_method))


# ─────────────────────────────────────────────────────────────────────────────
# §4.3.2 RUN
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> RUN\n")

cc_ols_rows <- list()

for (stratum in strata) {
    md_s        <- filter_to_stratum(md, stratum, STUDY_GROUP_COL)
    arms_s      <- build_donor_arms(md_s, DONOR_COL, SEN_LABEL_COL,
                                    STATISTICAL_PARAMS$min_cells_per_group)
    paired_ids  <- unique(arms_s$donor[arms_s$paired])
    n_paired    <- length(paired_ids)

    if (n_paired < 5) {
        cat(sprintf("  [%s] only %d paired donors -- skipping\n",
                    stratum, n_paired))
        next
    }

    donor_meta_s <- build_donor_meta(
        md_s, paired_ids,
        DONOR_COL, SEN_LABEL_COL,
        SEX_COL, AGE_COL, COHORT_COL,
        has_cohort = HAS_COHORT
    )

    md_pair <- md_s[md_s[[DONOR_COL]] %in% paired_ids, , drop = FALSE]
    md_pair$is_snc_arm <- as.character(md_pair[[SEN_LABEL_COL]]) %in%
                          c("1", "TRUE", "True", "Senescent")

    counts_df <- md_pair %>%
        group_by(donor   = .data[[DONOR_COL]],
                 is_snc  = is_snc_arm,
                 Phase   = Phase) %>%
        tally(name = "n") %>%
        ungroup() %>%
        group_by(donor, is_snc) %>%
        mutate(prop = n / sum(n)) %>%
        ungroup()

    for (phase in phase_levels) {
        wide_df <- counts_df %>%
            filter(Phase == phase) %>%
            select(donor, is_snc, prop) %>%
            tidyr::pivot_wider(names_from   = is_snc,
                              values_from  = prop,
                              names_prefix = "prop_")

        if (!"prop_TRUE"  %in% names(wide_df)) wide_df$prop_TRUE  <- 0
        if (!"prop_FALSE" %in% names(wide_df)) wide_df$prop_FALSE <- 0
        wide_df$prop_TRUE[is.na(wide_df$prop_TRUE)]   <- 0
        wide_df$prop_FALSE[is.na(wide_df$prop_FALSE)] <- 0

        wide_df$diff <- wide_df$prop_TRUE - wide_df$prop_FALSE

        reg_df <- wide_df %>% left_join(donor_meta_s, by = "donor")

        sex_levels    <- length(unique(reg_df[[SEX_COL]]))
        cohort_levels <- if (HAS_COHORT) length(unique(reg_df[[COHORT_COL]])) else 0

        rhs <- character(0)
        if (sex_levels    >= 2) rhs <- c(rhs, SEX_COL)
        if (cohort_levels >= 2) rhs <- c(rhs, COHORT_COL)
        rhs <- c(rhs, "delta_log10_umi")

        formula_str <- paste("diff ~", paste(rhs, collapse = " + "))
        formula_obj <- as.formula(formula_str)

        fit <- tryCatch({
            lm(formula_obj, data = reg_df)
        }, error = function(e) {
            cat(sprintf("  [%s | %s] lm ERROR: %s\n",
                        stratum, phase, e$message))
            NULL
        })

        if (is.null(fit)) next

        co <- summary(fit)$coefficients
        if (!"(Intercept)" %in% rownames(co)) next

        beta      <- co["(Intercept)", "Estimate"]
        se        <- co["(Intercept)", "Std. Error"]
        statistic <- co["(Intercept)", "t value"]
        p_value   <- co["(Intercept)", "Pr(>|t|)"]

        z <- qnorm(1 - (1 - STATISTICAL_PARAMS$confidence_level) / 2)
        ci_low  <- beta - z * se
        ci_high <- beta + z * se

        cell_cnt_snc <- sum(arms_s$n_cells[arms_s$donor %in% paired_ids &
                                           arms_s$arm %in% c("1", "TRUE", "True", "Senescent")])
        cell_cnt_non <- sum(arms_s$n_cells[arms_s$donor %in% paired_ids &
                                           !(arms_s$arm %in% c("1", "TRUE", "True", "Senescent"))])

        row <- tidy_model_results(
            stratum      = stratum,
            outcome      = phase,
            model        = "ols",
            n_donors     = n_paired,
            n_cells_test = cell_cnt_snc,
            n_cells_ref  = cell_cnt_non,
            estimate     = beta,
            se           = se,
            ci_low       = ci_low,
            ci_high      = ci_high,
            statistic    = statistic,
            p_value      = p_value,
            extra        = list(
                beta_scale         = "probability_pts",
                formula            = formula_str,
                cohort_in_model    = cohort_levels >= 2,
                sex_in_model       = sex_levels >= 2,
                delta_umi_in_model = TRUE,
                r_squared          = summary(fit)$r.squared,
                adj_r_squared      = summary(fit)$adj.r.squared,
                n_covariate_terms  = nrow(co) - 1
            )
        )
        cc_ols_rows[[paste(stratum, phase, sep = "|")]] <- row
    }
}

cc_ols_df <- bind_rows(cc_ols_rows)


# ─────────────────────────────────────────────────────────────────────────────
# §4.3.3 BH-FDR within (stratum)
# ─────────────────────────────────────────────────────────────────────────────
cc_ols_df <- cc_ols_df %>%
    group_by(stratum) %>%
    mutate(p_adj = p.adjust(p_value, method = STATISTICAL_PARAMS$fdr_method)) %>%
    ungroup() %>%
    mutate(sig = sig_stars(p_adj))


# ─────────────────────────────────────────────────────────────────────────────
# §4.3.4 RESULTS
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> RESULTS  (beta = adjusted intercept, scale = probability_pts)\n")

cat(sprintf("\n  %-22s %-5s %5s %+10s %+10s %+10s %8s %8s %4s %6s\n",
            "Stratum", "Phase", "N",
            "beta", "CI_low", "CI_high",
            "p_raw", "p_adj", "Sig", "R^2"))
cat("  ", strrep("-", 100), "\n", sep = "")

for (i in seq_len(nrow(cc_ols_df))) {
    r <- cc_ols_df[i, ]
    cat(sprintf("  %-22s %-5s %5d %+9.4f %+9.4f %+9.4f %8.1e %8.1e %4s %6.3f\n",
                substr(r$stratum, 1, 22), r$outcome, r$n_donors,
                r$estimate, r$ci_low, r$ci_high,
                r$p_value, r$p_adj, r$sig, r$r_squared))
}

save_table(cc_ols_df, paste0(CELL_TYPE, "_cellcycle_ols"))


# ─────────────────────────────────────────────────────────────────────────────
# §4.3.5 FIGURE -- compact per-stratum table-style forest plot
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> FIGURE\n")

cfg <- EFFECT_CONFIGS$probability_pts

for (s in strata) {
    df_s <- cc_ols_df %>%
        filter(stratum == s) %>%
        arrange(desc(abs(estimate)))

    if (nrow(df_s) == 0) {
        cat(sprintf("  [%s] no rows -- skipping figure\n", s))
        next
    }

    df_s$row_label <- as.character(df_s$outcome)
    df_s$eff_str   <- sapply(df_s$estimate, cfg$fmt_effect)
    df_s$ci_str    <- mapply(cfg$fmt_ci, df_s$ci_low, df_s$ci_high)
    df_s$p_str     <- sapply(df_s$p_adj, fmt_p_short)
    df_s$row_color <- PHASE_COLORS[df_s$outcome]
    df_s$is_sig    <- df_s$sig != "ns"

    n_rows <- nrow(df_s)
    df_s$y <- n_rows:1
    header_y    <- n_rows + 0.55
    underline_y <- n_rows + 0.20
    y_lim <- c(0.4, n_rows + 1.0)

    stratum_color <- if (s %in% names(STUDY_GROUP_COLORS)) STUDY_GROUP_COLORS[[s]] else "#7F7F7F"

    n_sig <- sum(df_s$is_sig)
    title_txt <- sprintf("Cell cycle OLS | %s | [%s] | %d/%d sig at BH-FDR",
                         CELL_TYPE, s, n_sig, nrow(df_s))

    p_left <- ggplot(df_s) +
        geom_text(aes(x = 0.05, y = y, label = row_label, color = row_color,
                     fontface = ifelse(is_sig, "bold", "plain")),
                 hjust = 0, size = 2.4) +
        geom_text(aes(x = 0.55, y = y, label = eff_str,
                     fontface = ifelse(is_sig, "bold", "plain")),
                 hjust = 0, size = 2.2, family = "mono", color = "#222222") +
        geom_text(aes(x = 0.78, y = y, label = ci_str),
                 hjust = 0, size = 2.0, family = "mono", color = "#666666") +
        annotate("text", x = 0.05, y = header_y, label = "Phase",
                fontface = "bold", hjust = 0, size = 2.4, color = "#222222") +
        annotate("text", x = 0.55, y = header_y, label = cfg$eff_h_label,
                fontface = "bold", hjust = 0, size = 2.4, color = "#222222") +
        annotate("text", x = 0.78, y = header_y, label = cfg$ci_h_label,
                fontface = "bold", hjust = 0, size = 2.2, color = "#222222") +
        annotate("segment", x = 0, xend = 1.0,
                y = underline_y, yend = underline_y,
                color = "#333333", linewidth = 0.3) +
        scale_color_identity() +
        scale_x_continuous(limits = c(0, 1), expand = c(0, 0)) +
        scale_y_continuous(limits = y_lim, expand = c(0, 0)) +
        labs(title = title_txt) +
        theme_void() +
        theme(
            plot.title = element_text(size = 9, face = "bold",
                                     color = stratum_color,
                                     hjust = 0,
                                     margin = margin(t = 2, b = 1)),
            plot.margin = margin(2, 2, 2, 4)
        )

    eff_vals    <- c(df_s$estimate, df_s$ci_low, df_s$ci_high)
    eff_finite  <- eff_vals[is.finite(eff_vals)]
    if (length(eff_finite) == 0) eff_finite <- c(-0.2, 0.2)
    pad         <- max(diff(range(eff_finite)) * 0.15, 0.05)
    x_range     <- range(eff_finite) + c(-pad, pad)
    x_range[1]  <- min(x_range[1], cfg$null_value - 0.05)
    x_range[2]  <- max(x_range[2], cfg$null_value + 0.05)

    p_forest <- ggplot(df_s) +
        geom_vline(xintercept = cfg$null_value,
                  linetype = "dashed", color = "#999999", linewidth = 0.4) +
        geom_errorbar(aes(y = y, xmin = ci_low, xmax = ci_high),
                     width = 0.18, linewidth = 0.4, color = "#4D4D4D") +
        geom_point(aes(x = estimate, y = y, fill = row_color,
                      size = ifelse(is_sig, 3.5, 2.5)),
                  shape = 23, color = "#222222", stroke = 0.4) +
        scale_fill_identity() +
        scale_size_identity() +
        scale_x_continuous(limits = x_range,
                          labels = cfg$axis_format,
                          breaks = scales::breaks_pretty(n = 4)) +
        scale_y_continuous(limits = y_lim, expand = c(0, 0)) +
        labs(x = cfg$x_label, y = NULL) +
        theme_classic(base_size = 7) +
        theme(
            axis.title.x = element_text(size = 6.5, margin = margin(t = 1)),
            axis.text.x  = element_text(size = 5.5),
            axis.text.y  = element_blank(),
            axis.ticks.y = element_blank(),
            axis.line.y  = element_blank(),
            axis.line.x  = element_line(color = "#444444", linewidth = 0.4),
            panel.grid   = element_blank(),
            plot.margin  = margin(1, 2, 1, 2)
        )

    p_right <- ggplot(df_s) +
        geom_text(aes(x = 0.25, y = y, label = p_str,
                     fontface = ifelse(is_sig, "bold", "plain"),
                     color    = ifelse(is_sig, "#222222", "#666666")),
                 hjust = 0.5, size = 2.2, family = "mono") +
        geom_text(aes(x = 0.75, y = y, label = sig),
                 fontface = "bold", hjust = 0.5, size = 2.4,
                 family = "mono", color = "#222222") +
        annotate("text", x = 0.25, y = header_y, label = "p(adj)",
                fontface = "bold", hjust = 0.5, size = 2.4, color = "#222222") +
        annotate("text", x = 0.75, y = header_y, label = "Sig",
                fontface = "bold", hjust = 0.5, size = 2.4, color = "#222222") +
        annotate("segment", x = 0, xend = 1.0,
                y = underline_y, yend = underline_y,
                color = "#333333", linewidth = 0.3) +
        scale_color_identity() +
        scale_x_continuous(limits = c(0, 1), expand = c(0, 0)) +
        scale_y_continuous(limits = y_lim, expand = c(0, 0)) +
        theme_void() +
        theme(plot.margin = margin(2, 4, 2, 2))

    composed <- (p_left | p_forest | p_right) +
        plot_layout(widths = c(3, 3.5, 1.5))

    fig_height <- max(2.0, 0.9 + n_rows * 0.30)
    slug <- sprintf("%s_cellcycle_ols_%s_forest", CELL_TYPE, s)

    save_figure(composed, slug = slug, width = 7, height = fig_height)

    options(repr.plot.width = 8, repr.plot.height = fig_height + 0.5)
    tryCatch(
        print(composed),
        error = function(e) {
            cat(sprintf("    [inline preview unavailable: %s]\n",
                        conditionMessage(e)))
        }
    )
}


# ─────────────────────────────────────────────────────────────────────────────
# §4.3.6 RLM vs OLS direction agreement (TEXT TABLE ONLY)
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> AGREEMENT WITH §4.2 RLM (text only -- no comparison figure)\n")

agreement <- cc_rlm_df %>%
    select(stratum, outcome, beta_rlm = estimate, p_rlm = p_value) %>%
    inner_join(
        cc_ols_df %>% select(stratum, outcome, beta_ols = estimate, p_ols = p_value),
        by = c("stratum", "outcome")
    ) %>%
    mutate(
        same_sign     = sign(beta_rlm) == sign(beta_ols),
        ratio_rlm_ols = beta_rlm / beta_ols,
        abs_diff_pct  = abs(beta_rlm - beta_ols) * 100
    )

cat(sprintf("\n  %-22s %-5s %+10s %+10s %5s %8s\n",
            "Stratum", "Phase", "beta_rlm", "beta_ols", "Same", "AbsDiff%"))
cat("  ", strrep("-", 70), "\n", sep = "")

for (i in seq_len(nrow(agreement))) {
    r <- agreement[i, ]
    cat(sprintf("  %-22s %-5s %+9.4f %+9.4f %5s %8.3f\n",
                substr(r$stratum, 1, 22), r$outcome,
                r$beta_rlm, r$beta_ols,
                ifelse(r$same_sign, "yes", "FLIP"),
                r$abs_diff_pct))
}

n_sign_disagree <- sum(!agreement$same_sign)
if (n_sign_disagree == 0) {
    cat(sprintf("\n  v All %d (stratum x phase) results agree in sign between RLM and OLS\n",
                nrow(agreement)))
} else {
    cat(sprintf("\n  ! %d / %d (stratum x phase) results FLIP sign between RLM and OLS\n",
                n_sign_disagree, nrow(agreement)))
    cat("    Sign disagreement = outlier-driven; trust RLM\n")
}


# ─────────────────────────────────────────────────────────────────────────────
# §4.3.7 SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
n_sig_rows <- sum(cc_ols_df$p_adj < STATISTICAL_PARAMS$fdr_threshold,
                 na.rm = TRUE)

cat("\n", strrep("-", 72), "\n", sep = "")
cat(sprintf("  §4.3 SUMMARY  |  %s / %s\n", DATASET, CELL_TYPE))
cat(strrep("-", 72), "\n", sep = "")
cat(sprintf("  Test          : Classical LM (lm)\n"))
cat(sprintf("  beta_scale    : probability_pts\n"))
cat(sprintf("  Adjustments   : Sex + Cohort + delta_log10_umi\n"))
cat(sprintf("  Significant   : %d of %d at BH-FDR < %.2f\n",
            n_sig_rows, nrow(cc_ols_df), STATISTICAL_PARAMS$fdr_threshold))
cat(sprintf("  Sign agreement (vs RLM): %d / %d\n",
            sum(agreement$same_sign), nrow(agreement)))
cat(sprintf("  Mean R^2: %.3f\n",
            mean(cc_ols_df$r_squared, na.rm = TRUE)))
cat(sprintf("  Per-stratum forests:\n"))
for (s in strata) {
    cat(sprintf("    %-22s -> %s_cellcycle_ols_%s_forest.{pdf,png,svg}\n",
                s, CELL_TYPE, s))
}
cat(strrep("-", 72), "\n", sep = "")
cat("\nv §4.3 OLS complete\n")
cat("  Next: §4.4 -- LMM (cell-level binomial, beta on log-odds scale)\n")

---
## 11 · Cell cycle — GLMM (cell-level binomial)

**Why.** The cell-level view. Each cell contributes its own phase call, and `(1 | donor)` absorbs the correlation between cells from the same donor. Without that random intercept the p-values would be inflated by the hundreds of cells each donor contributes.

The source header calls this "LMM"; the call is `glmer(family = binomial)`, so it is a **GLMM** and reports on the log-odds scale. That is why section 13 compares it on direction only.

**Test.** Binomial GLMM, `lme4::glmer`.  
**Outcome.** Cell-level phase indicator (0/1).  
**Formula.** `I(Phase == X) ~ is_senescent + Sex + Cohort + log10(nCount_RNA) + (1 | donor)`  
**β.** Log-odds; reported as OR.  
**FDR.** BH within stratum, across 3 phases.  

**Display.** Per-stratum forest on a log axis, null at OR = 1.

In [ ]:
# §4.4 -- CELL CYCLE: LMM (cell-level binomial, donor random effect)

cat("=", strrep("=", 71), "\n", sep = "")
cat(sprintf("§4.4 -- CELL CYCLE: LMM (cell-level binomial)  |  %s / %s\n",
            DATASET, CELL_TYPE))
cat("=", strrep("=", 71), "\n", sep = "")


# ─────────────────────────────────────────────────────────────────────────────
# §4.4.1 STATS
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> STATS\n")
cat("  Test     : GLMM (binomial, logit link) via glmer\n")
cat("  Formula  : I(Phase == X) ~ is_senescent + Sex + Cohort + log10(nCount_RNA) + (1|donor)\n")
cat("  Family   : binomial(link='logit')\n")
cat("  beta     : beta_SnC on log-odds scale; figure displays OR = exp(beta)\n")
cat("  Scale    : log_odds  (companion OR computed)\n")
cat("  Inference: Wald z-test (lme4 summary)\n")
cat(sprintf("  Strata   : %s\n", paste(strata, collapse = ", ")))
cat(sprintf("  FDR      : %s within (stratum) across 3 phases\n",
            STATISTICAL_PARAMS$fdr_method))


# ─────────────────────────────────────────────────────────────────────────────
# §4.4.2 RUN
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> RUN\n")
cat("  Note: 9 GLMM fits at 60-200 sec each. Total ~10-30 min.\n")

cc_lmm_rows <- list()

for (stratum in strata) {
    cat(sprintf("\n  [%s]\n", stratum))

    md_s        <- filter_to_stratum(md, stratum, STUDY_GROUP_COL)
    arms_s      <- build_donor_arms(md_s, DONOR_COL, SEN_LABEL_COL,
                                    STATISTICAL_PARAMS$min_cells_per_group)
    paired_ids  <- unique(arms_s$donor[arms_s$paired])
    n_paired    <- length(paired_ids)

    if (n_paired < 5) {
        cat(sprintf("    only %d paired donors -- skipping\n", n_paired))
        next
    }

    md_pair <- md_s[md_s[[DONOR_COL]] %in% paired_ids, , drop = FALSE]
    md_pair$is_snc_arm <- as.integer(
        as.character(md_pair[[SEN_LABEL_COL]]) %in%
            c("1", "TRUE", "True", "Senescent")
    )
    md_pair$donor      <- as.factor(md_pair[[DONOR_COL]])
    md_pair$Sex_factor <- as.factor(md_pair[[SEX_COL]])
    if (HAS_COHORT) {
        md_pair$Cohort_factor <- as.factor(md_pair[[COHORT_COL]])
    }
    md_pair$log10_umi <- log10(md_pair$nCount_RNA)

    sex_levels    <- length(unique(md_pair$Sex_factor))
    cohort_levels <- if (HAS_COHORT) length(unique(md_pair$Cohort_factor)) else 0

    rhs_fixed <- c("is_snc_arm")
    if (sex_levels    >= 2) rhs_fixed <- c(rhs_fixed, "Sex_factor")
    if (cohort_levels >= 2) rhs_fixed <- c(rhs_fixed, "Cohort_factor")
    rhs_fixed <- c(rhs_fixed, "log10_umi")

    fixed_str  <- paste(rhs_fixed, collapse = " + ")
    random_str <- "(1 | donor)"

    cell_cnt_snc <- sum(md_pair$is_snc_arm == 1)
    cell_cnt_non <- sum(md_pair$is_snc_arm == 0)

    cat(sprintf("    cells: %s SnC + %s Non-SnC = %s total\n",
                fmt_n(cell_cnt_snc), fmt_n(cell_cnt_non),
                fmt_n(cell_cnt_snc + cell_cnt_non)))
    cat(sprintf("    fixed: %s\n", fixed_str))

    for (phase in phase_levels) {
        cat(sprintf("    [%s | %s] fitting glmer ...", stratum, phase))
        t0 <- Sys.time()

        md_pair$y <- as.integer(md_pair$Phase == phase)

        formula_str <- paste("y ~", fixed_str, "+", random_str)
        formula_obj <- as.formula(formula_str)

        fit <- tryCatch({
            glmer(formula_obj,
                  data    = md_pair,
                  family  = binomial(link = "logit"),
                  control = glmerControl(optimizer = "bobyqa",
                                         optCtrl   = list(maxfun = 100000)))
        }, error = function(e) {
            cat(sprintf(" ERROR: %s\n", e$message))
            NULL
        }, warning = function(w) {
            tryCatch(glmer(formula_obj, data = md_pair,
                          family  = binomial(link = "logit"),
                          control = glmerControl(optimizer = "bobyqa",
                                                 optCtrl   = list(maxfun = 100000))),
                    error = function(e) NULL)
        })

        elapsed <- as.numeric(difftime(Sys.time(), t0, units = "secs"))

        if (is.null(fit)) {
            cat(sprintf(" failed after %s\n", fmt_elapsed(elapsed)))
            next
        }
        cat(sprintf(" %s\n", fmt_elapsed(elapsed)))

        co <- summary(fit)$coefficients
        snc_term_idx <- grep("^is_snc_arm$", rownames(co))
        if (length(snc_term_idx) == 0) {
            cat("        No is_snc_arm coefficient found -- skipping\n")
            next
        }

        beta_snc  <- co[snc_term_idx, "Estimate"]
        se        <- co[snc_term_idx, "Std. Error"]
        statistic <- co[snc_term_idx, "z value"]
        p_value   <- co[snc_term_idx, "Pr(>|z|)"]

        z <- qnorm(1 - (1 - STATISTICAL_PARAMS$confidence_level) / 2)
        ci_low  <- beta_snc - z * se
        ci_high <- beta_snc + z * se

        or       <- exp(beta_snc)
        or_low   <- exp(ci_low)
        or_high  <- exp(ci_high)

        conv_msg  <- fit@optinfo$conv$lme4$messages
        converged <- length(conv_msg) == 0
        n_warns   <- length(conv_msg)

        row <- tidy_model_results(
            stratum      = stratum,
            outcome      = phase,
            model        = "lmm",
            n_donors     = n_paired,
            n_cells_test = cell_cnt_snc,
            n_cells_ref  = cell_cnt_non,
            estimate     = beta_snc,
            se           = se,
            ci_low       = ci_low,
            ci_high      = ci_high,
            statistic    = statistic,
            p_value      = p_value,
            extra        = list(
                beta_scale          = "log_odds",
                formula             = formula_str,
                or                  = or,
                or_low              = or_low,
                or_high             = or_high,
                converged           = converged,
                n_warnings          = n_warns,
                fit_seconds         = elapsed,
                cohort_in_model     = cohort_levels >= 2,
                sex_in_model        = sex_levels >= 2,
                n_cells_total       = nrow(md_pair),
                n_random_levels     = length(unique(md_pair$donor))
            )
        )
        cc_lmm_rows[[paste(stratum, phase, sep = "|")]] <- row
    }

    rm(md_pair); gc(verbose = FALSE)
}

cc_lmm_df <- bind_rows(cc_lmm_rows)


# ─────────────────────────────────────────────────────────────────────────────
# §4.4.3 BH-FDR within (stratum)
# ─────────────────────────────────────────────────────────────────────────────
cc_lmm_df <- cc_lmm_df %>%
    group_by(stratum) %>%
    mutate(p_adj = p.adjust(p_value, method = STATISTICAL_PARAMS$fdr_method)) %>%
    ungroup() %>%
    mutate(sig = sig_stars(p_adj))


# ─────────────────────────────────────────────────────────────────────────────
# §4.4.4 RESULTS
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> RESULTS  (beta_SnC on log-odds scale; OR = exp(beta))\n")

cat(sprintf("\n  %-22s %-5s %5s %+9s %+9s %+9s %5s %5s %8s %8s %4s\n",
            "Stratum", "Phase", "N",
            "beta", "CI_low", "CI_high",
            "OR", "CI_OR",
            "p_raw", "p_adj", "Sig"))
cat("  ", strrep("-", 110), "\n", sep = "")

for (i in seq_len(nrow(cc_lmm_df))) {
    r <- cc_lmm_df[i, ]
    cat(sprintf("  %-22s %-5s %5d %+9.4f %+9.4f %+9.4f %5.2f [%4.2f-%4.2f] %8.1e %8.1e %4s\n",
                substr(r$stratum, 1, 22), r$outcome, r$n_donors,
                r$estimate, r$ci_low, r$ci_high,
                r$or, r$or_low, r$or_high,
                r$p_value, r$p_adj, r$sig))
}

save_table(cc_lmm_df, paste0(CELL_TYPE, "_cellcycle_lmm"))


# ─────────────────────────────────────────────────────────────────────────────
# §4.4.5 FIGURE -- compact per-stratum table-style forest plot (log-odds)
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> FIGURE\n")

cfg <- EFFECT_CONFIGS$log_odds

for (s in strata) {
    df_s <- cc_lmm_df %>%
        filter(stratum == s) %>%
        arrange(desc(abs(estimate)))

    if (nrow(df_s) == 0) {
        cat(sprintf("  [%s] no rows -- skipping figure\n", s))
        next
    }

    df_s$row_label <- as.character(df_s$outcome)
    df_s$eff_str   <- sapply(df_s$estimate, cfg$fmt_effect)
    df_s$ci_str    <- mapply(cfg$fmt_ci, df_s$ci_low, df_s$ci_high)
    df_s$p_str     <- sapply(df_s$p_adj, fmt_p_short)
    df_s$row_color <- PHASE_COLORS[df_s$outcome]
    df_s$is_sig    <- df_s$sig != "ns"

    df_s$plot_x       <- cfg$plot_transform(df_s$estimate)
    df_s$plot_ci_low  <- cfg$plot_transform(df_s$ci_low)
    df_s$plot_ci_high <- cfg$plot_transform(df_s$ci_high)

    n_rows <- nrow(df_s)
    df_s$y <- n_rows:1
    header_y    <- n_rows + 0.55
    underline_y <- n_rows + 0.20
    y_lim <- c(0.4, n_rows + 1.0)

    stratum_color <- if (s %in% names(STUDY_GROUP_COLORS)) STUDY_GROUP_COLORS[[s]] else "#7F7F7F"

    n_sig <- sum(df_s$is_sig)
    title_txt <- sprintf("Cell cycle LMM | %s | [%s] | %d/%d sig at BH-FDR",
                         CELL_TYPE, s, n_sig, nrow(df_s))

    p_left <- ggplot(df_s) +
        geom_text(aes(x = 0.05, y = y, label = row_label, color = row_color,
                     fontface = ifelse(is_sig, "bold", "plain")),
                 hjust = 0, size = 2.4) +
        geom_text(aes(x = 0.55, y = y, label = eff_str,
                     fontface = ifelse(is_sig, "bold", "plain")),
                 hjust = 0, size = 2.2, family = "mono", color = "#222222") +
        geom_text(aes(x = 0.78, y = y, label = ci_str),
                 hjust = 0, size = 2.0, family = "mono", color = "#666666") +
        annotate("text", x = 0.05, y = header_y, label = "Phase",
                fontface = "bold", hjust = 0, size = 2.4, color = "#222222") +
        annotate("text", x = 0.55, y = header_y, label = cfg$eff_h_label,
                fontface = "bold", hjust = 0, size = 2.4, color = "#222222") +
        annotate("text", x = 0.78, y = header_y, label = cfg$ci_h_label,
                fontface = "bold", hjust = 0, size = 2.2, color = "#222222") +
        annotate("segment", x = 0, xend = 1.0,
                y = underline_y, yend = underline_y,
                color = "#333333", linewidth = 0.3) +
        scale_color_identity() +
        scale_x_continuous(limits = c(0, 1), expand = c(0, 0)) +
        scale_y_continuous(limits = y_lim, expand = c(0, 0)) +
        labs(title = title_txt) +
        theme_void() +
        theme(
            plot.title = element_text(size = 9, face = "bold",
                                     color = stratum_color,
                                     hjust = 0,
                                     margin = margin(t = 2, b = 1)),
            plot.margin = margin(2, 2, 2, 4)
        )

    eff_vals    <- c(df_s$plot_x, df_s$plot_ci_low, df_s$plot_ci_high)
    eff_finite  <- eff_vals[is.finite(eff_vals) & eff_vals > 0]
    if (length(eff_finite) == 0) eff_finite <- c(0.5, 2)
    log_vals    <- log(eff_finite)
    log_pad     <- max(diff(range(log_vals)) * 0.15, 0.15)
    x_range     <- exp(range(log_vals) + c(-log_pad, log_pad))
    x_range[1]  <- min(x_range[1], cfg$null_value / 1.05)
    x_range[2]  <- max(x_range[2], cfg$null_value * 1.05)

    p_forest <- ggplot(df_s) +
        geom_vline(xintercept = cfg$null_value,
                  linetype = "dashed", color = "#999999", linewidth = 0.4) +
        geom_errorbar(aes(y = y, xmin = plot_ci_low, xmax = plot_ci_high),
                     width = 0.18, linewidth = 0.4, color = "#4D4D4D") +
        geom_point(aes(x = plot_x, y = y, fill = row_color,
                      size = ifelse(is_sig, 3.5, 2.5)),
                  shape = 23, color = "#222222", stroke = 0.4) +
        scale_fill_identity() +
        scale_size_identity() +
        scale_x_log10(limits = x_range,
                     labels = cfg$axis_format,
                     breaks = scales::breaks_log(n = 4)) +
        scale_y_continuous(limits = y_lim, expand = c(0, 0)) +
        labs(x = cfg$x_label, y = NULL) +
        theme_classic(base_size = 7) +
        theme(
            axis.title.x = element_text(size = 6.5, margin = margin(t = 1)),
            axis.text.x  = element_text(size = 5.5),
            axis.text.y  = element_blank(),
            axis.ticks.y = element_blank(),
            axis.line.y  = element_blank(),
            axis.line.x  = element_line(color = "#444444", linewidth = 0.4),
            panel.grid   = element_blank(),
            plot.margin  = margin(1, 2, 1, 2)
        )

    p_right <- ggplot(df_s) +
        geom_text(aes(x = 0.25, y = y, label = p_str,
                     fontface = ifelse(is_sig, "bold", "plain"),
                     color    = ifelse(is_sig, "#222222", "#666666")),
                 hjust = 0.5, size = 2.2, family = "mono") +
        geom_text(aes(x = 0.75, y = y, label = sig),
                 fontface = "bold", hjust = 0.5, size = 2.4,
                 family = "mono", color = "#222222") +
        annotate("text", x = 0.25, y = header_y, label = "p(adj)",
                fontface = "bold", hjust = 0.5, size = 2.4, color = "#222222") +
        annotate("text", x = 0.75, y = header_y, label = "Sig",
                fontface = "bold", hjust = 0.5, size = 2.4, color = "#222222") +
        annotate("segment", x = 0, xend = 1.0,
                y = underline_y, yend = underline_y,
                color = "#333333", linewidth = 0.3) +
        scale_color_identity() +
        scale_x_continuous(limits = c(0, 1), expand = c(0, 0)) +
        scale_y_continuous(limits = y_lim, expand = c(0, 0)) +
        theme_void() +
        theme(plot.margin = margin(2, 4, 2, 2))

    composed <- (p_left | p_forest | p_right) +
        plot_layout(widths = c(3, 3.5, 1.5))

    fig_height <- max(2.0, 0.9 + n_rows * 0.30)
    slug <- sprintf("%s_cellcycle_lmm_%s_forest", CELL_TYPE, s)

    save_figure(composed, slug = slug, width = 7, height = fig_height)

    options(repr.plot.width = 8, repr.plot.height = fig_height + 0.5)
    tryCatch(
        print(composed),
        error = function(e) {
            cat(sprintf("    [inline preview unavailable: %s]\n",
                        conditionMessage(e)))
        }
    )
}


# ─────────────────────────────────────────────────────────────────────────────
# §4.4.6 SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
n_sig_rows  <- sum(cc_lmm_df$p_adj < STATISTICAL_PARAMS$fdr_threshold,
                  na.rm = TRUE)
n_converged <- sum(cc_lmm_df$converged, na.rm = TRUE)

cat("\n", strrep("-", 72), "\n", sep = "")
cat(sprintf("  §4.4 SUMMARY  |  %s / %s\n", DATASET, CELL_TYPE))
cat(strrep("-", 72), "\n", sep = "")
cat(sprintf("  Test          : GLMM (binomial, glmer)\n"))
cat(sprintf("  beta_scale    : log_odds  (companion OR computed)\n"))
cat(sprintf("  Adjustments   : Sex + Cohort + log10(nCount_RNA)\n"))
cat(sprintf("  Random effect : (1 | donor)\n"))
cat(sprintf("  Converged     : %d / %d\n", n_converged, nrow(cc_lmm_df)))
cat(sprintf("  Significant   : %d of %d at BH-FDR < %.2f\n",
            n_sig_rows, nrow(cc_lmm_df), STATISTICAL_PARAMS$fdr_threshold))
cat(sprintf("  Total fit time: %s\n",
            fmt_elapsed(sum(cc_lmm_df$fit_seconds, na.rm = TRUE))))
cat(sprintf("  Per-stratum forests:\n"))
for (s in strata) {
    cat(sprintf("    %-22s -> %s_cellcycle_lmm_%s_forest.{pdf,png,svg}\n",
                s, CELL_TYPE, s))
}
cat(strrep("-", 72), "\n", sep = "")
cat("\nv §4.4 LMM complete\n")
cat("  Next: §4.5 -- Balanced OLS bootstrap (cell-level, per-donor downsampled)\n")

---
## 12 · Cell cycle — balanced RLM bootstrap

**Why.** The arm sizes are wildly unequal — a donor may have thousands of Non-SnC cells and a few dozen SnC. This resamples `min(n_SnC, n_NonSnC)` cells from each arm of each donor, recomputes the donor proportions, and refits, 100 times. The estimate is the median across iterations and the CI is the percentile interval, so the uncertainty reflects within-donor sampling rather than the point estimate alone.

Despite the cell-level resampling this remains a **donor-paired** model: every iteration collapses to one difference per donor before the fit.

**Method.** Balanced cell-level bootstrap, `lmrob` per iteration (100 iterations, seed 42).  
**Outcome.** Per-donor, per-phase `prop_SnC − prop_NonSnC`.  
**Formula.** `diff ~ Sex + Cohort + delta_log10_umi`  
**β.** Median of the intercepts across iterations.  
**CI.** 2.5% / 97.5% percentiles of the bootstrap distribution.  
**p.** `2 × min(P(β > 0), P(β < 0))`.  
**FDR.** BH within stratum, across 3 phases.  

**Display.** Per-stratum forest; convergence rate reported per stratum × phase.

In [ ]:
# MODULE 05 -- Senescence Enrichment & Cell Cycle Analysis
# §4.5 -- CELL CYCLE: BALANCED RLM BOOTSTRAP
# Stats:
#   Method:     Cell-level balanced bootstrap, then lmrob fit per iteration
#   Outcome:    prop_SnC - prop_NonSnC (per donor, per phase)
#   Per-iter:   For each donor, resample min(n_SnC, n_NonSnC) cells per arm
#               (with replacement), recompute donor proportions, then fit
#               lmrob(diff ~ Sex + Cohort + delta_log10_umi).
#   beta:       median(intercepts) across 100 iterations
#   beta_scale: probability_pts
#   CI:         2.5% and 97.5% percentiles of bootstrap intercept distribution
#   p-value:    2 * min(P(beta_iter > 0), P(beta_iter < 0))  (two-sided)
#   FDR:        BH within (stratum) across 3 phases
#
# Convergence: iterations where lmrob fails are skipped; convergence rate
# is reported per (stratum, phase).
#
# Figure: per-stratum compact 3-panel forest plot (3 figures × 3 formats = 9
#         files). Display dispatched via EFFECT_CONFIGS$probability_pts.

cat("=", strrep("=", 71), "\n", sep = "")
cat(sprintf("§4.5 -- CELL CYCLE: BALANCED RLM BOOTSTRAP  |  %s / %s\n",
            DATASET, CELL_TYPE))
cat("=", strrep("=", 71), "\n", sep = "")


# ─────────────────────────────────────────────────────────────────────────────
# §4.5.1 STATS
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> STATS\n")
cat("  Method   : Cell-level balanced bootstrap + lmrob per iteration\n")
cat("  Outcome  : prop_SnC - prop_NonSnC (per donor, per phase)\n")
cat("  Per-iter : Resample min(n_SnC, n_NonSnC) cells/arm/donor with replacement\n")
cat("  Formula  : diff ~ Sex + Cohort + delta_log10_umi  (lmrob, KS2014)\n")
cat("  beta     : median of bootstrap intercepts\n")
cat("  CI       : 2.5%/97.5% percentiles\n")
cat("  p-value  : 2 * min(P(beta > 0), P(beta < 0))\n")
cat(sprintf("  N iter   : %d (seed=%d)\n",
            STATISTICAL_PARAMS$bootstrap_n_iter,
            STATISTICAL_PARAMS$bootstrap_seed))
cat(sprintf("  Strata   : %s\n", paste(strata, collapse = ", ")))
cat(sprintf("  FDR      : %s within (stratum) across 3 phases\n",
            STATISTICAL_PARAMS$fdr_method))


# ─────────────────────────────────────────────────────────────────────────────
# §4.5.2 RUN
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> RUN\n")
cat(sprintf("  Note: %d iter * 9 (stratum x phase) = %d lmrob fits. ~10-30 min.\n",
            STATISTICAL_PARAMS$bootstrap_n_iter,
            STATISTICAL_PARAMS$bootstrap_n_iter * 9))

cc_boot_rows <- list()

for (stratum in strata) {
    cat(sprintf("\n  [%s]\n", stratum))

    md_s        <- filter_to_stratum(md, stratum, STUDY_GROUP_COL)
    arms_s      <- build_donor_arms(md_s, DONOR_COL, SEN_LABEL_COL,
                                    STATISTICAL_PARAMS$min_cells_per_group)
    paired_ids  <- unique(arms_s$donor[arms_s$paired])
    n_paired    <- length(paired_ids)

    if (n_paired < 5) {
        cat(sprintf("    only %d paired donors -- skipping\n", n_paired))
        next
    }

    donor_meta_s <- build_donor_meta(
        md_s, paired_ids,
        DONOR_COL, SEN_LABEL_COL,
        SEX_COL, AGE_COL, COHORT_COL,
        has_cohort = HAS_COHORT
    )

    md_pair <- md_s[md_s[[DONOR_COL]] %in% paired_ids, , drop = FALSE]
    md_pair$is_snc_arm <- as.character(md_pair[[SEN_LABEL_COL]]) %in%
                          c("1", "TRUE", "True", "Senescent")

    sex_levels    <- length(unique(donor_meta_s[[SEX_COL]]))
    cohort_levels <- if (HAS_COHORT) length(unique(donor_meta_s[[COHORT_COL]])) else 0

    rhs <- character(0)
    if (sex_levels    >= 2) rhs <- c(rhs, SEX_COL)
    if (cohort_levels >= 2) rhs <- c(rhs, COHORT_COL)
    rhs <- c(rhs, "delta_log10_umi")

    formula_str <- paste("diff ~", paste(rhs, collapse = " + "))
    formula_obj <- as.formula(formula_str)

    # Pre-compute per-donor cell index lists for fast resampling
    # (avoids re-filtering the full md_pair on every iteration)
    cat(sprintf("    Pre-indexing cells: %d donors\n", n_paired))
    donor_idx <- list()
    for (donor_id in paired_ids) {
        d_rows <- which(md_pair[[DONOR_COL]] == donor_id)
        donor_idx[[as.character(donor_id)]] <- list(
            snc_idx    = d_rows[md_pair$is_snc_arm[d_rows]],
            nonsnc_idx = d_rows[!md_pair$is_snc_arm[d_rows]]
        )
    }

    for (phase in phase_levels) {
        cat(sprintf("    [%s | %s] bootstrapping ...", stratum, phase))
        t0 <- Sys.time()

        # Phase membership (binary) -- resampling references this
        is_in_phase <- (md_pair$Phase == phase)

        set.seed(STATISTICAL_PARAMS$bootstrap_seed)

        boot_intercepts <- numeric(0)
        n_failed        <- 0L

        for (b in seq_len(STATISTICAL_PARAMS$bootstrap_n_iter)) {
            # Build per-donor diff for this iteration
            donor_diffs <- numeric(n_paired)
            names(donor_diffs) <- paired_ids

            for (i in seq_along(paired_ids)) {
                donor_id <- paired_ids[i]
                idx_set  <- donor_idx[[as.character(donor_id)]]

                n_snc    <- length(idx_set$snc_idx)
                n_non    <- length(idx_set$nonsnc_idx)
                n_match  <- min(n_snc, n_non)

                # Balanced resample: draw n_match cells with replacement from each arm
                snc_b    <- sample(idx_set$snc_idx,    size = n_match, replace = TRUE)
                non_b    <- sample(idx_set$nonsnc_idx, size = n_match, replace = TRUE)

                prop_snc <- mean(is_in_phase[snc_b])
                prop_non <- mean(is_in_phase[non_b])

                donor_diffs[i] <- prop_snc - prop_non
            }

            reg_df <- data.frame(donor = paired_ids, diff = donor_diffs,
                                 stringsAsFactors = FALSE) %>%
                left_join(donor_meta_s, by = "donor")

            fit <- tryCatch({
                robustbase::lmrob(formula_obj, data = reg_df, setting = "KS2014")
            }, error   = function(e) NULL,
               warning = function(w) {
                tryCatch(robustbase::lmrob(formula_obj, data = reg_df,
                                           setting = "KS2014"),
                        error = function(e) NULL)
            })

            if (is.null(fit)) {
                n_failed <- n_failed + 1L
                next
            }

            co <- summary(fit)$coefficients
            if (!"(Intercept)" %in% rownames(co)) {
                n_failed <- n_failed + 1L
                next
            }

            boot_intercepts <- c(boot_intercepts, co["(Intercept)", "Estimate"])
        }

        elapsed     <- as.numeric(difftime(Sys.time(), t0, units = "secs"))
        n_converged <- length(boot_intercepts)

        if (n_converged < 10) {
            cat(sprintf(" only %d/%d converged after %s -- skipping\n",
                        n_converged, STATISTICAL_PARAMS$bootstrap_n_iter,
                        fmt_elapsed(elapsed)))
            next
        }
        cat(sprintf(" %d/%d converged in %s\n",
                    n_converged, STATISTICAL_PARAMS$bootstrap_n_iter,
                    fmt_elapsed(elapsed)))

        # Summary statistics across bootstrap distribution
        beta_med <- median(boot_intercepts)
        beta_se  <- sd(boot_intercepts)
        ci       <- quantile(boot_intercepts,
                            probs = c((1 - STATISTICAL_PARAMS$confidence_level) / 2,
                                      1 - (1 - STATISTICAL_PARAMS$confidence_level) / 2),
                            na.rm = TRUE)

        # Two-sided bootstrap p-value
        p_above <- mean(boot_intercepts > 0)
        p_below <- mean(boot_intercepts < 0)
        p_value <- min(2 * min(p_above, p_below), 1)
        # Floor at 1 / n_iter to avoid p = 0
        p_value <- max(p_value, 1 / STATISTICAL_PARAMS$bootstrap_n_iter)

        cell_cnt_snc <- sum(arms_s$n_cells[arms_s$donor %in% paired_ids &
                                           arms_s$arm %in% c("1", "TRUE", "True", "Senescent")])
        cell_cnt_non <- sum(arms_s$n_cells[arms_s$donor %in% paired_ids &
                                           !(arms_s$arm %in% c("1", "TRUE", "True", "Senescent"))])

        row <- tidy_model_results(
            stratum      = stratum,
            outcome      = phase,
            model        = "rlm_bootstrap",
            n_donors     = n_paired,
            n_cells_test = cell_cnt_snc,
            n_cells_ref  = cell_cnt_non,
            estimate     = beta_med,
            se           = beta_se,
            ci_low       = unname(ci[1]),
            ci_high      = unname(ci[2]),
            statistic    = NA_real_,
            p_value      = p_value,
            extra        = list(
                beta_scale         = "probability_pts",
                formula            = formula_str,
                cohort_in_model    = cohort_levels >= 2,
                sex_in_model       = sex_levels >= 2,
                delta_umi_in_model = TRUE,
                n_iterations       = STATISTICAL_PARAMS$bootstrap_n_iter,
                n_converged        = n_converged,
                n_failed           = n_failed,
                bootstrap_seed     = STATISTICAL_PARAMS$bootstrap_seed,
                fit_seconds        = elapsed
            )
        )
        cc_boot_rows[[paste(stratum, phase, sep = "|")]] <- row
    }

    rm(donor_idx, md_pair); gc(verbose = FALSE)
}

cc_boot_df <- bind_rows(cc_boot_rows)


# ─────────────────────────────────────────────────────────────────────────────
# §4.5.3 BH-FDR within (stratum)
# ─────────────────────────────────────────────────────────────────────────────
cc_boot_df <- cc_boot_df %>%
    group_by(stratum) %>%
    mutate(p_adj = p.adjust(p_value, method = STATISTICAL_PARAMS$fdr_method)) %>%
    ungroup() %>%
    mutate(sig = sig_stars(p_adj))


# ─────────────────────────────────────────────────────────────────────────────
# §4.5.4 RESULTS
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> RESULTS  (beta = median of bootstrap intercepts, scale = probability_pts)\n")

cat(sprintf("\n  %-22s %-5s %5s %+10s %+10s %+10s %5s %8s %8s %4s\n",
            "Stratum", "Phase", "N",
            "beta", "CI_low", "CI_high",
            "Conv.",
            "p_raw", "p_adj", "Sig"))
cat("  ", strrep("-", 100), "\n", sep = "")

for (i in seq_len(nrow(cc_boot_df))) {
    r <- cc_boot_df[i, ]
    cat(sprintf("  %-22s %-5s %5d %+9.4f %+9.4f %+9.4f %4d/%d %8.1e %8.1e %4s\n",
                substr(r$stratum, 1, 22), r$outcome, r$n_donors,
                r$estimate, r$ci_low, r$ci_high,
                r$n_converged, r$n_iterations,
                r$p_value, r$p_adj, r$sig))
}

save_table(cc_boot_df, paste0(CELL_TYPE, "_cellcycle_bootrlm"))


# ─────────────────────────────────────────────────────────────────────────────
# §4.5.5 FIGURE -- compact per-stratum table-style forest plot
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> FIGURE\n")

cfg <- EFFECT_CONFIGS$probability_pts

for (s in strata) {
    df_s <- cc_boot_df %>%
        filter(stratum == s) %>%
        arrange(desc(abs(estimate)))

    if (nrow(df_s) == 0) {
        cat(sprintf("  [%s] no rows -- skipping figure\n", s))
        next
    }

    df_s$row_label  <- as.character(df_s$outcome)
    df_s$eff_str    <- sapply(df_s$estimate, cfg$fmt_effect)
    df_s$ci_str     <- mapply(cfg$fmt_ci, df_s$ci_low, df_s$ci_high)
    df_s$p_str      <- sapply(df_s$p_adj, fmt_p_short)
    df_s$row_color  <- PHASE_COLORS[df_s$outcome]
    df_s$is_sig     <- df_s$sig != "ns"

    n_rows <- nrow(df_s)
    df_s$y <- n_rows:1
    header_y    <- n_rows + 0.55
    underline_y <- n_rows + 0.20
    y_lim <- c(0.4, n_rows + 1.0)

    stratum_color <- if (s %in% names(STUDY_GROUP_COLORS)) STUDY_GROUP_COLORS[[s]] else "#7F7F7F"

    n_sig <- sum(df_s$is_sig)
    title_txt <- sprintf("Cell cycle bootRLM | %s | [%s] | %d/%d sig at BH-FDR",
                         CELL_TYPE, s, n_sig, nrow(df_s))

    p_left <- ggplot(df_s) +
        geom_text(aes(x = 0.05, y = y, label = row_label,
                     color = row_color,
                     fontface = ifelse(is_sig, "bold", "plain")),
                 hjust = 0, size = 2.4) +
        geom_text(aes(x = 0.55, y = y, label = eff_str,
                     fontface = ifelse(is_sig, "bold", "plain")),
                 hjust = 0, size = 2.2, family = "mono", color = "#222222") +
        geom_text(aes(x = 0.78, y = y, label = ci_str),
                 hjust = 0, size = 2.0, family = "mono", color = "#666666") +
        annotate("text", x = 0.05, y = header_y, label = "Phase",
                fontface = "bold", hjust = 0, size = 2.4, color = "#222222") +
        annotate("text", x = 0.55, y = header_y, label = cfg$eff_h_label,
                fontface = "bold", hjust = 0, size = 2.4, color = "#222222") +
        annotate("text", x = 0.78, y = header_y, label = cfg$ci_h_label,
                fontface = "bold", hjust = 0, size = 2.2, color = "#222222") +
        annotate("segment", x = 0, xend = 1.0,
                y = underline_y, yend = underline_y,
                color = "#333333", linewidth = 0.3) +
        scale_color_identity() +
        scale_x_continuous(limits = c(0, 1), expand = c(0, 0)) +
        scale_y_continuous(limits = y_lim, expand = c(0, 0)) +
        labs(title = title_txt) +
        theme_void() +
        theme(
            plot.title = element_text(size = 9, face = "bold",
                                     color = stratum_color,
                                     hjust = 0,
                                     margin = margin(t = 2, b = 1)),
            plot.margin = margin(2, 2, 2, 4)
        )

    eff_vals    <- c(df_s$estimate, df_s$ci_low, df_s$ci_high)
    eff_finite  <- eff_vals[is.finite(eff_vals)]
    if (length(eff_finite) == 0) eff_finite <- c(-0.2, 0.2)
    pad         <- max(diff(range(eff_finite)) * 0.15, 0.05)
    x_range     <- range(eff_finite) + c(-pad, pad)
    x_range[1]  <- min(x_range[1], cfg$null_value - 0.05)
    x_range[2]  <- max(x_range[2], cfg$null_value + 0.05)

    p_forest <- ggplot(df_s) +
        geom_vline(xintercept = cfg$null_value,
                  linetype = "dashed", color = "#999999", linewidth = 0.4) +
        geom_errorbar(aes(y = y, xmin = ci_low, xmax = ci_high),
                     width = 0.18, linewidth = 0.4, color = "#4D4D4D") +
        geom_point(aes(x = estimate, y = y, fill = row_color,
                      size = ifelse(is_sig, 3.5, 2.5)),
                  shape = 23, color = "#222222", stroke = 0.4) +
        scale_fill_identity() +
        scale_size_identity() +
        scale_x_continuous(limits = x_range,
                          labels = cfg$axis_format,
                          breaks = scales::breaks_pretty(n = 4)) +
        scale_y_continuous(limits = y_lim, expand = c(0, 0)) +
        labs(x = cfg$x_label, y = NULL) +
        theme_classic(base_size = 7) +
        theme(
            axis.title.x = element_text(size = 6.5, margin = margin(t = 1)),
            axis.text.x  = element_text(size = 5.5),
            axis.text.y  = element_blank(),
            axis.ticks.y = element_blank(),
            axis.line.y  = element_blank(),
            axis.line.x  = element_line(color = "#444444", linewidth = 0.4),
            panel.grid   = element_blank(),
            plot.margin  = margin(1, 2, 1, 2)
        )

    p_right <- ggplot(df_s) +
        geom_text(aes(x = 0.25, y = y, label = p_str,
                     fontface = ifelse(is_sig, "bold", "plain"),
                     color    = ifelse(is_sig, "#222222", "#666666")),
                 hjust = 0.5, size = 2.2, family = "mono") +
        geom_text(aes(x = 0.75, y = y, label = sig),
                 fontface = "bold", hjust = 0.5, size = 2.4,
                 family = "mono", color = "#222222") +
        annotate("text", x = 0.25, y = header_y, label = "p(adj)",
                fontface = "bold", hjust = 0.5, size = 2.4, color = "#222222") +
        annotate("text", x = 0.75, y = header_y, label = "Sig",
                fontface = "bold", hjust = 0.5, size = 2.4, color = "#222222") +
        annotate("segment", x = 0, xend = 1.0,
                y = underline_y, yend = underline_y,
                color = "#333333", linewidth = 0.3) +
        scale_color_identity() +
        scale_x_continuous(limits = c(0, 1), expand = c(0, 0)) +
        scale_y_continuous(limits = y_lim, expand = c(0, 0)) +
        theme_void() +
        theme(plot.margin = margin(2, 4, 2, 2))

    composed <- (p_left | p_forest | p_right) +
        plot_layout(widths = c(3, 3.5, 1.5))

    fig_height <- max(2.0, 0.9 + n_rows * 0.30)
    slug <- sprintf("%s_cellcycle_bootrlm_%s_forest", CELL_TYPE, s)

    save_figure(composed, slug = slug, width = 7, height = fig_height)

    options(repr.plot.width = 8, repr.plot.height = fig_height + 0.5)
    tryCatch(
        print(composed),
        error = function(e) {
            cat(sprintf("    [inline preview unavailable: %s]\n",
                        conditionMessage(e)))
        }
    )
}


# ─────────────────────────────────────────────────────────────────────────────
# §4.5.6 SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
n_sig_rows  <- sum(cc_boot_df$p_adj < STATISTICAL_PARAMS$fdr_threshold,
                  na.rm = TRUE)
mean_conv   <- mean(cc_boot_df$n_converged / cc_boot_df$n_iterations,
                   na.rm = TRUE) * 100

cat("\n", strrep("-", 72), "\n", sep = "")
cat(sprintf("  §4.5 SUMMARY  |  %s / %s\n", DATASET, CELL_TYPE))
cat(strrep("-", 72), "\n", sep = "")
cat(sprintf("  Method        : Cell-level balanced bootstrap + lmrob (KS2014)\n"))
cat(sprintf("  beta_scale    : probability_pts\n"))
cat(sprintf("  Adjustments   : Sex + Cohort + delta_log10_umi\n"))
cat(sprintf("  Iterations    : %d (seed=%d)\n",
            STATISTICAL_PARAMS$bootstrap_n_iter,
            STATISTICAL_PARAMS$bootstrap_seed))
cat(sprintf("  Mean conv.    : %.1f%%\n", mean_conv))
cat(sprintf("  Significant   : %d of %d at BH-FDR < %.2f\n",
            n_sig_rows, nrow(cc_boot_df), STATISTICAL_PARAMS$fdr_threshold))
cat(sprintf("  Total fit time: %s\n",
            fmt_elapsed(sum(cc_boot_df$fit_seconds, na.rm = TRUE))))
cat(sprintf("  Per-stratum forests:\n"))
for (s in strata) {
    cat(sprintf("    %-22s -> %s_cellcycle_bootrlm_%s_forest.{pdf,png,svg}\n",
                s, CELL_TYPE, s))
}
cat(strrep("-", 72), "\n", sep = "")
cat("\nv §4.5 balanced RLM bootstrap complete\n")
cat("  Next: §4.6 -- Cross-model agreement (5-model comparison)\n")

---
## 13 · Cell cycle — cross-model agreement

**Why.** The synthesis. Five estimators, one table: how many agree in direction, how many reach significance. This is the output that gets read, not the individual forests — a phase that moves under one estimator and not the others has not moved.

Magnitude comparison excludes the GLMM (log-odds vs probability points); direction comparison includes it.

**Inputs.** `cc_wilcox_df`, `cc_rlm_df`, `cc_ols_df`, `cc_lmm_df`, `cc_boot_df`.  
**Reports.** `majority_direction`, `n_majority`, `n_models_sig`, `any_sig`, `all_sig`, `median_beta_pp`, `lmm_or`.  
**Verdict.** H1/H2 from section 06 is re-stated against `any_sig`.  

**Display.** Heatmap, rows = stratum × phase, columns = model, fill = direction × significance.

In [ ]:
# MODULE 05 -- Senescence Enrichment & Cell Cycle Analysis
# §4.6 -- CELL CYCLE: CROSS-MODEL AGREEMENT
# Synthesizes results across the 5 cell-cycle models from §4.1-§4.5:
#   §4.1 wilcoxon         (donor-level paired Wilcoxon)
#   §4.2 rlm              (donor-level RLM, lmrob KS2014)
#   §4.3 ols              (donor-level OLS)
#   §4.4 lmm              (cell-level binomial GLMM, log-odds)
#   §4.5 rlm_bootstrap    (cell-level balanced bootstrap + lmrob)
#
# Reads each cc_*_df from memory and produces:
#   - Long-form combined table cc_crossmodel_long
#   - Wide summary table cc_crossmodel_wide (one row per stratum x phase)
#   - Sign agreement metrics (how many of 5 models agree in direction)
#   - Significance metrics (n_models_sig at BH-FDR<0.05)
#   - Heatmap figure: rows = stratum x phase, cols = model, fill encodes
#       direction x significance via MODEL_AGREEMENT_COLORS
#
# Note: LMM operates on log-odds scale (beta_scale = "log_odds"); the other
# 4 use probability_pts. Direction comparisons are sign-agnostic to scale.
# Magnitude comparisons exclude LMM (different units).

cat("=", strrep("=", 71), "\n", sep = "")
cat(sprintf("§4.6 -- CROSS-MODEL AGREEMENT  |  %s / %s\n",
            DATASET, CELL_TYPE))
cat("=", strrep("=", 71), "\n", sep = "")


# ─────────────────────────────────────────────────────────────────────────────
# §4.6.1 PRECONDITION CHECK -- all 5 model dfs must be in scope
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> PRECONDITION\n")

required_dfs <- list(
    wilcoxon       = "cc_wilcox_df",
    rlm            = "cc_rlm_df",
    ols            = "cc_ols_df",
    lmm            = "cc_lmm_df",
    rlm_bootstrap  = "cc_boot_df"
)

missing <- character(0)
for (model_name in names(required_dfs)) {
    df_name <- required_dfs[[model_name]]
    if (!exists(df_name) || is.null(get(df_name)) || nrow(get(df_name)) == 0) {
        missing <- c(missing, df_name)
    } else {
        cat(sprintf("  %-15s -> %s (%d rows)\n",
                    model_name, df_name, nrow(get(df_name))))
    }
}

if (length(missing) > 0) {
    stop(sprintf("✗ Missing model dataframes: %s\n  Re-run §4.1-§4.5 before §4.6.",
                 paste(missing, collapse = ", ")))
}


# ─────────────────────────────────────────────────────────────────────────────
# §4.6.2 BUILD LONG-FORM COMBINED TABLE
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> BUILD COMBINED TABLE\n")

# Standard columns each row should expose
keep_cols <- c("stratum", "outcome", "model",
              "n_donors", "estimate", "se", "ci_low", "ci_high",
              "p_value", "p_adj", "sig", "beta_scale")

extract_cols <- function(df, model_label) {
    out <- df %>% select(any_of(keep_cols))
    out$model <- model_label
    if (!"beta_scale" %in% colnames(out)) out$beta_scale <- NA_character_
    out
}

cc_crossmodel_long <- bind_rows(
    extract_cols(cc_wilcox_df, "wilcoxon"),
    extract_cols(cc_rlm_df,    "rlm"),
    extract_cols(cc_ols_df,    "ols"),
    extract_cols(cc_lmm_df,    "lmm"),
    extract_cols(cc_boot_df,   "rlm_bootstrap")
)

# Stable model factor for plotting
cc_crossmodel_long$model <- factor(
    cc_crossmodel_long$model,
    levels = c("wilcoxon", "rlm", "ols", "rlm_bootstrap", "lmm")
)
cc_crossmodel_long$stratum <- factor(cc_crossmodel_long$stratum, levels = strata)
cc_crossmodel_long$outcome <- factor(cc_crossmodel_long$outcome, levels = phase_levels)

# Direction (using OR-1 sign for LMM, beta sign otherwise)
cc_crossmodel_long$direction <- ifelse(
    cc_crossmodel_long$beta_scale == "log_odds",
    sign(cc_crossmodel_long$estimate),  # log-odds: beta > 0 == OR > 1
    sign(cc_crossmodel_long$estimate)
)

# Sign-x-sig category for heatmap fill
cc_crossmodel_long$direction_label <- with(cc_crossmodel_long, {
    sig_flag <- !is.na(p_adj) & p_adj < STATISTICAL_PARAMS$fdr_threshold
    case_when(
        is.na(direction) | is.na(p_adj)   ~ "ns",
        direction > 0 &  sig_flag          ~ "Up (sig)",
        direction > 0 & !sig_flag          ~ "Up (ns)",
        direction < 0 &  sig_flag          ~ "Down (sig)",
        direction < 0 & !sig_flag          ~ "Down (ns)",
        TRUE                                ~ "ns"
    )
})
cc_crossmodel_long$direction_label <- factor(
    cc_crossmodel_long$direction_label,
    levels = c("Up (sig)", "Up (ns)", "ns", "Down (ns)", "Down (sig)")
)

cat(sprintf("  Long table: %d rows (5 models x 3 strata x 3 phases)\n",
            nrow(cc_crossmodel_long)))

save_table(cc_crossmodel_long, paste0(CELL_TYPE, "_cellcycle_crossmodel_long"))


# ─────────────────────────────────────────────────────────────────────────────
# §4.6.3 BUILD WIDE SUMMARY (one row per stratum x phase)
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> BUILD SUMMARY TABLE\n")

# Per-row: how many models agree in direction, how many significant
cc_crossmodel_wide <- cc_crossmodel_long %>%
    group_by(stratum, outcome) %>%
    summarise(
        n_donors           = first(n_donors),
        n_models           = n(),
        # Sign agreement: how many models point in the majority direction?
        n_pos              = sum(direction > 0, na.rm = TRUE),
        n_neg              = sum(direction < 0, na.rm = TRUE),
        n_zero             = sum(direction == 0 | is.na(direction)),
        majority_direction = case_when(
            n_pos > n_neg ~ "+",
            n_neg > n_pos ~ "-",
            TRUE          ~ "0"
        ),
        n_majority         = pmax(n_pos, n_neg),
        # Significance
        n_models_sig       = sum(!is.na(p_adj) & p_adj < STATISTICAL_PARAMS$fdr_threshold,
                                na.rm = TRUE),
        any_sig            = n_models_sig > 0,
        all_sig            = n_models_sig == n_models,
        # Magnitude consensus on probability-points scale (4 models, drop LMM)
        median_beta_pp     = median(estimate[beta_scale == "probability_pts"],
                                   na.rm = TRUE),
        mean_beta_pp       = mean(estimate[beta_scale == "probability_pts"],
                                  na.rm = TRUE),
        # LMM odds ratio (separate scale)
        lmm_or             = exp(estimate[model == "lmm"][1]),
        .groups = "drop"
    )

cat(sprintf("  Wide summary: %d rows (%d strata x %d phases)\n",
            nrow(cc_crossmodel_wide),
            length(strata), length(phase_levels)))

save_table(cc_crossmodel_wide, paste0(CELL_TYPE, "_cellcycle_crossmodel_summary"))


# ─────────────────────────────────────────────────────────────────────────────
# §4.6.4 RESULTS TEXT
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> RESULTS\n")

cat(sprintf("\n  %-22s %-5s %5s %4s %8s %8s %8s %5s %5s\n",
            "Stratum", "Phase", "N",
            "Maj.", "med_pp", "mean_pp", "LMM_OR",
            "n_sig", "all_sig"))
cat("  ", strrep("-", 90), "\n", sep = "")

for (i in seq_len(nrow(cc_crossmodel_wide))) {
    r <- cc_crossmodel_wide[i, ]
    cat(sprintf("  %-22s %-5s %5d  %s%d  %+7.4f  %+7.4f  %6.2f  %2d/%d  %5s\n",
                substr(as.character(r$stratum), 1, 22),
                as.character(r$outcome), r$n_donors,
                r$majority_direction, r$n_majority,
                r$median_beta_pp, r$mean_beta_pp, r$lmm_or,
                r$n_models_sig, r$n_models,
                ifelse(r$all_sig, "yes", "no")))
}

# Direction-agreement quick summary
n_unanimous <- sum(cc_crossmodel_wide$n_majority == cc_crossmodel_wide$n_models,
                  na.rm = TRUE)
cat(sprintf("\n  Unanimous direction across all 5 models: %d / %d (stratum x phase)\n",
            n_unanimous, nrow(cc_crossmodel_wide)))

if (sum(cc_crossmodel_wide$any_sig) == 0) {
    cat("  No model reached BH-FDR < 0.05 in any (stratum x phase).\n")
} else {
    cat(sprintf("  Any-model significance: %d / %d rows\n",
                sum(cc_crossmodel_wide$any_sig), nrow(cc_crossmodel_wide)))
    cat(sprintf("  All-model significance: %d / %d rows\n",
                sum(cc_crossmodel_wide$all_sig), nrow(cc_crossmodel_wide)))
}


# ─────────────────────────────────────────────────────────────────────────────
# §4.6.5 FIGURE -- direction x significance heatmap (5 models x stratum x phase)
#
# One panel per stratum, faceted columns = phase, rows = models.
# Fill = direction_label encoded via MODEL_AGREEMENT_COLORS (§0).
# Cell text = beta value (decimal for prob_pts, OR for log_odds).
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> FIGURE\n")

# Cell label: beta or OR depending on scale
cc_crossmodel_long$cell_label <- with(cc_crossmodel_long, {
    out <- character(nrow(cc_crossmodel_long))
    is_log <- !is.na(beta_scale) & beta_scale == "log_odds"
    out[is_log]  <- sprintf("OR=%.2f", exp(estimate[is_log]))
    out[!is_log] <- sprintf("%+.3f",   estimate[!is_log])
    sig_flag <- !is.na(p_adj) & p_adj < STATISTICAL_PARAMS$fdr_threshold
    out[sig_flag] <- paste0(out[sig_flag], "*")
    out
})

n_strata <- length(strata)
n_models <- nlevels(cc_crossmodel_long$model)

p_heatmap <- ggplot(cc_crossmodel_long,
                   aes(x = outcome, y = model, fill = direction_label)) +
    geom_tile(color = "white", linewidth = 0.4) +
    geom_text(aes(label = cell_label),
             size = 2.4, family = "mono", color = "#222222") +
    facet_wrap(~ stratum, ncol = 1, strip.position = "right") +
    scale_fill_manual(values = MODEL_AGREEMENT_COLORS,
                     drop = FALSE,
                     name = "Direction x sig") +
    scale_x_discrete(position = "top") +
    scale_y_discrete(limits = rev(levels(cc_crossmodel_long$model))) +
    labs(title    = sprintf("Cell cycle cross-model agreement | %s",
                            CELL_TYPE),
         subtitle = sprintf("5 models x 3 strata x 3 phases | %d/%d unanimous direction | %d/%d any-model sig",
                            n_unanimous, nrow(cc_crossmodel_wide),
                            sum(cc_crossmodel_wide$any_sig),
                            nrow(cc_crossmodel_wide)),
         x        = "Phase",
         y        = "Model") +
    theme_minimal(base_size = 8) +
    theme(
        plot.title       = element_text(size = 10, face = "bold",
                                       color = "#222222",
                                       margin = margin(b = 1)),
        plot.subtitle    = element_text(size = 7, color = "#444444",
                                       margin = margin(b = 4)),
        axis.title.x     = element_text(size = 7),
        axis.title.y     = element_text(size = 7),
        axis.text.x      = element_text(size = 7, face = "bold"),
        axis.text.y      = element_text(size = 7, family = "mono"),
        strip.text       = element_text(size = 7, face = "bold"),
        strip.background = element_rect(fill = "#F0F0F0", color = "#222222"),
        legend.position  = "bottom",
        legend.title     = element_text(size = 7),
        legend.text      = element_text(size = 6),
        legend.key.size  = unit(0.3, "cm"),
        panel.grid       = element_blank()
    )

fig_height <- max(3.0, 1.2 + n_strata * 1.0)
slug <- paste0(CELL_TYPE, "_cellcycle_crossmodel_heatmap")

save_figure(p_heatmap, slug = slug, width = 7, height = fig_height)

options(repr.plot.width = 8, repr.plot.height = fig_height + 0.5)
tryCatch(
    print(p_heatmap),
    error = function(e) {
        cat(sprintf("    [inline preview unavailable: %s]\n",
                    conditionMessage(e)))
    }
)


# ─────────────────────────────────────────────────────────────────────────────
# §4.6.6 SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
cat("\n", strrep("-", 72), "\n", sep = "")
cat(sprintf("  §4.6 SUMMARY  |  %s / %s\n", DATASET, CELL_TYPE))
cat(strrep("-", 72), "\n", sep = "")
cat(sprintf("  Models compared    : %d (wilcoxon, rlm, ols, lmm, rlm_bootstrap)\n",
            nlevels(cc_crossmodel_long$model)))
cat(sprintf("  Stratum x phase    : %d\n", nrow(cc_crossmodel_wide)))
cat(sprintf("  Unanimous direction: %d / %d\n",
            n_unanimous, nrow(cc_crossmodel_wide)))
cat(sprintf("  Any-model sig      : %d / %d (BH-FDR < %.2f)\n",
            sum(cc_crossmodel_wide$any_sig),
            nrow(cc_crossmodel_wide),
            STATISTICAL_PARAMS$fdr_threshold))
cat(sprintf("  All-model sig      : %d / %d\n",
            sum(cc_crossmodel_wide$all_sig),
            nrow(cc_crossmodel_wide)))
cat(sprintf("  H1/H2 verdict (§3.5): %s\n",
            if (sum(cc_crossmodel_wide$any_sig) == 0)
                "supported (no model significant)"
            else
                "challenged (re-examine)"))
cat(sprintf("\n  Output:\n"))
cat(sprintf("    results/%s_cellcycle_crossmodel_long.csv\n", CELL_TYPE))
cat(sprintf("    results/%s_cellcycle_crossmodel_summary.csv\n", CELL_TYPE))
cat(sprintf("    figures/%s.{pdf,png,svg}\n", slug))
cat(strrep("-", 72), "\n", sep = "")
cat("\nv §4.6 cross-model agreement complete\n")
cat("  Next: §5 -- module score testing (5-model structure on 10 Sloan scores)\n")

---
## 14 · Module scores — Wilcoxon (donor-paired)

**Why.** The same distribution-free anchor, now on the 10 Sloan Table S9 hallmark panels rather than on phase calls. One mean score per donor per arm; the contrast is the paired difference.

**Test.** Paired Wilcoxon signed-rank.  
**Outcome.** Per-donor mean module score.  
**β.** Mean paired difference, `SnC − Non-SnC` (score units).  
**CI.** Hodges-Lehmann.  
**FDR.** BH within stratum, across 10 modules.  

**Display.** Box + violin + jitter, `facet_grid2(stratum ~ module)`. Module strips coloured by list type — dark grey = individual hallmark, purple = multi-hallmark composite. Order fixed by `SLOAN_HALLMARK_NAMES`.

In [ ]:
# MODULE 05 -- Senescence Enrichment & Cell Cycle Analysis
# §5.1 -- MODULE SCORES: WILCOXON (donor-level paired)
# Stats:  Paired Wilcoxon signed-rank on per-donor mean module score
#         (one mean per donor x arm). beta = mean(SnC - Non-SnC).
#         Hodges-Lehmann CI from wilcox.test(conf.int=TRUE).
# Scale:  score_units (continuous module score)
# FDR:    BH within (stratum) across 10 modules
#
# Figure: Single box+violin+jitter figure with facet_grid2(stratum ~ module),
#         matching §4.1 cell-cycle Wilcoxon pattern.
#         Module top-strips colored by hallmark type (dark gray = individual
#         hallmark, purple = multi-hallmark composite). Stratum side-strips
#         colored by STUDY_GROUP_COLORS. SnC red / Non-SnC grey.
#         Modules in FIXED canonical order from SLOAN_HALLMARK_NAMES.

cat("=", strrep("=", 71), "\n", sep = "")
cat(sprintf("§5.1 -- MODULE SCORES: WILCOXON  |  %s / %s\n",
            DATASET, CELL_TYPE))
cat("=", strrep("=", 71), "\n", sep = "")


# ─────────────────────────────────────────────────────────────────────────────
# §5.1.1 STATS
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> STATS\n")
cat("  Test     : Paired Wilcoxon signed-rank\n")
cat("  Outcome  : per-donor mean module score (SnC vs Non-SnC)\n")
cat("  beta     : mean paired difference (SnC - Non-SnC)\n")
cat("  CI       : Hodges-Lehmann 95% from wilcox.test(conf.int=TRUE)\n")
cat("  Scale    : score_units\n")
cat(sprintf("  FDR      : %s within (stratum) across %d modules\n",
            STATISTICAL_PARAMS$fdr_method,
            length(SLOAN_HALLMARK_NAMES)))
cat(sprintf("  Strata   : %s\n", paste(strata, collapse = ", ")))
cat(sprintf("  Modules  : %s\n",
            paste(SLOAN_HALLMARK_NAMES, collapse = ", ")))


# ─────────────────────────────────────────────────────────────────────────────
# §5.1.2 RUN -- resolve module score columns, then test per (stratum, module)
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> RUN\n")

md_cols <- colnames(md)

score_col_lookup <- character(length(SLOAN_HALLMARK_NAMES))
names(score_col_lookup) <- SLOAN_HALLMARK_NAMES
for (mod in SLOAN_HALLMARK_NAMES) {
    candidates <- c(
        paste0("Score_", mod),
        paste0("score_", mod),
        paste0("module_", mod),
        mod,
        paste0(mod, "1"),
        paste0("Score_", mod, "1")
    )
    found <- intersect(candidates, md_cols)
    if (length(found) > 0) {
        score_col_lookup[mod] <- found[1]
    } else {
        score_col_lookup[mod] <- NA_character_
    }
}

n_resolved <- sum(!is.na(score_col_lookup))
cat(sprintf("  Resolved %d/%d module score columns:\n",
            n_resolved, length(SLOAN_HALLMARK_NAMES)))
for (mod in SLOAN_HALLMARK_NAMES) {
    col <- score_col_lookup[mod]
    if (is.na(col)) {
        cat(sprintf("    %-22s  --  ✗ NOT FOUND\n", mod))
    } else {
        cat(sprintf("    %-22s  ->  %s\n", mod, col))
    }
}

if (n_resolved == 0) {
    stop("✗ No module score columns found in metadata. Re-run §3 (scoring).")
}

mod_wilcox_rows <- list()

for (stratum in strata) {
    md_s        <- filter_to_stratum(md, stratum, STUDY_GROUP_COL)
    arms_s      <- build_donor_arms(md_s, DONOR_COL, SEN_LABEL_COL,
                                    STATISTICAL_PARAMS$min_cells_per_group)
    paired_ids  <- unique(arms_s$donor[arms_s$paired])
    n_paired    <- length(paired_ids)

    if (n_paired < 5) {
        cat(sprintf("  [%s] only %d paired donors -- skipping\n",
                    stratum, n_paired))
        next
    }

    md_pair <- md_s[md_s[[DONOR_COL]] %in% paired_ids, , drop = FALSE]
    md_pair$is_snc_arm <- as.character(md_pair[[SEN_LABEL_COL]]) %in%
                          c("1", "TRUE", "True", "Senescent")

    cell_cnt_snc <- sum(arms_s$n_cells[arms_s$donor %in% paired_ids &
                                       arms_s$arm %in% c("1", "TRUE", "True", "Senescent")])
    cell_cnt_non <- sum(arms_s$n_cells[arms_s$donor %in% paired_ids &
                                       !(arms_s$arm %in% c("1", "TRUE", "True", "Senescent"))])

    for (mod in SLOAN_HALLMARK_NAMES) {
        score_col <- score_col_lookup[mod]
        if (is.na(score_col)) next

        per_donor <- md_pair %>%
            group_by(donor   = .data[[DONOR_COL]],
                     is_snc  = is_snc_arm) %>%
            summarise(mean_score = mean(.data[[score_col]], na.rm = TRUE),
                     .groups = "drop")

        wide_df <- per_donor %>%
            tidyr::pivot_wider(names_from   = is_snc,
                              values_from  = mean_score,
                              names_prefix = "score_")

        if (!"score_TRUE"  %in% names(wide_df)) next
        if (!"score_FALSE" %in% names(wide_df)) next
        wide_df <- wide_df[!is.na(wide_df$score_TRUE) &
                           !is.na(wide_df$score_FALSE), , drop = FALSE]

        if (nrow(wide_df) < 5) next

        snc_scores <- wide_df$score_TRUE
        non_scores <- wide_df$score_FALSE
        diffs      <- snc_scores - non_scores

        wt <- tryCatch({
            wilcox.test(snc_scores, non_scores, paired = TRUE,
                        exact = FALSE, conf.int = TRUE)
        }, error = function(e) NULL)

        if (is.null(wt)) next

        n_obs           <- sum(diffs != 0)
        W_stat          <- as.numeric(wt$statistic)
        W_max           <- if (n_obs > 0) n_obs * (n_obs + 1) / 2 else NA_real_
        rank_biserial_r <- if (!is.na(W_max) && W_max > 0) (2 * W_stat / W_max) - 1 else NA_real_
        cohens_d        <- if (sd(diffs) > 0) mean(diffs) / sd(diffs) else NA_real_
        pct_higher      <- 100 * mean(diffs > 0)

        row <- tidy_model_results(
            stratum      = stratum,
            outcome      = mod,
            model        = "wilcoxon",
            n_donors     = nrow(wide_df),
            n_cells_test = cell_cnt_snc,
            n_cells_ref  = cell_cnt_non,
            estimate     = mean(diffs),
            se           = sd(diffs) / sqrt(nrow(wide_df)),
            ci_low       = if (!is.null(wt$conf.int)) wt$conf.int[1] else NA_real_,
            ci_high      = if (!is.null(wt$conf.int)) wt$conf.int[2] else NA_real_,
            statistic    = W_stat,
            p_value      = wt$p.value,
            extra        = list(
                beta_scale        = "score_units",
                module_type       = SLOAN_LIST_TYPES[which(SLOAN_HALLMARK_NAMES == mod)],
                score_col         = score_col,
                hl_pseudomedian   = if (!is.null(wt$estimate)) as.numeric(wt$estimate) else NA_real_,
                median_diff       = median(diffs),
                mean_snc          = mean(snc_scores),
                mean_nonsnc       = mean(non_scores),
                rank_biserial_r   = rank_biserial_r,
                cohens_d          = cohens_d,
                pct_donors_higher = pct_higher
            )
        )
        mod_wilcox_rows[[paste(stratum, mod, sep = "|")]] <- row
    }
}

mod_wilcox_df <- bind_rows(mod_wilcox_rows)


# ─────────────────────────────────────────────────────────────────────────────
# §5.1.3 BH-FDR within (stratum)
# ─────────────────────────────────────────────────────────────────────────────
mod_wilcox_df <- mod_wilcox_df %>%
    group_by(stratum) %>%
    mutate(p_adj = p.adjust(p_value, method = STATISTICAL_PARAMS$fdr_method)) %>%
    ungroup() %>%
    mutate(sig = sig_stars(p_adj))


# ─────────────────────────────────────────────────────────────────────────────
# §5.1.4 RESULTS
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> RESULTS  (beta = mean paired difference, scale = score_units)\n")

cat(sprintf("\n  %-22s %-22s %5s %+10s %+10s %+10s %8s %8s %4s\n",
            "Stratum", "Module", "N",
            "beta", "CI_low", "CI_high",
            "p_raw", "p_adj", "Sig"))
cat("  ", strrep("-", 110), "\n", sep = "")

for (s in strata) {
    df_s <- mod_wilcox_df %>% filter(stratum == s)
    df_s <- df_s[match(SLOAN_HALLMARK_NAMES, df_s$outcome), ]
    df_s <- df_s[!is.na(df_s$stratum), ]

    for (i in seq_len(nrow(df_s))) {
        r <- df_s[i, ]
        cat(sprintf("  %-22s %-22s %5d %+9.4f %+9.4f %+9.4f %8.1e %8.1e %4s\n",
                    substr(r$stratum, 1, 22), substr(r$outcome, 1, 22),
                    r$n_donors,
                    r$estimate, r$ci_low, r$ci_high,
                    r$p_value, r$p_adj, r$sig))
    }
}

save_table(mod_wilcox_df, paste0(CELL_TYPE, "_modulescores_wilcoxon"))


# ─────────────────────────────────────────────────────────────────────────────
# §5.1.5 FIGURE -- single box+violin grid (mirrors §4.1 pattern)
#
# Layout: facet_grid2(stratum ~ module). 3 stratum rows x 10 module cols.
# Module top-strips: hallmark type colors (dark gray = individual, purple =
# multi-hallmark composite). Stratum side-strips: STUDY_GROUP_COLORS.
# Each panel: x = Non-SnC vs SnC, y = per-donor mean module score.
#             violin (alpha 0.5) + boxplot (white fill) + jittered points.
# Annotations above each panel: BH-FDR sig stars + "beta=+0.0045" label.
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> FIGURE\n")

if (!requireNamespace("ggh4x", quietly = TRUE)) {
    cat("  Installing ggh4x...\n")
    install.packages("ggh4x", repos = "https://cloud.r-project.org")
}
suppressPackageStartupMessages(library(ggh4x))

# Build per-donor plot data (one mean module score per donor x arm x module)
plot_rows <- list()
for (stratum in strata) {
    md_s   <- filter_to_stratum(md, stratum, STUDY_GROUP_COL)
    arms_s <- build_donor_arms(md_s, DONOR_COL, SEN_LABEL_COL,
                               STATISTICAL_PARAMS$min_cells_per_group)
    paired_ids <- unique(arms_s$donor[arms_s$paired])
    if (length(paired_ids) < 5) next

    md_pair <- md_s[md_s[[DONOR_COL]] %in% paired_ids, , drop = FALSE]
    md_pair$is_snc_arm <- as.character(md_pair[[SEN_LABEL_COL]]) %in%
                          c("1", "TRUE", "True", "Senescent")
    md_pair$arm_label  <- ifelse(md_pair$is_snc_arm, "SnC", "Non-SnC")

    # For each module: compute per-donor x arm mean score
    for (mod in SLOAN_HALLMARK_NAMES) {
        score_col <- score_col_lookup[mod]
        if (is.na(score_col)) next

        donor_means <- md_pair %>%
            group_by(donor     = .data[[DONOR_COL]],
                     arm_label = arm_label) %>%
            summarise(mean_score = mean(.data[[score_col]], na.rm = TRUE),
                     .groups = "drop") %>%
            mutate(stratum = stratum,
                  module  = mod)

        plot_rows[[paste(stratum, mod, sep = "|")]] <- donor_means
    }
}
plot_df <- bind_rows(plot_rows)
plot_df$stratum   <- factor(plot_df$stratum,   levels = strata)
plot_df$arm_label <- factor(plot_df$arm_label, levels = c("Non-SnC", "SnC"))
plot_df$module    <- factor(plot_df$module,    levels = SLOAN_HALLMARK_NAMES)

# Significance annotations: positioned above each panel's max value
sig_annot <- mod_wilcox_df %>%
    select(stratum, module = outcome, sig, p_adj, estimate) %>%
    mutate(stratum = factor(stratum, levels = strata),
          module  = factor(module,  levels = SLOAN_HALLMARK_NAMES))

annot_y <- plot_df %>%
    group_by(stratum, module) %>%
    summarise(panel_max = max(mean_score, na.rm = TRUE),
             panel_min = min(mean_score, na.rm = TRUE),
             panel_range = panel_max - panel_min,
             .groups = "drop") %>%
    mutate(y_sig  = panel_max + panel_range * 0.18,
          y_beta = panel_max + panel_range * 0.07)

sig_annot <- sig_annot %>% left_join(annot_y, by = c("stratum", "module"))

# Strip palettes
module_strip_palette <- SLOAN_HALLMARK_COLORS[SLOAN_HALLMARK_NAMES]
stratum_strip_palette <- sapply(strata, function(s) {
    if (s %in% names(STUDY_GROUP_COLORS)) STUDY_GROUP_COLORS[[s]] else "#7F7F7F"
})
names(stratum_strip_palette) <- strata

module_text_colors  <- text_color_for_bg(module_strip_palette)
stratum_text_colors <- text_color_for_bg(stratum_strip_palette)

p_mod_wilcox <- ggplot(plot_df,
                      aes(x = arm_label, y = mean_score, fill = arm_label)) +
    geom_violin(alpha = 0.5, linewidth = 0.3, scale = "width") +
    geom_boxplot(width = 0.18, outlier.shape = NA, alpha = 0.7,
                fill = "white", color = "black", linewidth = 0.3) +
    geom_jitter(width = 0.12, size = 0.4, alpha = 0.4, color = "grey30") +
    geom_text(data = sig_annot,
             aes(x = 1.5, y = y_sig, label = sig),
             inherit.aes = FALSE,
             size = 3.5, fontface = "bold") +
    geom_text(data = sig_annot,
             aes(x = 1.5, y = y_beta,
                 label = sprintf("beta=%+.3f", estimate)),
             inherit.aes = FALSE,
             size = 2.2, color = "grey30") +
    scale_fill_manual(values = c("Non-SnC" = "#D3D3D3", "SnC" = "#C44E52")) +
    facet_grid2(stratum ~ module,
               scales = "free_y",
               strip = strip_themed(
                   background_x = elem_list_rect(
                       fill  = unname(module_strip_palette),
                       color = "black"
                   ),
                   text_x = elem_list_text(
                       color = unname(module_text_colors),
                       face  = "bold",
                       size  = 7
                   ),
                   background_y = elem_list_rect(
                       fill  = unname(stratum_strip_palette),
                       color = "black"
                   ),
                   text_y = elem_list_text(
                       color = unname(stratum_text_colors),
                       face  = "bold",
                       size  = 8
                   )
               )) +
    labs(title    = sprintf("Module scores -- Wilcoxon (paired, donor-level) -- %s",
                            CELL_TYPE),
         subtitle = sprintf("Top strips = Module (dark = hallmark, purple = multi-hallmark). Right strips = Stratum. SnC red / Non-SnC grey. *,**,*** = BH-FDR < %.2f.",
                            STATISTICAL_PARAMS$fdr_threshold),
         x        = NULL,
         y        = "Per-donor mean module score",
         fill     = NULL) +
    theme_clean(base_size = 8) +
    theme(legend.position = "top",
         strip.text.x    = element_text(size = 6, face = "bold"),
         axis.text.x     = element_text(size = 6),
         axis.text.y     = element_text(size = 6),
         panel.spacing.x = unit(0.15, "lines"),
         panel.spacing.y = unit(0.30, "lines"))

# Figure dims: ~1.2"/module x 1.7"/stratum
fig_width  <- 2 + length(SLOAN_HALLMARK_NAMES) * 1.3
fig_height <- 2 + length(strata) * 1.7

save_figure(p_mod_wilcox,
           slug   = paste0(CELL_TYPE, "_modulescores_wilcoxon"),
           width  = fig_width,
           height = fig_height)

options(repr.plot.width = fig_width + 1, repr.plot.height = fig_height + 0.5)
tryCatch(
    print(p_mod_wilcox),
    error = function(e) {
        cat(sprintf("    [inline preview unavailable: %s]\n",
                    conditionMessage(e)))
    }
)


# ─────────────────────────────────────────────────────────────────────────────
# §5.1.6 SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
n_sig_rows <- sum(mod_wilcox_df$p_adj < STATISTICAL_PARAMS$fdr_threshold,
                 na.rm = TRUE)

cat("\n", strrep("-", 72), "\n", sep = "")
cat(sprintf("  §5.1 SUMMARY  |  %s / %s\n", DATASET, CELL_TYPE))
cat(strrep("-", 72), "\n", sep = "")
cat(sprintf("  Test          : Paired Wilcoxon signed-rank\n"))
cat(sprintf("  beta_scale    : score_units\n"))
cat(sprintf("  Modules       : %d (8 hallmarks + 2 multi-hallmark composites)\n",
            length(SLOAN_HALLMARK_NAMES)))
cat(sprintf("  Significant   : %d of %d at BH-FDR < %.2f\n",
            n_sig_rows, nrow(mod_wilcox_df), STATISTICAL_PARAMS$fdr_threshold))

if (n_sig_rows > 0) {
    cat(sprintf("\n  Significant modules (beta > 0 = SnC enriched):\n"))
    sig_rows <- mod_wilcox_df %>%
        filter(p_adj < STATISTICAL_PARAMS$fdr_threshold) %>%
        arrange(stratum, desc(estimate))
    for (i in seq_len(nrow(sig_rows))) {
        r <- sig_rows[i, ]
        direction <- ifelse(r$estimate > 0, "+", "-")
        cat(sprintf("    %s %-22s [%s]  beta=%+.4f  p_adj=%.2e\n",
                    direction, r$outcome, r$stratum,
                    r$estimate, r$p_adj))
    }
}

cat(sprintf("\n  Figure: %s_modulescores_wilcoxon.{pdf,png,svg}  (%.1f x %.1f in)\n",
            CELL_TYPE, fig_width, fig_height))
cat(strrep("-", 72), "\n", sep = "")
cat("\nv §5.1 module scores Wilcoxon complete\n")
cat("  Next: §5.2 -- RLM on paired diffs (covariate-adjusted)\n")

---
## 15 · Module scores — RLM on paired differences

**Why.** Covariate-adjusted, outlier-resistant. The depth covariate matters more here than anywhere else in the module: senescent cells carry more UMIs, and `AddModuleScore` is not immune to that.

**Test.** Robust LM, `lmrob` MM-estimator, `setting = "KS2014"`.  
**Outcome.** Donor-level paired difference in mean module score.  
**Formula.** `diff ~ Sex + Cohort + delta_log10_umi`  
**β.** Intercept — the adjusted mean difference.  
**FDR.** BH within stratum, across 10 modules.  

**Display.** Per-stratum forest, adaptive layout, divider between the 8 individual hallmarks and the 2 composites.

In [ ]:
# MODULE 05 -- Senescence Enrichment & Cell Cycle Analysis
# §5.2 -- MODULE SCORES: RLM (paired diffs, covariate-adjusted)
# Stats:
#   Test:     Robust LM via lmrob (MM-estimator, KS2014)
#   Outcome:  donor-level paired difference (mean_SnC - mean_NonSnC)
#   Formula:  diff ~ Sex + Cohort + delta_log10_umi
#   beta:     intercept (= adjusted mean diff after covariate adjustment)
#   Scale:    score_units
#   FDR:      BH within (stratum) across 10 modules
#
# Modules tested in fixed canonical order (SLOAN_HALLMARK_NAMES, §0).
# Figure: per-stratum compact 3-panel forest plot, fully adaptive layout
# (column widths and figure dimensions derived from data).

cat("=", strrep("=", 71), "\n", sep = "")
cat(sprintf("§5.2 -- MODULE SCORES: RLM (paired diffs)  |  %s / %s\n",
            DATASET, CELL_TYPE))
cat("=", strrep("=", 71), "\n", sep = "")


# ─────────────────────────────────────────────────────────────────────────────
# §5.2.1 STATS
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> STATS\n")
cat("  Test     : Robust LM via lmrob (MM-estimator, KS2014)\n")
cat("  Outcome  : donor-level paired difference (mean_SnC - mean_NonSnC)\n")
cat("  Formula  : diff ~ Sex + Cohort + delta_log10_umi\n")
cat("  beta     : intercept (= adjusted mean diff)\n")
cat("  Scale    : score_units\n")
cat(sprintf("  Strata   : %s\n", paste(strata, collapse = ", ")))
cat(sprintf("  FDR      : %s within (stratum) across %d modules\n",
            STATISTICAL_PARAMS$fdr_method,
            length(SLOAN_HALLMARK_NAMES)))


# ─────────────────────────────────────────────────────────────────────────────
# §5.2.2 RUN
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> RUN\n")

# Score column lookup -- assumes §5.1 has run; rebuild defensively if needed
if (!exists("score_col_lookup")) {
    md_cols <- colnames(md)
    score_col_lookup <- character(length(SLOAN_HALLMARK_NAMES))
    names(score_col_lookup) <- SLOAN_HALLMARK_NAMES
    for (mod in SLOAN_HALLMARK_NAMES) {
        candidates <- c(paste0("Score_", mod), paste0("score_", mod),
                       paste0("module_", mod), mod, paste0(mod, "1"))
        found <- intersect(candidates, md_cols)
        if (length(found) > 0) score_col_lookup[mod] <- found[1]
        else                   score_col_lookup[mod] <- NA_character_
    }
}

mod_rlm_rows <- list()

for (stratum in strata) {
    md_s        <- filter_to_stratum(md, stratum, STUDY_GROUP_COL)
    arms_s      <- build_donor_arms(md_s, DONOR_COL, SEN_LABEL_COL,
                                    STATISTICAL_PARAMS$min_cells_per_group)
    paired_ids  <- unique(arms_s$donor[arms_s$paired])
    n_paired    <- length(paired_ids)

    if (n_paired < 5) {
        cat(sprintf("  [%s] only %d paired donors -- skipping\n",
                    stratum, n_paired))
        next
    }

    donor_meta_s <- build_donor_meta(
        md_s, paired_ids,
        DONOR_COL, SEN_LABEL_COL,
        SEX_COL, AGE_COL, COHORT_COL,
        has_cohort = HAS_COHORT
    )

    md_pair <- md_s[md_s[[DONOR_COL]] %in% paired_ids, , drop = FALSE]
    md_pair$is_snc_arm <- as.character(md_pair[[SEN_LABEL_COL]]) %in%
                          c("1", "TRUE", "True", "Senescent")

    cell_cnt_snc <- sum(arms_s$n_cells[arms_s$donor %in% paired_ids &
                                       arms_s$arm %in% c("1", "TRUE", "True", "Senescent")])
    cell_cnt_non <- sum(arms_s$n_cells[arms_s$donor %in% paired_ids &
                                       !(arms_s$arm %in% c("1", "TRUE", "True", "Senescent"))])

    for (mod in SLOAN_HALLMARK_NAMES) {
        score_col <- score_col_lookup[mod]
        if (is.na(score_col)) next

        # Per-donor mean score by arm; pivot to wide; compute paired diff
        per_donor <- md_pair %>%
            group_by(donor   = .data[[DONOR_COL]],
                     is_snc  = is_snc_arm) %>%
            summarise(mean_score = mean(.data[[score_col]], na.rm = TRUE),
                     .groups = "drop")

        wide_df <- per_donor %>%
            tidyr::pivot_wider(names_from   = is_snc,
                              values_from  = mean_score,
                              names_prefix = "score_")

        if (!"score_TRUE"  %in% names(wide_df)) next
        if (!"score_FALSE" %in% names(wide_df)) next
        wide_df <- wide_df[!is.na(wide_df$score_TRUE) &
                           !is.na(wide_df$score_FALSE), , drop = FALSE]
        if (nrow(wide_df) < 5) next

        wide_df$diff <- wide_df$score_TRUE - wide_df$score_FALSE

        reg_df <- wide_df %>% left_join(donor_meta_s, by = "donor")

        sex_levels    <- length(unique(reg_df[[SEX_COL]]))
        cohort_levels <- if (HAS_COHORT) length(unique(reg_df[[COHORT_COL]])) else 0

        rhs <- character(0)
        if (sex_levels    >= 2) rhs <- c(rhs, SEX_COL)
        if (cohort_levels >= 2) rhs <- c(rhs, COHORT_COL)
        rhs <- c(rhs, "delta_log10_umi")

        formula_str <- paste("diff ~", paste(rhs, collapse = " + "))
        formula_obj <- as.formula(formula_str)

        fit <- tryCatch({
            robustbase::lmrob(formula_obj, data = reg_df, setting = "KS2014")
        }, error = function(e) {
            cat(sprintf("  [%s | %s] lmrob ERROR: %s\n",
                        stratum, mod, e$message))
            NULL
        }, warning = function(w) {
            tryCatch(robustbase::lmrob(formula_obj, data = reg_df,
                                       setting = "KS2014"),
                    error = function(e) NULL)
        })

        if (is.null(fit)) next

        co <- summary(fit)$coefficients
        if (!"(Intercept)" %in% rownames(co)) next

        beta      <- co["(Intercept)", "Estimate"]
        se        <- co["(Intercept)", "Std. Error"]
        statistic <- co["(Intercept)", "t value"]
        p_value   <- co["(Intercept)", "Pr(>|t|)"]

        z <- qnorm(1 - (1 - STATISTICAL_PARAMS$confidence_level) / 2)
        ci_low  <- beta - z * se
        ci_high <- beta + z * se

        row <- tidy_model_results(
            stratum      = stratum,
            outcome      = mod,
            model        = "rlm",
            n_donors     = nrow(wide_df),
            n_cells_test = cell_cnt_snc,
            n_cells_ref  = cell_cnt_non,
            estimate     = beta,
            se           = se,
            ci_low       = ci_low,
            ci_high      = ci_high,
            statistic    = statistic,
            p_value      = p_value,
            extra        = list(
                beta_scale         = "score_units",
                module_type        = SLOAN_LIST_TYPES[which(SLOAN_HALLMARK_NAMES == mod)],
                score_col          = score_col,
                formula            = formula_str,
                cohort_in_model    = cohort_levels >= 2,
                sex_in_model       = sex_levels >= 2,
                delta_umi_in_model = TRUE,
                converged          = isTRUE(fit$converged),
                n_covariate_terms  = nrow(co) - 1
            )
        )
        mod_rlm_rows[[paste(stratum, mod, sep = "|")]] <- row
    }
}

mod_rlm_df <- bind_rows(mod_rlm_rows)


# ─────────────────────────────────────────────────────────────────────────────
# §5.2.3 BH-FDR within (stratum)
# ─────────────────────────────────────────────────────────────────────────────
mod_rlm_df <- mod_rlm_df %>%
    group_by(stratum) %>%
    mutate(p_adj = p.adjust(p_value, method = STATISTICAL_PARAMS$fdr_method)) %>%
    ungroup() %>%
    mutate(sig = sig_stars(p_adj))


# ─────────────────────────────────────────────────────────────────────────────
# §5.2.4 RESULTS
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> RESULTS  (beta = adjusted intercept, scale = score_units)\n")

cat(sprintf("\n  %-22s %-22s %5s %+10s %+10s %+10s %8s %8s %4s\n",
            "Stratum", "Module", "N",
            "beta", "CI_low", "CI_high",
            "p_raw", "p_adj", "Sig"))
cat("  ", strrep("-", 110), "\n", sep = "")

for (s in strata) {
    df_s <- mod_rlm_df %>% filter(stratum == s)
    df_s <- df_s[match(SLOAN_HALLMARK_NAMES, df_s$outcome), ]
    df_s <- df_s[!is.na(df_s$stratum), ]

    for (i in seq_len(nrow(df_s))) {
        r <- df_s[i, ]
        cat(sprintf("  %-22s %-22s %5d %+9.4f %+9.4f %+9.4f %8.1e %8.1e %4s\n",
                    substr(r$stratum, 1, 22), substr(r$outcome, 1, 22),
                    r$n_donors,
                    r$estimate, r$ci_low, r$ci_high,
                    r$p_value, r$p_adj, r$sig))
    }
}

save_table(mod_rlm_df, paste0(CELL_TYPE, "_modulescores_rlm"))


# ─────────────────────────────────────────────────────────────────────────────
# §5.2.5 FIGURE -- per-stratum forest plot, FULLY ADAPTIVE layout
#
# Layout: column widths derived from longest text string in each column;
# figure dimensions derived from total em count + n_rows. No hardcoded
# data-specific dimensions. Universal typography constants in FOREST_LAYOUT.
#
# Modules in fixed canonical order with hallmark/composite divider line.
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> FIGURE\n")

# Universal layout tuning constants (move to §0 once stable across cells)
if (!exists("FOREST_LAYOUT")) {
    FOREST_LAYOUT <- list(
        em_per_char_mono   = 0.62,
        em_per_char_prop   = 0.55,
        inter_col_gap      = 1.5,
        panel_right_pad    = 0.8,
        forest_share_mult  = 0.7,
        inches_per_em      = 0.020,
        min_fig_width_in   = 6.0,
        min_fig_height_in  = 2.5,
        row_height_in      = 0.30,
        base_height_in     = 0.9,
        pad_x_axis_frac    = 0.10,
        pad_x_axis_null    = 0.02
    )
}

cfg <- EFFECT_CONFIGS$score_units

divider_after_idx <- sum(SLOAN_LIST_TYPES == "Individual hallmark")  # = 8

for (s in strata) {
    df_s <- mod_rlm_df %>%
        filter(stratum == s) %>%
        mutate(outcome = factor(outcome, levels = SLOAN_HALLMARK_NAMES)) %>%
        arrange(outcome)

    if (nrow(df_s) == 0) {
        cat(sprintf("  [%s] no rows -- skipping figure\n", s))
        next
    }

    df_s$row_label  <- as.character(df_s$outcome)
    df_s$eff_str    <- sapply(df_s$estimate, cfg$fmt_effect)
    df_s$ci_str     <- mapply(cfg$fmt_ci, df_s$ci_low, df_s$ci_high)
    df_s$p_str      <- sapply(df_s$p_adj, fmt_p_short)
    df_s$row_color  <- SLOAN_HALLMARK_COLORS[as.character(df_s$outcome)]
    df_s$is_sig     <- df_s$sig != "ns"

    n_rows <- nrow(df_s)

    # ── Adaptive column widths from longest string in each column ───────
    h_mod  <- "Module"
    h_eff  <- cfg$eff_h_label
    h_ci   <- cfg$ci_h_label
    h_padj <- "p(adj)"
    h_sig  <- "Sig"

    nchar_label <- max(nchar(c(df_s$row_label, h_mod)),  na.rm = TRUE)
    nchar_eff   <- max(nchar(c(df_s$eff_str,   h_eff)),  na.rm = TRUE)
    nchar_ci    <- max(nchar(c(df_s$ci_str,    h_ci)),   na.rm = TRUE)
    nchar_padj  <- max(nchar(c(df_s$p_str,     h_padj)), na.rm = TRUE)
    nchar_sig   <- max(nchar(c(df_s$sig,       h_sig)),  na.rm = TRUE)

    em_label <- nchar_label * FOREST_LAYOUT$em_per_char_prop
    em_eff   <- nchar_eff   * FOREST_LAYOUT$em_per_char_mono
    em_ci    <- nchar_ci    * FOREST_LAYOUT$em_per_char_mono
    em_padj  <- nchar_padj  * FOREST_LAYOUT$em_per_char_mono
    em_sig   <- nchar_sig   * FOREST_LAYOUT$em_per_char_mono

    gap <- FOREST_LAYOUT$inter_col_gap

    x_label_pos <- 0
    x_eff_pos   <- em_label + gap
    x_ci_pos    <- em_label + gap + em_eff + gap
    left_em     <- em_label + gap + em_eff + gap + em_ci + FOREST_LAYOUT$panel_right_pad

    x_padj_pos  <- em_padj / 2
    x_sig_pos   <- em_padj + gap + em_sig / 2
    right_em    <- em_padj + gap + em_sig + FOREST_LAYOUT$panel_right_pad

    forest_em <- (left_em + right_em) * FOREST_LAYOUT$forest_share_mult
    total_em  <- left_em + forest_em + right_em
    fig_width <- max(total_em * FOREST_LAYOUT$inches_per_em,
                    FOREST_LAYOUT$min_fig_width_in)

    df_s$y      <- n_rows:1
    header_y    <- n_rows + 0.55
    underline_y <- n_rows + 0.20
    y_lim       <- c(0.4, n_rows + 1.0)
    divider_y   <- (n_rows - divider_after_idx) + 0.5

    stratum_color <- if (s %in% names(STUDY_GROUP_COLORS)) STUDY_GROUP_COLORS[[s]] else "#7F7F7F"

    n_sig <- sum(df_s$is_sig)
    title_txt <- sprintf("Module scores RLM | %s | [%s] | %d/%d sig at BH-FDR",
                         CELL_TYPE, s, n_sig, nrow(df_s))

    # ── LEFT panel ──────────────────────────────────────────────────────
    p_left <- ggplot(df_s) +
        annotate("segment", x = 0, xend = left_em,
                y = divider_y, yend = divider_y,
                color = "#CCCCCC", linewidth = 0.4, linetype = "dashed") +
        geom_text(aes(x = x_label_pos, y = y, label = row_label,
                     color = row_color,
                     fontface = ifelse(is_sig, "bold", "plain")),
                 hjust = 0, size = 2.4) +
        geom_text(aes(x = x_eff_pos, y = y, label = eff_str,
                     fontface = ifelse(is_sig, "bold", "plain")),
                 hjust = 0, size = 2.2, family = "mono", color = "#222222") +
        geom_text(aes(x = x_ci_pos, y = y, label = ci_str),
                 hjust = 0, size = 2.0, family = "mono", color = "#666666") +
        annotate("text", x = x_label_pos, y = header_y, label = h_mod,
                fontface = "bold", hjust = 0, size = 2.4, color = "#222222") +
        annotate("text", x = x_eff_pos, y = header_y, label = h_eff,
                fontface = "bold", hjust = 0, size = 2.4, color = "#222222") +
        annotate("text", x = x_ci_pos, y = header_y, label = h_ci,
                fontface = "bold", hjust = 0, size = 2.2, color = "#222222") +
        annotate("segment", x = 0, xend = left_em,
                y = underline_y, yend = underline_y,
                color = "#333333", linewidth = 0.3) +
        scale_color_identity() +
        scale_x_continuous(limits = c(0, left_em), expand = c(0, 0)) +
        scale_y_continuous(limits = y_lim, expand = c(0, 0)) +
        labs(title = title_txt) +
        theme_void() +
        theme(
            plot.title = element_text(size = 9, face = "bold",
                                     color = stratum_color,
                                     hjust = 0,
                                     margin = margin(t = 2, b = 1)),
            plot.margin = margin(2, 2, 2, 4)
        )

    # ── FOREST panel: x-range adapts tightly to data ────────────────────
    eff_vals    <- c(df_s$estimate, df_s$ci_low, df_s$ci_high)
    eff_finite  <- eff_vals[is.finite(eff_vals)]
    if (length(eff_finite) == 0) eff_finite <- c(-0.001, 0.001)

    data_range  <- diff(range(eff_finite))
    if (data_range == 0) data_range <- max(abs(eff_finite[1]), 1e-6) * 0.2

    pad         <- data_range * FOREST_LAYOUT$pad_x_axis_frac
    null_pad    <- data_range * FOREST_LAYOUT$pad_x_axis_null
    x_range     <- range(eff_finite) + c(-pad, pad)
    x_range[1]  <- min(x_range[1], cfg$null_value - null_pad)
    x_range[2]  <- max(x_range[2], cfg$null_value + null_pad)

    p_forest <- ggplot(df_s) +
        annotate("segment", x = -Inf, xend = Inf,
                y = divider_y, yend = divider_y,
                color = "#CCCCCC", linewidth = 0.4, linetype = "dashed") +
        geom_vline(xintercept = cfg$null_value,
                  linetype = "dashed", color = "#999999", linewidth = 0.4) +
        geom_errorbar(aes(y = y, xmin = ci_low, xmax = ci_high),
                     width = 0.18, linewidth = 0.4, color = "#4D4D4D") +
        geom_point(aes(x = estimate, y = y, fill = row_color,
                      size = ifelse(is_sig, 3.5, 2.5)),
                  shape = 23, color = "#222222", stroke = 0.4) +
        scale_fill_identity() +
        scale_size_identity() +
        scale_x_continuous(limits = x_range,
                          labels = cfg$axis_format,
                          breaks = scales::breaks_pretty(n = 4)) +
        scale_y_continuous(limits = y_lim, expand = c(0, 0)) +
        labs(x = cfg$x_label, y = NULL) +
        theme_classic(base_size = 7) +
        theme(
            axis.title.x = element_text(size = 6.5, margin = margin(t = 1)),
            axis.text.x  = element_text(size = 5.5),
            axis.text.y  = element_blank(),
            axis.ticks.y = element_blank(),
            axis.line.y  = element_blank(),
            axis.line.x  = element_line(color = "#444444", linewidth = 0.4),
            panel.grid   = element_blank(),
            plot.margin  = margin(1, 2, 1, 2)
        )

    # ── RIGHT panel ─────────────────────────────────────────────────────
    p_right <- ggplot(df_s) +
        annotate("segment", x = 0, xend = right_em,
                y = divider_y, yend = divider_y,
                color = "#CCCCCC", linewidth = 0.4, linetype = "dashed") +
        geom_text(aes(x = x_padj_pos, y = y, label = p_str,
                     fontface = ifelse(is_sig, "bold", "plain"),
                     color    = ifelse(is_sig, "#222222", "#666666")),
                 hjust = 0.5, size = 2.2, family = "mono") +
        geom_text(aes(x = x_sig_pos, y = y, label = sig),
                 fontface = "bold", hjust = 0.5, size = 2.4,
                 family = "mono", color = "#222222") +
        annotate("text", x = x_padj_pos, y = header_y, label = h_padj,
                fontface = "bold", hjust = 0.5, size = 2.4, color = "#222222") +
        annotate("text", x = x_sig_pos, y = header_y, label = h_sig,
                fontface = "bold", hjust = 0.5, size = 2.4, color = "#222222") +
        annotate("segment", x = 0, xend = right_em,
                y = underline_y, yend = underline_y,
                color = "#333333", linewidth = 0.3) +
        scale_color_identity() +
        scale_x_continuous(limits = c(0, right_em), expand = c(0, 0)) +
        scale_y_continuous(limits = y_lim, expand = c(0, 0)) +
        theme_void() +
        theme(plot.margin = margin(2, 4, 2, 2))

    composed <- (p_left | p_forest | p_right) +
        plot_layout(widths = c(left_em, forest_em, right_em))

    fig_height <- max(FOREST_LAYOUT$min_fig_height_in,
                     FOREST_LAYOUT$base_height_in +
                         n_rows * FOREST_LAYOUT$row_height_in)

    slug <- sprintf("%s_modulescores_rlm_%s_forest", CELL_TYPE, s)

    save_figure(composed, slug = slug, width = fig_width, height = fig_height)

    options(repr.plot.width = fig_width + 1, repr.plot.height = fig_height + 0.5)
    tryCatch(
        print(composed),
        error = function(e) {
            cat(sprintf("    [inline preview unavailable: %s]\n",
                        conditionMessage(e)))
        }
    )

    cat(sprintf("    fig dims: %.2f x %.2f in (left_em=%.1f, forest_em=%.1f, right_em=%.1f)\n",
                fig_width, fig_height, left_em, forest_em, right_em))
}


# ─────────────────────────────────────────────────────────────────────────────
# §5.2.6 SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
n_sig_rows <- sum(mod_rlm_df$p_adj < STATISTICAL_PARAMS$fdr_threshold,
                 na.rm = TRUE)
n_converged <- sum(mod_rlm_df$converged, na.rm = TRUE)

cat("\n", strrep("-", 72), "\n", sep = "")
cat(sprintf("  §5.2 SUMMARY  |  %s / %s\n", DATASET, CELL_TYPE))
cat(strrep("-", 72), "\n", sep = "")
cat(sprintf("  Test          : Robust LM (lmrob, MM-estimator, KS2014)\n"))
cat(sprintf("  beta_scale    : score_units\n"))
cat(sprintf("  Adjustments   : Sex + Cohort + delta_log10_umi\n"))
cat(sprintf("  Converged     : %d / %d\n", n_converged, nrow(mod_rlm_df)))
cat(sprintf("  Significant   : %d of %d at BH-FDR < %.2f\n",
            n_sig_rows, nrow(mod_rlm_df), STATISTICAL_PARAMS$fdr_threshold))

if (n_sig_rows > 0) {
    cat(sprintf("\n  Significant modules (beta > 0 = SnC enriched):\n"))
    sig_rows <- mod_rlm_df %>%
        filter(p_adj < STATISTICAL_PARAMS$fdr_threshold) %>%
        arrange(stratum, desc(estimate))
    for (i in seq_len(nrow(sig_rows))) {
        r <- sig_rows[i, ]
        direction <- ifelse(r$estimate > 0, "+", "-")
        cat(sprintf("    %s %-22s [%s]  beta=%+.4f  p_adj=%.2e\n",
                    direction, r$outcome, r$stratum,
                    r$estimate, r$p_adj))
    }
}

cat(sprintf("\n  Per-stratum forests:\n"))
for (s in strata) {
    cat(sprintf("    %-22s -> %s_modulescores_rlm_%s_forest.{pdf,png,svg}\n",
                s, CELL_TYPE, s))
}
cat(strrep("-", 72), "\n", sep = "")
cat("\nv §5.2 module scores RLM complete\n")
cat("  Next: §5.3 -- OLS on paired diffs (classical, same formula)\n")

---
## 16 · Module scores — OLS on paired differences

**Why.** The non-robust twin of section 15, with a sign-agreement table at the end. Where RLM and OLS part company, influential donors are the reason.

**Test.** Classical LM (`lm`).  
**Outcome.** Donor-level paired difference in mean module score.  
**Formula.** `diff ~ Sex + Cohort + delta_log10_umi`  
**β.** Intercept.  
**FDR.** BH within stratum, across 10 modules.  

**Display.** Per-stratum forest, plus an RLM-vs-OLS sign agreement table.

In [ ]:
# MODULE 05 -- Senescence Enrichment & Cell Cycle Analysis
# §5.3 -- MODULE SCORES: OLS (paired diffs, classical)
# Stats:
#   Test:     Classical LM (lm)
#   Outcome:  donor-level paired difference (mean_SnC - mean_NonSnC)
#   Formula:  diff ~ Sex + Cohort + delta_log10_umi
#   beta:     intercept (= adjusted mean diff after covariate adjustment)
#   Scale:    score_units
#   FDR:      BH within (stratum) across 10 modules
#
# Modules tested in fixed canonical order (SLOAN_HALLMARK_NAMES, §0).
# Figure: per-stratum forest plot, fully adaptive layout (matches §5.2).
# Companion: RLM-vs-OLS sign agreement table at end.

cat("=", strrep("=", 71), "\n", sep = "")
cat(sprintf("§5.3 -- MODULE SCORES: OLS (paired diffs)  |  %s / %s\n",
            DATASET, CELL_TYPE))
cat("=", strrep("=", 71), "\n", sep = "")


# ─────────────────────────────────────────────────────────────────────────────
# §5.3.1 STATS
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> STATS\n")
cat("  Test     : Classical LM (lm)\n")
cat("  Outcome  : donor-level paired difference (mean_SnC - mean_NonSnC)\n")
cat("  Formula  : diff ~ Sex + Cohort + delta_log10_umi\n")
cat("  beta     : intercept (= adjusted mean diff)\n")
cat("  Scale    : score_units\n")
cat(sprintf("  Strata   : %s\n", paste(strata, collapse = ", ")))
cat(sprintf("  FDR      : %s within (stratum) across %d modules\n",
            STATISTICAL_PARAMS$fdr_method,
            length(SLOAN_HALLMARK_NAMES)))


# ─────────────────────────────────────────────────────────────────────────────
# §5.3.2 RUN
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> RUN\n")

if (!exists("score_col_lookup")) {
    md_cols <- colnames(md)
    score_col_lookup <- character(length(SLOAN_HALLMARK_NAMES))
    names(score_col_lookup) <- SLOAN_HALLMARK_NAMES
    for (mod in SLOAN_HALLMARK_NAMES) {
        candidates <- c(paste0("Score_", mod), paste0("score_", mod),
                       paste0("module_", mod), mod, paste0(mod, "1"))
        found <- intersect(candidates, md_cols)
        if (length(found) > 0) score_col_lookup[mod] <- found[1]
        else                   score_col_lookup[mod] <- NA_character_
    }
}

mod_ols_rows <- list()

for (stratum in strata) {
    md_s        <- filter_to_stratum(md, stratum, STUDY_GROUP_COL)
    arms_s      <- build_donor_arms(md_s, DONOR_COL, SEN_LABEL_COL,
                                    STATISTICAL_PARAMS$min_cells_per_group)
    paired_ids  <- unique(arms_s$donor[arms_s$paired])
    n_paired    <- length(paired_ids)

    if (n_paired < 5) {
        cat(sprintf("  [%s] only %d paired donors -- skipping\n",
                    stratum, n_paired))
        next
    }

    donor_meta_s <- build_donor_meta(
        md_s, paired_ids,
        DONOR_COL, SEN_LABEL_COL,
        SEX_COL, AGE_COL, COHORT_COL,
        has_cohort = HAS_COHORT
    )

    md_pair <- md_s[md_s[[DONOR_COL]] %in% paired_ids, , drop = FALSE]
    md_pair$is_snc_arm <- as.character(md_pair[[SEN_LABEL_COL]]) %in%
                          c("1", "TRUE", "True", "Senescent")

    cell_cnt_snc <- sum(arms_s$n_cells[arms_s$donor %in% paired_ids &
                                       arms_s$arm %in% c("1", "TRUE", "True", "Senescent")])
    cell_cnt_non <- sum(arms_s$n_cells[arms_s$donor %in% paired_ids &
                                       !(arms_s$arm %in% c("1", "TRUE", "True", "Senescent"))])

    for (mod in SLOAN_HALLMARK_NAMES) {
        score_col <- score_col_lookup[mod]
        if (is.na(score_col)) next

        per_donor <- md_pair %>%
            group_by(donor   = .data[[DONOR_COL]],
                     is_snc  = is_snc_arm) %>%
            summarise(mean_score = mean(.data[[score_col]], na.rm = TRUE),
                     .groups = "drop")

        wide_df <- per_donor %>%
            tidyr::pivot_wider(names_from   = is_snc,
                              values_from  = mean_score,
                              names_prefix = "score_")

        if (!"score_TRUE"  %in% names(wide_df)) next
        if (!"score_FALSE" %in% names(wide_df)) next
        wide_df <- wide_df[!is.na(wide_df$score_TRUE) &
                           !is.na(wide_df$score_FALSE), , drop = FALSE]
        if (nrow(wide_df) < 5) next

        wide_df$diff <- wide_df$score_TRUE - wide_df$score_FALSE

        reg_df <- wide_df %>% left_join(donor_meta_s, by = "donor")

        sex_levels    <- length(unique(reg_df[[SEX_COL]]))
        cohort_levels <- if (HAS_COHORT) length(unique(reg_df[[COHORT_COL]])) else 0

        rhs <- character(0)
        if (sex_levels    >= 2) rhs <- c(rhs, SEX_COL)
        if (cohort_levels >= 2) rhs <- c(rhs, COHORT_COL)
        rhs <- c(rhs, "delta_log10_umi")

        formula_str <- paste("diff ~", paste(rhs, collapse = " + "))
        formula_obj <- as.formula(formula_str)

        fit <- tryCatch({
            lm(formula_obj, data = reg_df)
        }, error = function(e) {
            cat(sprintf("  [%s | %s] lm ERROR: %s\n",
                        stratum, mod, e$message))
            NULL
        })

        if (is.null(fit)) next

        co <- summary(fit)$coefficients
        if (!"(Intercept)" %in% rownames(co)) next

        beta      <- co["(Intercept)", "Estimate"]
        se        <- co["(Intercept)", "Std. Error"]
        statistic <- co["(Intercept)", "t value"]
        p_value   <- co["(Intercept)", "Pr(>|t|)"]

        z <- qnorm(1 - (1 - STATISTICAL_PARAMS$confidence_level) / 2)
        ci_low  <- beta - z * se
        ci_high <- beta + z * se

        row <- tidy_model_results(
            stratum      = stratum,
            outcome      = mod,
            model        = "ols",
            n_donors     = nrow(wide_df),
            n_cells_test = cell_cnt_snc,
            n_cells_ref  = cell_cnt_non,
            estimate     = beta,
            se           = se,
            ci_low       = ci_low,
            ci_high      = ci_high,
            statistic    = statistic,
            p_value      = p_value,
            extra        = list(
                beta_scale         = "score_units",
                module_type        = SLOAN_LIST_TYPES[which(SLOAN_HALLMARK_NAMES == mod)],
                score_col          = score_col,
                formula            = formula_str,
                cohort_in_model    = cohort_levels >= 2,
                sex_in_model       = sex_levels >= 2,
                delta_umi_in_model = TRUE,
                r_squared          = summary(fit)$r.squared,
                adj_r_squared      = summary(fit)$adj.r.squared,
                n_covariate_terms  = nrow(co) - 1
            )
        )
        mod_ols_rows[[paste(stratum, mod, sep = "|")]] <- row
    }
}

mod_ols_df <- bind_rows(mod_ols_rows)


# ─────────────────────────────────────────────────────────────────────────────
# §5.3.3 BH-FDR within (stratum)
# ─────────────────────────────────────────────────────────────────────────────
mod_ols_df <- mod_ols_df %>%
    group_by(stratum) %>%
    mutate(p_adj = p.adjust(p_value, method = STATISTICAL_PARAMS$fdr_method)) %>%
    ungroup() %>%
    mutate(sig = sig_stars(p_adj))


# ─────────────────────────────────────────────────────────────────────────────
# §5.3.4 RESULTS
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> RESULTS  (beta = adjusted intercept, scale = score_units)\n")

cat(sprintf("\n  %-22s %-22s %5s %+10s %+10s %+10s %8s %8s %4s %6s\n",
            "Stratum", "Module", "N",
            "beta", "CI_low", "CI_high",
            "p_raw", "p_adj", "Sig", "R^2"))
cat("  ", strrep("-", 120), "\n", sep = "")

for (s in strata) {
    df_s <- mod_ols_df %>% filter(stratum == s)
    df_s <- df_s[match(SLOAN_HALLMARK_NAMES, df_s$outcome), ]
    df_s <- df_s[!is.na(df_s$stratum), ]

    for (i in seq_len(nrow(df_s))) {
        r <- df_s[i, ]
        cat(sprintf("  %-22s %-22s %5d %+9.4f %+9.4f %+9.4f %8.1e %8.1e %4s %6.3f\n",
                    substr(r$stratum, 1, 22), substr(r$outcome, 1, 22),
                    r$n_donors,
                    r$estimate, r$ci_low, r$ci_high,
                    r$p_value, r$p_adj, r$sig, r$r_squared))
    }
}

save_table(mod_ols_df, paste0(CELL_TYPE, "_modulescores_ols"))


# ─────────────────────────────────────────────────────────────────────────────
# §5.3.5 FIGURE -- per-stratum forest plot, FULLY ADAPTIVE layout
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> FIGURE\n")

if (!exists("FOREST_LAYOUT")) {
    FOREST_LAYOUT <- list(
        em_per_char_mono   = 0.62,
        em_per_char_prop   = 0.55,
        inter_col_gap      = 1.5,
        panel_right_pad    = 0.8,
        forest_share_mult  = 0.7,
        inches_per_em      = 0.020,
        min_fig_width_in   = 6.0,
        min_fig_height_in  = 2.5,
        row_height_in      = 0.30,
        base_height_in     = 0.9,
        pad_x_axis_frac    = 0.10,
        pad_x_axis_null    = 0.02
    )
}

cfg <- EFFECT_CONFIGS$score_units

divider_after_idx <- sum(SLOAN_LIST_TYPES == "Individual hallmark")  # = 8

for (s in strata) {
    df_s <- mod_ols_df %>%
        filter(stratum == s) %>%
        mutate(outcome = factor(outcome, levels = SLOAN_HALLMARK_NAMES)) %>%
        arrange(outcome)

    if (nrow(df_s) == 0) {
        cat(sprintf("  [%s] no rows -- skipping figure\n", s))
        next
    }

    df_s$row_label  <- as.character(df_s$outcome)
    df_s$eff_str    <- sapply(df_s$estimate, cfg$fmt_effect)
    df_s$ci_str     <- mapply(cfg$fmt_ci, df_s$ci_low, df_s$ci_high)
    df_s$p_str      <- sapply(df_s$p_adj, fmt_p_short)
    df_s$row_color  <- SLOAN_HALLMARK_COLORS[as.character(df_s$outcome)]
    df_s$is_sig     <- df_s$sig != "ns"

    n_rows <- nrow(df_s)

    # Adaptive column widths
    h_mod  <- "Module"
    h_eff  <- cfg$eff_h_label
    h_ci   <- cfg$ci_h_label
    h_padj <- "p(adj)"
    h_sig  <- "Sig"

    nchar_label <- max(nchar(c(df_s$row_label, h_mod)),  na.rm = TRUE)
    nchar_eff   <- max(nchar(c(df_s$eff_str,   h_eff)),  na.rm = TRUE)
    nchar_ci    <- max(nchar(c(df_s$ci_str,    h_ci)),   na.rm = TRUE)
    nchar_padj  <- max(nchar(c(df_s$p_str,     h_padj)), na.rm = TRUE)
    nchar_sig   <- max(nchar(c(df_s$sig,       h_sig)),  na.rm = TRUE)

    em_label <- nchar_label * FOREST_LAYOUT$em_per_char_prop
    em_eff   <- nchar_eff   * FOREST_LAYOUT$em_per_char_mono
    em_ci    <- nchar_ci    * FOREST_LAYOUT$em_per_char_mono
    em_padj  <- nchar_padj  * FOREST_LAYOUT$em_per_char_mono
    em_sig   <- nchar_sig   * FOREST_LAYOUT$em_per_char_mono

    gap <- FOREST_LAYOUT$inter_col_gap

    x_label_pos <- 0
    x_eff_pos   <- em_label + gap
    x_ci_pos    <- em_label + gap + em_eff + gap
    left_em     <- em_label + gap + em_eff + gap + em_ci + FOREST_LAYOUT$panel_right_pad

    x_padj_pos  <- em_padj / 2
    x_sig_pos   <- em_padj + gap + em_sig / 2
    right_em    <- em_padj + gap + em_sig + FOREST_LAYOUT$panel_right_pad

    forest_em <- (left_em + right_em) * FOREST_LAYOUT$forest_share_mult
    total_em  <- left_em + forest_em + right_em
    fig_width <- max(total_em * FOREST_LAYOUT$inches_per_em,
                    FOREST_LAYOUT$min_fig_width_in)

    df_s$y      <- n_rows:1
    header_y    <- n_rows + 0.55
    underline_y <- n_rows + 0.20
    y_lim       <- c(0.4, n_rows + 1.0)
    divider_y   <- (n_rows - divider_after_idx) + 0.5

    stratum_color <- if (s %in% names(STUDY_GROUP_COLORS)) STUDY_GROUP_COLORS[[s]] else "#7F7F7F"

    n_sig <- sum(df_s$is_sig)
    title_txt <- sprintf("Module scores OLS | %s | [%s] | %d/%d sig at BH-FDR",
                         CELL_TYPE, s, n_sig, nrow(df_s))

    p_left <- ggplot(df_s) +
        annotate("segment", x = 0, xend = left_em,
                y = divider_y, yend = divider_y,
                color = "#CCCCCC", linewidth = 0.4, linetype = "dashed") +
        geom_text(aes(x = x_label_pos, y = y, label = row_label,
                     color = row_color,
                     fontface = ifelse(is_sig, "bold", "plain")),
                 hjust = 0, size = 2.4) +
        geom_text(aes(x = x_eff_pos, y = y, label = eff_str,
                     fontface = ifelse(is_sig, "bold", "plain")),
                 hjust = 0, size = 2.2, family = "mono", color = "#222222") +
        geom_text(aes(x = x_ci_pos, y = y, label = ci_str),
                 hjust = 0, size = 2.0, family = "mono", color = "#666666") +
        annotate("text", x = x_label_pos, y = header_y, label = h_mod,
                fontface = "bold", hjust = 0, size = 2.4, color = "#222222") +
        annotate("text", x = x_eff_pos, y = header_y, label = h_eff,
                fontface = "bold", hjust = 0, size = 2.4, color = "#222222") +
        annotate("text", x = x_ci_pos, y = header_y, label = h_ci,
                fontface = "bold", hjust = 0, size = 2.2, color = "#222222") +
        annotate("segment", x = 0, xend = left_em,
                y = underline_y, yend = underline_y,
                color = "#333333", linewidth = 0.3) +
        scale_color_identity() +
        scale_x_continuous(limits = c(0, left_em), expand = c(0, 0)) +
        scale_y_continuous(limits = y_lim, expand = c(0, 0)) +
        labs(title = title_txt) +
        theme_void() +
        theme(
            plot.title = element_text(size = 9, face = "bold",
                                     color = stratum_color,
                                     hjust = 0,
                                     margin = margin(t = 2, b = 1)),
            plot.margin = margin(2, 2, 2, 4)
        )

    eff_vals    <- c(df_s$estimate, df_s$ci_low, df_s$ci_high)
    eff_finite  <- eff_vals[is.finite(eff_vals)]
    if (length(eff_finite) == 0) eff_finite <- c(-0.001, 0.001)

    data_range  <- diff(range(eff_finite))
    if (data_range == 0) data_range <- max(abs(eff_finite[1]), 1e-6) * 0.2

    pad         <- data_range * FOREST_LAYOUT$pad_x_axis_frac
    null_pad    <- data_range * FOREST_LAYOUT$pad_x_axis_null
    x_range     <- range(eff_finite) + c(-pad, pad)
    x_range[1]  <- min(x_range[1], cfg$null_value - null_pad)
    x_range[2]  <- max(x_range[2], cfg$null_value + null_pad)

    p_forest <- ggplot(df_s) +
        annotate("segment", x = -Inf, xend = Inf,
                y = divider_y, yend = divider_y,
                color = "#CCCCCC", linewidth = 0.4, linetype = "dashed") +
        geom_vline(xintercept = cfg$null_value,
                  linetype = "dashed", color = "#999999", linewidth = 0.4) +
        geom_errorbar(aes(y = y, xmin = ci_low, xmax = ci_high),
                     width = 0.18, linewidth = 0.4, color = "#4D4D4D") +
        geom_point(aes(x = estimate, y = y, fill = row_color,
                      size = ifelse(is_sig, 3.5, 2.5)),
                  shape = 23, color = "#222222", stroke = 0.4) +
        scale_fill_identity() +
        scale_size_identity() +
        scale_x_continuous(limits = x_range,
                          labels = cfg$axis_format,
                          breaks = scales::breaks_pretty(n = 4)) +
        scale_y_continuous(limits = y_lim, expand = c(0, 0)) +
        labs(x = cfg$x_label, y = NULL) +
        theme_classic(base_size = 7) +
        theme(
            axis.title.x = element_text(size = 6.5, margin = margin(t = 1)),
            axis.text.x  = element_text(size = 5.5),
            axis.text.y  = element_blank(),
            axis.ticks.y = element_blank(),
            axis.line.y  = element_blank(),
            axis.line.x  = element_line(color = "#444444", linewidth = 0.4),
            panel.grid   = element_blank(),
            plot.margin  = margin(1, 2, 1, 2)
        )

    p_right <- ggplot(df_s) +
        annotate("segment", x = 0, xend = right_em,
                y = divider_y, yend = divider_y,
                color = "#CCCCCC", linewidth = 0.4, linetype = "dashed") +
        geom_text(aes(x = x_padj_pos, y = y, label = p_str,
                     fontface = ifelse(is_sig, "bold", "plain"),
                     color    = ifelse(is_sig, "#222222", "#666666")),
                 hjust = 0.5, size = 2.2, family = "mono") +
        geom_text(aes(x = x_sig_pos, y = y, label = sig),
                 fontface = "bold", hjust = 0.5, size = 2.4,
                 family = "mono", color = "#222222") +
        annotate("text", x = x_padj_pos, y = header_y, label = h_padj,
                fontface = "bold", hjust = 0.5, size = 2.4, color = "#222222") +
        annotate("text", x = x_sig_pos, y = header_y, label = h_sig,
                fontface = "bold", hjust = 0.5, size = 2.4, color = "#222222") +
        annotate("segment", x = 0, xend = right_em,
                y = underline_y, yend = underline_y,
                color = "#333333", linewidth = 0.3) +
        scale_color_identity() +
        scale_x_continuous(limits = c(0, right_em), expand = c(0, 0)) +
        scale_y_continuous(limits = y_lim, expand = c(0, 0)) +
        theme_void() +
        theme(plot.margin = margin(2, 4, 2, 2))

    composed <- (p_left | p_forest | p_right) +
        plot_layout(widths = c(left_em, forest_em, right_em))

    fig_height <- max(FOREST_LAYOUT$min_fig_height_in,
                     FOREST_LAYOUT$base_height_in +
                         n_rows * FOREST_LAYOUT$row_height_in)

    slug <- sprintf("%s_modulescores_ols_%s_forest", CELL_TYPE, s)

    save_figure(composed, slug = slug, width = fig_width, height = fig_height)

    options(repr.plot.width = fig_width + 1, repr.plot.height = fig_height + 0.5)
    tryCatch(
        print(composed),
        error = function(e) {
            cat(sprintf("    [inline preview unavailable: %s]\n",
                        conditionMessage(e)))
        }
    )

    cat(sprintf("    fig dims: %.2f x %.2f in (left_em=%.1f, forest_em=%.1f, right_em=%.1f)\n",
                fig_width, fig_height, left_em, forest_em, right_em))
}


# ─────────────────────────────────────────────────────────────────────────────
# §5.3.6 RLM vs OLS direction agreement (TEXT TABLE ONLY)
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> AGREEMENT WITH §5.2 RLM (text only -- no comparison figure)\n")

mod_agreement <- mod_rlm_df %>%
    select(stratum, outcome, beta_rlm = estimate, p_rlm = p_value) %>%
    inner_join(
        mod_ols_df %>% select(stratum, outcome, beta_ols = estimate, p_ols = p_value),
        by = c("stratum", "outcome")
    ) %>%
    mutate(
        same_sign     = sign(beta_rlm) == sign(beta_ols),
        ratio_rlm_ols = beta_rlm / beta_ols,
        abs_diff      = abs(beta_rlm - beta_ols)
    )

cat(sprintf("\n  %-22s %-22s %+10s %+10s %5s %10s\n",
            "Stratum", "Module", "beta_rlm", "beta_ols", "Same", "AbsDiff"))
cat("  ", strrep("-", 90), "\n", sep = "")

# Print in fixed canonical order
for (s in strata) {
    df_s <- mod_agreement %>% filter(stratum == s)
    df_s <- df_s[match(SLOAN_HALLMARK_NAMES, df_s$outcome), ]
    df_s <- df_s[!is.na(df_s$stratum), ]

    for (i in seq_len(nrow(df_s))) {
        r <- df_s[i, ]
        cat(sprintf("  %-22s %-22s %+9.4f %+9.4f %5s %10.4f\n",
                    substr(r$stratum, 1, 22), substr(r$outcome, 1, 22),
                    r$beta_rlm, r$beta_ols,
                    ifelse(r$same_sign, "yes", "FLIP"),
                    r$abs_diff))
    }
}

n_sign_disagree <- sum(!mod_agreement$same_sign)
if (n_sign_disagree == 0) {
    cat(sprintf("\n  v All %d (stratum x module) results agree in sign between RLM and OLS\n",
                nrow(mod_agreement)))
} else {
    cat(sprintf("\n  ! %d / %d (stratum x module) results FLIP sign between RLM and OLS\n",
                n_sign_disagree, nrow(mod_agreement)))
    cat("    Sign disagreement = outlier-driven; trust RLM\n")
}


# ─────────────────────────────────────────────────────────────────────────────
# §5.3.7 SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
n_sig_rows <- sum(mod_ols_df$p_adj < STATISTICAL_PARAMS$fdr_threshold,
                 na.rm = TRUE)

cat("\n", strrep("-", 72), "\n", sep = "")
cat(sprintf("  §5.3 SUMMARY  |  %s / %s\n", DATASET, CELL_TYPE))
cat(strrep("-", 72), "\n", sep = "")
cat(sprintf("  Test          : Classical LM (lm)\n"))
cat(sprintf("  beta_scale    : score_units\n"))
cat(sprintf("  Adjustments   : Sex + Cohort + delta_log10_umi\n"))
cat(sprintf("  Significant   : %d of %d at BH-FDR < %.2f\n",
            n_sig_rows, nrow(mod_ols_df), STATISTICAL_PARAMS$fdr_threshold))
cat(sprintf("  Sign agreement (vs RLM): %d / %d\n",
            sum(mod_agreement$same_sign), nrow(mod_agreement)))
cat(sprintf("  Mean R^2: %.3f\n",
            mean(mod_ols_df$r_squared, na.rm = TRUE)))

if (n_sig_rows > 0) {
    cat(sprintf("\n  Significant modules (beta > 0 = SnC enriched):\n"))
    sig_rows <- mod_ols_df %>%
        filter(p_adj < STATISTICAL_PARAMS$fdr_threshold) %>%
        arrange(stratum, desc(estimate))
    for (i in seq_len(nrow(sig_rows))) {
        r <- sig_rows[i, ]
        direction <- ifelse(r$estimate > 0, "+", "-")
        cat(sprintf("    %s %-22s [%s]  beta=%+.4f  p_adj=%.2e\n",
                    direction, r$outcome, r$stratum,
                    r$estimate, r$p_adj))
    }
}

cat(sprintf("\n  Per-stratum forests:\n"))
for (s in strata) {
    cat(sprintf("    %-22s -> %s_modulescores_ols_%s_forest.{pdf,png,svg}\n",
                s, CELL_TYPE, s))
}
cat(strrep("-", 72), "\n", sep = "")
cat("\nv §5.3 module scores OLS complete\n")
cat("  Next: §5.4 -- LMM (cell-level Gaussian, donor random effect)\n")

---
## 17 · Module scores — LMM (cell-level Gaussian)

**Why.** The cell-level analogue of sections 15-16. Module scores are continuous, so this is a Gaussian LMM with Satterthwaite degrees of freedom rather than the binomial GLMM used for phases. `(1 | donor)` is what makes the cell-level n legitimate.

**Test.** Gaussian LMM, `lmerTest::lmer`, Satterthwaite df.  
**Outcome.** Cell-level module score.  
**Formula.** `score ~ is_senescent + Sex + Cohort + log10(nCount_RNA) + (1 | donor)`  
**β.** Adjusted cell-level score difference, `SnC − Non-SnC`.  
**FDR.** BH within stratum, across 10 modules.  

**Display.** Per-stratum forest, matching sections 15-16.

In [ ]:
# MODULE 05 -- Senescence Enrichment & Cell Cycle Analysis
# §5.4 -- MODULE SCORES: LMM (cell-level Gaussian, donor random effect)
# Stats:
#   Test:      LMM (Gaussian) via lmer + lmerTest (Satterthwaite df)
#   Outcome:   continuous module score (cell-level)
#   Formula:   score ~ is_senescent + Sex + Cohort + log10(nCount_RNA) + (1|donor)
#   Family:    Gaussian (identity link)
#   beta:      adjusted score difference (SnC - Non-SnC) at the cell level
#   Scale:     score_units
#   Inference: Wald-like t with Satterthwaite df (lmerTest)
#   FDR:       BH within (stratum) across 10 modules
#
# This is the cell-level ANALOG of §5.2/§5.3 donor-level paired-diff models.
# Random intercept (1|donor) properly accounts for cells-from-same-donor
# correlation; without it, p-values would be artificially small.
#
# Modules in fixed canonical order from SLOAN_HALLMARK_NAMES.
# Figure: per-stratum forest plot, fully adaptive layout (matches §5.2/§5.3).

cat("=", strrep("=", 71), "\n", sep = "")
cat(sprintf("§5.4 -- MODULE SCORES: LMM (cell-level Gaussian)  |  %s / %s\n",
            DATASET, CELL_TYPE))
cat("=", strrep("=", 71), "\n", sep = "")


# ─────────────────────────────────────────────────────────────────────────────
# §5.4.1 STATS
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> STATS\n")
cat("  Test     : LMM (Gaussian, identity link) via lmer + lmerTest\n")
cat("  Formula  : score ~ is_senescent + Sex + Cohort + log10(nCount_RNA) + (1|donor)\n")
cat("  Family   : Gaussian\n")
cat("  beta     : beta_SnC on score_units scale\n")
cat("  Scale    : score_units\n")
cat("  Inference: Satterthwaite-approximated t-test (lmerTest)\n")
cat(sprintf("  Strata   : %s\n", paste(strata, collapse = ", ")))
cat(sprintf("  FDR      : %s within (stratum) across %d modules\n",
            STATISTICAL_PARAMS$fdr_method,
            length(SLOAN_HALLMARK_NAMES)))


# ─────────────────────────────────────────────────────────────────────────────
# §5.4.2 RUN
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> RUN\n")
cat(sprintf("  Note: %d LMM fits (10 modules x 3 strata) at ~5-15 sec each. Total ~3-8 min.\n",
            length(SLOAN_HALLMARK_NAMES) * length(strata)))

# Score column lookup -- assumes §5.1 / §5.2 has run
if (!exists("score_col_lookup")) {
    md_cols <- colnames(md)
    score_col_lookup <- character(length(SLOAN_HALLMARK_NAMES))
    names(score_col_lookup) <- SLOAN_HALLMARK_NAMES
    for (mod in SLOAN_HALLMARK_NAMES) {
        candidates <- c(paste0("Score_", mod), paste0("score_", mod),
                       paste0("module_", mod), mod, paste0(mod, "1"))
        found <- intersect(candidates, md_cols)
        if (length(found) > 0) score_col_lookup[mod] <- found[1]
        else                   score_col_lookup[mod] <- NA_character_
    }
}

mod_lmm_rows <- list()

for (stratum in strata) {
    cat(sprintf("\n  [%s]\n", stratum))

    md_s        <- filter_to_stratum(md, stratum, STUDY_GROUP_COL)
    arms_s      <- build_donor_arms(md_s, DONOR_COL, SEN_LABEL_COL,
                                    STATISTICAL_PARAMS$min_cells_per_group)
    paired_ids  <- unique(arms_s$donor[arms_s$paired])
    n_paired    <- length(paired_ids)

    if (n_paired < 5) {
        cat(sprintf("    only %d paired donors -- skipping\n", n_paired))
        next
    }

    # Build cell-level data for this stratum (pre-extracted once, reused
    # across all 10 modules)
    md_pair <- md_s[md_s[[DONOR_COL]] %in% paired_ids, , drop = FALSE]
    md_pair$is_snc_arm <- as.integer(
        as.character(md_pair[[SEN_LABEL_COL]]) %in%
            c("1", "TRUE", "True", "Senescent")
    )
    md_pair$donor      <- as.factor(md_pair[[DONOR_COL]])
    md_pair$Sex_factor <- as.factor(md_pair[[SEX_COL]])
    if (HAS_COHORT) {
        md_pair$Cohort_factor <- as.factor(md_pair[[COHORT_COL]])
    }
    md_pair$log10_umi <- log10(md_pair$nCount_RNA)

    sex_levels    <- length(unique(md_pair$Sex_factor))
    cohort_levels <- if (HAS_COHORT) length(unique(md_pair$Cohort_factor)) else 0

    rhs_fixed <- c("is_snc_arm")
    if (sex_levels    >= 2) rhs_fixed <- c(rhs_fixed, "Sex_factor")
    if (cohort_levels >= 2) rhs_fixed <- c(rhs_fixed, "Cohort_factor")
    rhs_fixed <- c(rhs_fixed, "log10_umi")

    fixed_str  <- paste(rhs_fixed, collapse = " + ")
    random_str <- "(1 | donor)"

    cell_cnt_snc <- sum(md_pair$is_snc_arm == 1)
    cell_cnt_non <- sum(md_pair$is_snc_arm == 0)

    cat(sprintf("    cells: %s SnC + %s Non-SnC = %s total\n",
                fmt_n(cell_cnt_snc), fmt_n(cell_cnt_non),
                fmt_n(cell_cnt_snc + cell_cnt_non)))
    cat(sprintf("    fixed: %s\n", fixed_str))
    cat(sprintf("    n_donors (random levels): %d\n", n_paired))

    for (mod in SLOAN_HALLMARK_NAMES) {
        score_col <- score_col_lookup[mod]
        if (is.na(score_col)) {
            cat(sprintf("    [%s | %s] no score col -- skipping\n", stratum, mod))
            next
        }

        cat(sprintf("    [%s | %-22s] fitting lmer ...", stratum, mod))
        t0 <- Sys.time()

        # Subset to non-NA score values
        keep_rows <- !is.na(md_pair[[score_col]])
        md_fit    <- md_pair[keep_rows, , drop = FALSE]
        md_fit$y  <- md_fit[[score_col]]

        if (nrow(md_fit) < 100) {
            cat(sprintf(" only %d cells with non-NA score -- skipping\n", nrow(md_fit)))
            next
        }

        formula_str <- paste("y ~", fixed_str, "+", random_str)
        formula_obj <- as.formula(formula_str)

        fit <- tryCatch({
            lmerTest::lmer(formula_obj,
                          data    = md_fit,
                          REML    = TRUE,
                          control = lmerControl(optimizer = "bobyqa",
                                                optCtrl   = list(maxfun = 100000)))
        }, error = function(e) {
            cat(sprintf(" ERROR: %s\n", e$message))
            NULL
        }, warning = function(w) {
            tryCatch(lmerTest::lmer(formula_obj,
                                   data    = md_fit,
                                   REML    = TRUE,
                                   control = lmerControl(optimizer = "bobyqa",
                                                         optCtrl   = list(maxfun = 100000))),
                    error = function(e) NULL)
        })

        elapsed <- as.numeric(difftime(Sys.time(), t0, units = "secs"))

        if (is.null(fit)) {
            cat(sprintf(" failed after %s\n", fmt_elapsed(elapsed)))
            next
        }
        cat(sprintf(" %s\n", fmt_elapsed(elapsed)))

        co <- summary(fit)$coefficients
        snc_term_idx <- grep("^is_snc_arm$", rownames(co))
        if (length(snc_term_idx) == 0) {
            cat("        No is_snc_arm coefficient found -- skipping\n")
            next
        }

        beta_snc  <- co[snc_term_idx, "Estimate"]
        se        <- co[snc_term_idx, "Std. Error"]
        # lmerTest provides "df" and "Pr(>|t|)"; fall back to Wald if absent
        if ("Pr(>|t|)" %in% colnames(co)) {
            statistic <- co[snc_term_idx, "t value"]
            p_value   <- co[snc_term_idx, "Pr(>|t|)"]
            df_used   <- if ("df" %in% colnames(co)) co[snc_term_idx, "df"] else NA_real_
            inference_method <- "satterthwaite"
        } else {
            statistic <- co[snc_term_idx, "t value"]
            p_value   <- 2 * pnorm(-abs(statistic))
            df_used   <- NA_real_
            inference_method <- "wald_normal"
        }

        z <- qnorm(1 - (1 - STATISTICAL_PARAMS$confidence_level) / 2)
        ci_low  <- beta_snc - z * se
        ci_high <- beta_snc + z * se

        conv_msg  <- fit@optinfo$conv$lme4$messages
        converged <- length(conv_msg) == 0
        n_warns   <- length(conv_msg)

        # Variance components (donor random effect)
        vc <- as.data.frame(VarCorr(fit))
        donor_var  <- vc$vcov[vc$grp == "donor"][1]
        resid_var  <- vc$vcov[vc$grp == "Residual"][1]
        icc        <- if (!is.na(donor_var) && !is.na(resid_var)) donor_var / (donor_var + resid_var) else NA_real_

        row <- tidy_model_results(
            stratum      = stratum,
            outcome      = mod,
            model        = "lmm",
            n_donors     = n_paired,
            n_cells_test = cell_cnt_snc,
            n_cells_ref  = cell_cnt_non,
            estimate     = beta_snc,
            se           = se,
            ci_low       = ci_low,
            ci_high      = ci_high,
            statistic    = statistic,
            p_value      = p_value,
            extra        = list(
                beta_scale         = "score_units",
                module_type        = SLOAN_LIST_TYPES[which(SLOAN_HALLMARK_NAMES == mod)],
                score_col          = score_col,
                formula            = formula_str,
                inference_method   = inference_method,
                df_used            = df_used,
                converged          = converged,
                n_warnings         = n_warns,
                fit_seconds        = elapsed,
                cohort_in_model    = cohort_levels >= 2,
                sex_in_model       = sex_levels >= 2,
                n_cells_total      = nrow(md_fit),
                n_random_levels    = length(unique(md_fit$donor)),
                donor_var          = donor_var,
                resid_var          = resid_var,
                icc                = icc
            )
        )
        mod_lmm_rows[[paste(stratum, mod, sep = "|")]] <- row
    }

    rm(md_pair); gc(verbose = FALSE)
}

mod_lmm_df <- bind_rows(mod_lmm_rows)


# ─────────────────────────────────────────────────────────────────────────────
# §5.4.3 BH-FDR within (stratum)
# ─────────────────────────────────────────────────────────────────────────────
mod_lmm_df <- mod_lmm_df %>%
    group_by(stratum) %>%
    mutate(p_adj = p.adjust(p_value, method = STATISTICAL_PARAMS$fdr_method)) %>%
    ungroup() %>%
    mutate(sig = sig_stars(p_adj))


# ─────────────────────────────────────────────────────────────────────────────
# §5.4.4 RESULTS
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> RESULTS  (beta_SnC on score_units scale, lmer + Satterthwaite)\n")

cat(sprintf("\n  %-22s %-22s %5s %+10s %+10s %+10s %5s %8s %8s %4s\n",
            "Stratum", "Module", "N",
            "beta", "CI_low", "CI_high",
            "ICC",
            "p_raw", "p_adj", "Sig"))
cat("  ", strrep("-", 115), "\n", sep = "")

for (s in strata) {
    df_s <- mod_lmm_df %>% filter(stratum == s)
    df_s <- df_s[match(SLOAN_HALLMARK_NAMES, df_s$outcome), ]
    df_s <- df_s[!is.na(df_s$stratum), ]

    for (i in seq_len(nrow(df_s))) {
        r <- df_s[i, ]
        cat(sprintf("  %-22s %-22s %5d %+9.4f %+9.4f %+9.4f %5.2f %8.1e %8.1e %4s\n",
                    substr(r$stratum, 1, 22), substr(r$outcome, 1, 22),
                    r$n_donors,
                    r$estimate, r$ci_low, r$ci_high,
                    r$icc,
                    r$p_value, r$p_adj, r$sig))
    }
}

save_table(mod_lmm_df, paste0(CELL_TYPE, "_modulescores_lmm"))


# ─────────────────────────────────────────────────────────────────────────────
# §5.4.5 FIGURE -- per-stratum forest plot, FULLY ADAPTIVE layout
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> FIGURE\n")

if (!exists("FOREST_LAYOUT")) {
    FOREST_LAYOUT <- list(
        em_per_char_mono   = 0.62,
        em_per_char_prop   = 0.55,
        inter_col_gap      = 1.5,
        panel_right_pad    = 0.8,
        forest_share_mult  = 0.7,
        inches_per_em      = 0.020,
        min_fig_width_in   = 6.0,
        min_fig_height_in  = 2.5,
        row_height_in      = 0.30,
        base_height_in     = 0.9,
        pad_x_axis_frac    = 0.10,
        pad_x_axis_null    = 0.02
    )
}

cfg <- EFFECT_CONFIGS$score_units

divider_after_idx <- sum(SLOAN_LIST_TYPES == "Individual hallmark")  # = 8

for (s in strata) {
    df_s <- mod_lmm_df %>%
        filter(stratum == s) %>%
        mutate(outcome = factor(outcome, levels = SLOAN_HALLMARK_NAMES)) %>%
        arrange(outcome)

    if (nrow(df_s) == 0) {
        cat(sprintf("  [%s] no rows -- skipping figure\n", s))
        next
    }

    df_s$row_label  <- as.character(df_s$outcome)
    df_s$eff_str    <- sapply(df_s$estimate, cfg$fmt_effect)
    df_s$ci_str     <- mapply(cfg$fmt_ci, df_s$ci_low, df_s$ci_high)
    df_s$p_str      <- sapply(df_s$p_adj, fmt_p_short)
    df_s$row_color  <- SLOAN_HALLMARK_COLORS[as.character(df_s$outcome)]
    df_s$is_sig     <- df_s$sig != "ns"

    n_rows <- nrow(df_s)

    h_mod  <- "Module"
    h_eff  <- cfg$eff_h_label
    h_ci   <- cfg$ci_h_label
    h_padj <- "p(adj)"
    h_sig  <- "Sig"

    nchar_label <- max(nchar(c(df_s$row_label, h_mod)),  na.rm = TRUE)
    nchar_eff   <- max(nchar(c(df_s$eff_str,   h_eff)),  na.rm = TRUE)
    nchar_ci    <- max(nchar(c(df_s$ci_str,    h_ci)),   na.rm = TRUE)
    nchar_padj  <- max(nchar(c(df_s$p_str,     h_padj)), na.rm = TRUE)
    nchar_sig   <- max(nchar(c(df_s$sig,       h_sig)),  na.rm = TRUE)

    em_label <- nchar_label * FOREST_LAYOUT$em_per_char_prop
    em_eff   <- nchar_eff   * FOREST_LAYOUT$em_per_char_mono
    em_ci    <- nchar_ci    * FOREST_LAYOUT$em_per_char_mono
    em_padj  <- nchar_padj  * FOREST_LAYOUT$em_per_char_mono
    em_sig   <- nchar_sig   * FOREST_LAYOUT$em_per_char_mono

    gap <- FOREST_LAYOUT$inter_col_gap

    x_label_pos <- 0
    x_eff_pos   <- em_label + gap
    x_ci_pos    <- em_label + gap + em_eff + gap
    left_em     <- em_label + gap + em_eff + gap + em_ci + FOREST_LAYOUT$panel_right_pad

    x_padj_pos  <- em_padj / 2
    x_sig_pos   <- em_padj + gap + em_sig / 2
    right_em    <- em_padj + gap + em_sig + FOREST_LAYOUT$panel_right_pad

    forest_em <- (left_em + right_em) * FOREST_LAYOUT$forest_share_mult
    total_em  <- left_em + forest_em + right_em
    fig_width <- max(total_em * FOREST_LAYOUT$inches_per_em,
                    FOREST_LAYOUT$min_fig_width_in)

    df_s$y      <- n_rows:1
    header_y    <- n_rows + 0.55
    underline_y <- n_rows + 0.20
    y_lim       <- c(0.4, n_rows + 1.0)
    divider_y   <- (n_rows - divider_after_idx) + 0.5

    stratum_color <- if (s %in% names(STUDY_GROUP_COLORS)) STUDY_GROUP_COLORS[[s]] else "#7F7F7F"

    n_sig <- sum(df_s$is_sig)
    title_txt <- sprintf("Module scores LMM | %s | [%s] | %d/%d sig at BH-FDR",
                         CELL_TYPE, s, n_sig, nrow(df_s))

    p_left <- ggplot(df_s) +
        annotate("segment", x = 0, xend = left_em,
                y = divider_y, yend = divider_y,
                color = "#CCCCCC", linewidth = 0.4, linetype = "dashed") +
        geom_text(aes(x = x_label_pos, y = y, label = row_label,
                     color = row_color,
                     fontface = ifelse(is_sig, "bold", "plain")),
                 hjust = 0, size = 2.4) +
        geom_text(aes(x = x_eff_pos, y = y, label = eff_str,
                     fontface = ifelse(is_sig, "bold", "plain")),
                 hjust = 0, size = 2.2, family = "mono", color = "#222222") +
        geom_text(aes(x = x_ci_pos, y = y, label = ci_str),
                 hjust = 0, size = 2.0, family = "mono", color = "#666666") +
        annotate("text", x = x_label_pos, y = header_y, label = h_mod,
                fontface = "bold", hjust = 0, size = 2.4, color = "#222222") +
        annotate("text", x = x_eff_pos, y = header_y, label = h_eff,
                fontface = "bold", hjust = 0, size = 2.4, color = "#222222") +
        annotate("text", x = x_ci_pos, y = header_y, label = h_ci,
                fontface = "bold", hjust = 0, size = 2.2, color = "#222222") +
        annotate("segment", x = 0, xend = left_em,
                y = underline_y, yend = underline_y,
                color = "#333333", linewidth = 0.3) +
        scale_color_identity() +
        scale_x_continuous(limits = c(0, left_em), expand = c(0, 0)) +
        scale_y_continuous(limits = y_lim, expand = c(0, 0)) +
        labs(title = title_txt) +
        theme_void() +
        theme(
            plot.title = element_text(size = 9, face = "bold",
                                     color = stratum_color,
                                     hjust = 0,
                                     margin = margin(t = 2, b = 1)),
            plot.margin = margin(2, 2, 2, 4)
        )

    eff_vals    <- c(df_s$estimate, df_s$ci_low, df_s$ci_high)
    eff_finite  <- eff_vals[is.finite(eff_vals)]
    if (length(eff_finite) == 0) eff_finite <- c(-0.001, 0.001)

    data_range  <- diff(range(eff_finite))
    if (data_range == 0) data_range <- max(abs(eff_finite[1]), 1e-6) * 0.2

    pad         <- data_range * FOREST_LAYOUT$pad_x_axis_frac
    null_pad    <- data_range * FOREST_LAYOUT$pad_x_axis_null
    x_range     <- range(eff_finite) + c(-pad, pad)
    x_range[1]  <- min(x_range[1], cfg$null_value - null_pad)
    x_range[2]  <- max(x_range[2], cfg$null_value + null_pad)

    p_forest <- ggplot(df_s) +
        annotate("segment", x = -Inf, xend = Inf,
                y = divider_y, yend = divider_y,
                color = "#CCCCCC", linewidth = 0.4, linetype = "dashed") +
        geom_vline(xintercept = cfg$null_value,
                  linetype = "dashed", color = "#999999", linewidth = 0.4) +
        geom_errorbar(aes(y = y, xmin = ci_low, xmax = ci_high),
                     width = 0.18, linewidth = 0.4, color = "#4D4D4D") +
        geom_point(aes(x = estimate, y = y, fill = row_color,
                      size = ifelse(is_sig, 3.5, 2.5)),
                  shape = 23, color = "#222222", stroke = 0.4) +
        scale_fill_identity() +
        scale_size_identity() +
        scale_x_continuous(limits = x_range,
                          labels = cfg$axis_format,
                          breaks = scales::breaks_pretty(n = 4)) +
        scale_y_continuous(limits = y_lim, expand = c(0, 0)) +
        labs(x = cfg$x_label, y = NULL) +
        theme_classic(base_size = 7) +
        theme(
            axis.title.x = element_text(size = 6.5, margin = margin(t = 1)),
            axis.text.x  = element_text(size = 5.5),
            axis.text.y  = element_blank(),
            axis.ticks.y = element_blank(),
            axis.line.y  = element_blank(),
            axis.line.x  = element_line(color = "#444444", linewidth = 0.4),
            panel.grid   = element_blank(),
            plot.margin  = margin(1, 2, 1, 2)
        )

    p_right <- ggplot(df_s) +
        annotate("segment", x = 0, xend = right_em,
                y = divider_y, yend = divider_y,
                color = "#CCCCCC", linewidth = 0.4, linetype = "dashed") +
        geom_text(aes(x = x_padj_pos, y = y, label = p_str,
                     fontface = ifelse(is_sig, "bold", "plain"),
                     color    = ifelse(is_sig, "#222222", "#666666")),
                 hjust = 0.5, size = 2.2, family = "mono") +
        geom_text(aes(x = x_sig_pos, y = y, label = sig),
                 fontface = "bold", hjust = 0.5, size = 2.4,
                 family = "mono", color = "#222222") +
        annotate("text", x = x_padj_pos, y = header_y, label = h_padj,
                fontface = "bold", hjust = 0.5, size = 2.4, color = "#222222") +
        annotate("text", x = x_sig_pos, y = header_y, label = h_sig,
                fontface = "bold", hjust = 0.5, size = 2.4, color = "#222222") +
        annotate("segment", x = 0, xend = right_em,
                y = underline_y, yend = underline_y,
                color = "#333333", linewidth = 0.3) +
        scale_color_identity() +
        scale_x_continuous(limits = c(0, right_em), expand = c(0, 0)) +
        scale_y_continuous(limits = y_lim, expand = c(0, 0)) +
        theme_void() +
        theme(plot.margin = margin(2, 4, 2, 2))

    composed <- (p_left | p_forest | p_right) +
        plot_layout(widths = c(left_em, forest_em, right_em))

    fig_height <- max(FOREST_LAYOUT$min_fig_height_in,
                     FOREST_LAYOUT$base_height_in +
                         n_rows * FOREST_LAYOUT$row_height_in)

    slug <- sprintf("%s_modulescores_lmm_%s_forest", CELL_TYPE, s)

    save_figure(composed, slug = slug, width = fig_width, height = fig_height)

    options(repr.plot.width = fig_width + 1, repr.plot.height = fig_height + 0.5)
    tryCatch(
        print(composed),
        error = function(e) {
            cat(sprintf("    [inline preview unavailable: %s]\n",
                        conditionMessage(e)))
        }
    )

    cat(sprintf("    fig dims: %.2f x %.2f in (left_em=%.1f, forest_em=%.1f, right_em=%.1f)\n",
                fig_width, fig_height, left_em, forest_em, right_em))
}


# ─────────────────────────────────────────────────────────────────────────────
# §5.4.6 SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
n_sig_rows  <- sum(mod_lmm_df$p_adj < STATISTICAL_PARAMS$fdr_threshold,
                  na.rm = TRUE)
n_converged <- sum(mod_lmm_df$converged, na.rm = TRUE)

cat("\n", strrep("-", 72), "\n", sep = "")
cat(sprintf("  §5.4 SUMMARY  |  %s / %s\n", DATASET, CELL_TYPE))
cat(strrep("-", 72), "\n", sep = "")
cat(sprintf("  Test          : LMM (Gaussian, lmer + lmerTest)\n"))
cat(sprintf("  beta_scale    : score_units\n"))
cat(sprintf("  Adjustments   : Sex + Cohort + log10(nCount_RNA)\n"))
cat(sprintf("  Random effect : (1 | donor)\n"))
cat(sprintf("  Inference     : Satterthwaite-approximated t-test\n"))
cat(sprintf("  Converged     : %d / %d\n", n_converged, nrow(mod_lmm_df)))
cat(sprintf("  Significant   : %d of %d at BH-FDR < %.2f\n",
            n_sig_rows, nrow(mod_lmm_df), STATISTICAL_PARAMS$fdr_threshold))
cat(sprintf("  Mean ICC      : %.3f  (donor variance / total variance)\n",
            mean(mod_lmm_df$icc, na.rm = TRUE)))
cat(sprintf("  Total fit time: %s\n",
            fmt_elapsed(sum(mod_lmm_df$fit_seconds, na.rm = TRUE))))

if (n_sig_rows > 0) {
    cat(sprintf("\n  Significant modules (beta > 0 = SnC enriched):\n"))
    sig_rows <- mod_lmm_df %>%
        filter(p_adj < STATISTICAL_PARAMS$fdr_threshold) %>%
        arrange(stratum, desc(estimate))
    for (i in seq_len(nrow(sig_rows))) {
        r <- sig_rows[i, ]
        direction <- ifelse(r$estimate > 0, "+", "-")
        cat(sprintf("    %s %-22s [%s]  beta=%+.4f  p_adj=%.2e\n",
                    direction, r$outcome, r$stratum,
                    r$estimate, r$p_adj))
    }
}

cat(sprintf("\n  Per-stratum forests:\n"))
for (s in strata) {
    cat(sprintf("    %-22s -> %s_modulescores_lmm_%s_forest.{pdf,png,svg}\n",
                s, CELL_TYPE, s))
}
cat(strrep("-", 72), "\n", sep = "")
cat("\nv §5.4 module scores LMM complete\n")
cat("  Next: §5.5 -- Balanced RLM bootstrap (cell-level)\n")

---
## 18 · Module scores — balanced RLM bootstrap

**Why.** Arm-size balancing for the module scores. Same construction as section 12: resample within donor, collapse to a paired difference, refit robustly, 100 times.

**Method.** Balanced cell-level bootstrap, `lmrob` per iteration (100 iterations).  
**Outcome.** Per-donor `mean_SnC − mean_NonSnC`.  
**Formula.** `diff ~ Sex + Cohort + delta_log10_umi`  
**β.** Median of the intercepts.  
**CI.** Percentile interval.  
**p.** `2 × min(P(β > 0), P(β < 0))`.  
**FDR.** BH within stratum, across 10 modules.  

**Display.** Per-stratum forest, matching sections 15-17.

In [ ]:
# MODULE 05 -- Senescence Enrichment & Cell Cycle Analysis
# §5.5 -- MODULE SCORES: BALANCED RLM BOOTSTRAP (cell-level resampling)
# Stats:
#   Method:     Cell-level balanced bootstrap, lmrob fit per iteration
#   Outcome:    per-donor paired difference (mean_SnC - mean_NonSnC)
#   Per-iter:   For each donor, resample min(n_SnC, n_NonSnC) cells per arm
#               (with replacement), recompute donor mean module score per arm,
#               then fit lmrob(diff ~ Sex + Cohort + delta_log10_umi).
#   beta:       median(intercepts) across 100 iterations
#   beta_scale: score_units
#   CI:         2.5% and 97.5% percentiles of bootstrap intercept distribution
#   p-value:    2 * min(P(beta_iter > 0), P(beta_iter < 0))
#   FDR:        BH within (stratum) across 10 modules
#
# This is the cell-level resampling analog of §5.2 RLM. By resampling at
# the cell level, the bootstrap captures within-donor variability that
# point-estimate RLM cannot. Outlier-resistant via robust fit per iteration.
#
# Modules in fixed canonical order from SLOAN_HALLMARK_NAMES.
# Figure: per-stratum forest plot, fully adaptive layout (matches §5.2-§5.4).

cat("=", strrep("=", 71), "\n", sep = "")
cat(sprintf("§5.5 -- MODULE SCORES: BALANCED RLM BOOTSTRAP  |  %s / %s\n",
            DATASET, CELL_TYPE))
cat("=", strrep("=", 71), "\n", sep = "")


# ─────────────────────────────────────────────────────────────────────────────
# §5.5.1 STATS
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> STATS\n")
cat("  Method   : Cell-level balanced bootstrap + lmrob per iteration\n")
cat("  Outcome  : prop_SnC mean - prop_NonSnC mean module score (per donor)\n")
cat("  Per-iter : Resample min(n_SnC, n_NonSnC) cells/arm/donor with replacement\n")
cat("  Formula  : diff ~ Sex + Cohort + delta_log10_umi  (lmrob, KS2014)\n")
cat("  beta     : median of bootstrap intercepts\n")
cat("  CI       : 2.5%/97.5% percentiles\n")
cat("  p-value  : 2 * min(P(beta > 0), P(beta < 0))\n")
cat(sprintf("  N iter   : %d (seed=%d)\n",
            STATISTICAL_PARAMS$bootstrap_n_iter,
            STATISTICAL_PARAMS$bootstrap_seed))
cat(sprintf("  Strata   : %s\n", paste(strata, collapse = ", ")))
cat(sprintf("  FDR      : %s within (stratum) across %d modules\n",
            STATISTICAL_PARAMS$fdr_method,
            length(SLOAN_HALLMARK_NAMES)))


# ─────────────────────────────────────────────────────────────────────────────
# §5.5.2 RUN
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> RUN\n")
cat(sprintf("  Note: %d iter * %d (stratum x module) = %d lmrob fits. ~1-3 min.\n",
            STATISTICAL_PARAMS$bootstrap_n_iter,
            length(strata) * length(SLOAN_HALLMARK_NAMES),
            STATISTICAL_PARAMS$bootstrap_n_iter *
                length(strata) * length(SLOAN_HALLMARK_NAMES)))

# Score column lookup -- assumes §5.1-§5.4 has run
if (!exists("score_col_lookup")) {
    md_cols <- colnames(md)
    score_col_lookup <- character(length(SLOAN_HALLMARK_NAMES))
    names(score_col_lookup) <- SLOAN_HALLMARK_NAMES
    for (mod in SLOAN_HALLMARK_NAMES) {
        candidates <- c(paste0("Score_", mod), paste0("score_", mod),
                       paste0("module_", mod), mod, paste0(mod, "1"))
        found <- intersect(candidates, md_cols)
        if (length(found) > 0) score_col_lookup[mod] <- found[1]
        else                   score_col_lookup[mod] <- NA_character_
    }
}

mod_boot_rows <- list()

for (stratum in strata) {
    cat(sprintf("\n  [%s]\n", stratum))

    md_s        <- filter_to_stratum(md, stratum, STUDY_GROUP_COL)
    arms_s      <- build_donor_arms(md_s, DONOR_COL, SEN_LABEL_COL,
                                    STATISTICAL_PARAMS$min_cells_per_group)
    paired_ids  <- unique(arms_s$donor[arms_s$paired])
    n_paired    <- length(paired_ids)

    if (n_paired < 5) {
        cat(sprintf("    only %d paired donors -- skipping\n", n_paired))
        next
    }

    donor_meta_s <- build_donor_meta(
        md_s, paired_ids,
        DONOR_COL, SEN_LABEL_COL,
        SEX_COL, AGE_COL, COHORT_COL,
        has_cohort = HAS_COHORT
    )

    md_pair <- md_s[md_s[[DONOR_COL]] %in% paired_ids, , drop = FALSE]
    md_pair$is_snc_arm <- as.character(md_pair[[SEN_LABEL_COL]]) %in%
                          c("1", "TRUE", "True", "Senescent")

    sex_levels    <- length(unique(donor_meta_s[[SEX_COL]]))
    cohort_levels <- if (HAS_COHORT) length(unique(donor_meta_s[[COHORT_COL]])) else 0

    rhs <- character(0)
    if (sex_levels    >= 2) rhs <- c(rhs, SEX_COL)
    if (cohort_levels >= 2) rhs <- c(rhs, COHORT_COL)
    rhs <- c(rhs, "delta_log10_umi")

    formula_str <- paste("diff ~", paste(rhs, collapse = " + "))
    formula_obj <- as.formula(formula_str)

    # Pre-compute per-donor cell index lists for fast resampling
    cat(sprintf("    Pre-indexing cells: %d donors\n", n_paired))
    donor_idx <- list()
    for (donor_id in paired_ids) {
        d_rows <- which(md_pair[[DONOR_COL]] == donor_id)
        donor_idx[[as.character(donor_id)]] <- list(
            snc_idx    = d_rows[md_pair$is_snc_arm[d_rows]],
            nonsnc_idx = d_rows[!md_pair$is_snc_arm[d_rows]]
        )
    }

    cell_cnt_snc <- sum(arms_s$n_cells[arms_s$donor %in% paired_ids &
                                       arms_s$arm %in% c("1", "TRUE", "True", "Senescent")])
    cell_cnt_non <- sum(arms_s$n_cells[arms_s$donor %in% paired_ids &
                                       !(arms_s$arm %in% c("1", "TRUE", "True", "Senescent"))])

    for (mod in SLOAN_HALLMARK_NAMES) {
        score_col <- score_col_lookup[mod]
        if (is.na(score_col)) {
            cat(sprintf("    [%s | %s] no score col -- skipping\n", stratum, mod))
            next
        }

        cat(sprintf("    [%s | %-22s] bootstrapping ...", stratum, mod))
        t0 <- Sys.time()

        # Pre-extract score values for this module across all paired cells
        cell_scores <- md_pair[[score_col]]

        set.seed(STATISTICAL_PARAMS$bootstrap_seed)

        boot_intercepts <- numeric(0)
        n_failed        <- 0L

        for (b in seq_len(STATISTICAL_PARAMS$bootstrap_n_iter)) {
            donor_diffs <- numeric(n_paired)
            names(donor_diffs) <- paired_ids

            for (i in seq_along(paired_ids)) {
                donor_id <- paired_ids[i]
                idx_set  <- donor_idx[[as.character(donor_id)]]

                n_snc    <- length(idx_set$snc_idx)
                n_non    <- length(idx_set$nonsnc_idx)
                n_match  <- min(n_snc, n_non)

                snc_b    <- sample(idx_set$snc_idx,    size = n_match, replace = TRUE)
                non_b    <- sample(idx_set$nonsnc_idx, size = n_match, replace = TRUE)

                mean_snc <- mean(cell_scores[snc_b], na.rm = TRUE)
                mean_non <- mean(cell_scores[non_b], na.rm = TRUE)

                donor_diffs[i] <- mean_snc - mean_non
            }

            reg_df <- data.frame(donor = paired_ids, diff = donor_diffs,
                                stringsAsFactors = FALSE) %>%
                left_join(donor_meta_s, by = "donor")

            fit <- tryCatch({
                robustbase::lmrob(formula_obj, data = reg_df, setting = "KS2014")
            }, error   = function(e) NULL,
               warning = function(w) {
                tryCatch(robustbase::lmrob(formula_obj, data = reg_df,
                                           setting = "KS2014"),
                        error = function(e) NULL)
            })

            if (is.null(fit)) {
                n_failed <- n_failed + 1L
                next
            }

            co <- summary(fit)$coefficients
            if (!"(Intercept)" %in% rownames(co)) {
                n_failed <- n_failed + 1L
                next
            }

            boot_intercepts <- c(boot_intercepts, co["(Intercept)", "Estimate"])
        }

        elapsed     <- as.numeric(difftime(Sys.time(), t0, units = "secs"))
        n_converged <- length(boot_intercepts)

        if (n_converged < 10) {
            cat(sprintf(" only %d/%d converged after %s -- skipping\n",
                        n_converged, STATISTICAL_PARAMS$bootstrap_n_iter,
                        fmt_elapsed(elapsed)))
            next
        }
        cat(sprintf(" %d/%d converged in %s\n",
                    n_converged, STATISTICAL_PARAMS$bootstrap_n_iter,
                    fmt_elapsed(elapsed)))

        # Summary statistics across bootstrap distribution
        beta_med <- median(boot_intercepts)
        beta_se  <- sd(boot_intercepts)
        ci       <- quantile(boot_intercepts,
                            probs = c((1 - STATISTICAL_PARAMS$confidence_level) / 2,
                                      1 - (1 - STATISTICAL_PARAMS$confidence_level) / 2),
                            na.rm = TRUE)

        # Two-sided bootstrap p-value
        p_above <- mean(boot_intercepts > 0)
        p_below <- mean(boot_intercepts < 0)
        p_value <- min(2 * min(p_above, p_below), 1)
        p_value <- max(p_value, 1 / STATISTICAL_PARAMS$bootstrap_n_iter)

        row <- tidy_model_results(
            stratum      = stratum,
            outcome      = mod,
            model        = "rlm_bootstrap",
            n_donors     = n_paired,
            n_cells_test = cell_cnt_snc,
            n_cells_ref  = cell_cnt_non,
            estimate     = beta_med,
            se           = beta_se,
            ci_low       = unname(ci[1]),
            ci_high      = unname(ci[2]),
            statistic    = NA_real_,
            p_value      = p_value,
            extra        = list(
                beta_scale         = "score_units",
                module_type        = SLOAN_LIST_TYPES[which(SLOAN_HALLMARK_NAMES == mod)],
                score_col          = score_col,
                formula            = formula_str,
                cohort_in_model    = cohort_levels >= 2,
                sex_in_model       = sex_levels >= 2,
                delta_umi_in_model = TRUE,
                n_iterations       = STATISTICAL_PARAMS$bootstrap_n_iter,
                n_converged        = n_converged,
                n_failed           = n_failed,
                bootstrap_seed     = STATISTICAL_PARAMS$bootstrap_seed,
                fit_seconds        = elapsed
            )
        )
        mod_boot_rows[[paste(stratum, mod, sep = "|")]] <- row
    }

    rm(donor_idx, md_pair); gc(verbose = FALSE)
}

mod_boot_df <- bind_rows(mod_boot_rows)


# ─────────────────────────────────────────────────────────────────────────────
# §5.5.3 BH-FDR within (stratum)
# ─────────────────────────────────────────────────────────────────────────────
mod_boot_df <- mod_boot_df %>%
    group_by(stratum) %>%
    mutate(p_adj = p.adjust(p_value, method = STATISTICAL_PARAMS$fdr_method)) %>%
    ungroup() %>%
    mutate(sig = sig_stars(p_adj))


# ─────────────────────────────────────────────────────────────────────────────
# §5.5.4 RESULTS
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> RESULTS  (beta = median of bootstrap intercepts, scale = score_units)\n")

cat(sprintf("\n  %-22s %-22s %5s %+10s %+10s %+10s %5s %8s %8s %4s\n",
            "Stratum", "Module", "N",
            "beta", "CI_low", "CI_high",
            "Conv.",
            "p_raw", "p_adj", "Sig"))
cat("  ", strrep("-", 115), "\n", sep = "")

for (s in strata) {
    df_s <- mod_boot_df %>% filter(stratum == s)
    df_s <- df_s[match(SLOAN_HALLMARK_NAMES, df_s$outcome), ]
    df_s <- df_s[!is.na(df_s$stratum), ]

    for (i in seq_len(nrow(df_s))) {
        r <- df_s[i, ]
        cat(sprintf("  %-22s %-22s %5d %+9.4f %+9.4f %+9.4f %4d/%d %8.1e %8.1e %4s\n",
                    substr(r$stratum, 1, 22), substr(r$outcome, 1, 22),
                    r$n_donors,
                    r$estimate, r$ci_low, r$ci_high,
                    r$n_converged, r$n_iterations,
                    r$p_value, r$p_adj, r$sig))
    }
}

save_table(mod_boot_df, paste0(CELL_TYPE, "_modulescores_bootrlm"))


# ─────────────────────────────────────────────────────────────────────────────
# §5.5.5 FIGURE -- per-stratum forest plot, FULLY ADAPTIVE layout
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> FIGURE\n")

if (!exists("FOREST_LAYOUT")) {
    FOREST_LAYOUT <- list(
        em_per_char_mono   = 0.62,
        em_per_char_prop   = 0.55,
        inter_col_gap      = 1.5,
        panel_right_pad    = 0.8,
        forest_share_mult  = 0.7,
        inches_per_em      = 0.020,
        min_fig_width_in   = 6.0,
        min_fig_height_in  = 2.5,
        row_height_in      = 0.30,
        base_height_in     = 0.9,
        pad_x_axis_frac    = 0.10,
        pad_x_axis_null    = 0.02
    )
}

cfg <- EFFECT_CONFIGS$score_units

divider_after_idx <- sum(SLOAN_LIST_TYPES == "Individual hallmark")  # = 8

for (s in strata) {
    df_s <- mod_boot_df %>%
        filter(stratum == s) %>%
        mutate(outcome = factor(outcome, levels = SLOAN_HALLMARK_NAMES)) %>%
        arrange(outcome)

    if (nrow(df_s) == 0) {
        cat(sprintf("  [%s] no rows -- skipping figure\n", s))
        next
    }

    df_s$row_label  <- as.character(df_s$outcome)
    df_s$eff_str    <- sapply(df_s$estimate, cfg$fmt_effect)
    df_s$ci_str     <- mapply(cfg$fmt_ci, df_s$ci_low, df_s$ci_high)
    df_s$p_str      <- sapply(df_s$p_adj, fmt_p_short)
    df_s$row_color  <- SLOAN_HALLMARK_COLORS[as.character(df_s$outcome)]
    df_s$is_sig     <- df_s$sig != "ns"

    n_rows <- nrow(df_s)

    h_mod  <- "Module"
    h_eff  <- cfg$eff_h_label
    h_ci   <- cfg$ci_h_label
    h_padj <- "p(adj)"
    h_sig  <- "Sig"

    nchar_label <- max(nchar(c(df_s$row_label, h_mod)),  na.rm = TRUE)
    nchar_eff   <- max(nchar(c(df_s$eff_str,   h_eff)),  na.rm = TRUE)
    nchar_ci    <- max(nchar(c(df_s$ci_str,    h_ci)),   na.rm = TRUE)
    nchar_padj  <- max(nchar(c(df_s$p_str,     h_padj)), na.rm = TRUE)
    nchar_sig   <- max(nchar(c(df_s$sig,       h_sig)),  na.rm = TRUE)

    em_label <- nchar_label * FOREST_LAYOUT$em_per_char_prop
    em_eff   <- nchar_eff   * FOREST_LAYOUT$em_per_char_mono
    em_ci    <- nchar_ci    * FOREST_LAYOUT$em_per_char_mono
    em_padj  <- nchar_padj  * FOREST_LAYOUT$em_per_char_mono
    em_sig   <- nchar_sig   * FOREST_LAYOUT$em_per_char_mono

    gap <- FOREST_LAYOUT$inter_col_gap

    x_label_pos <- 0
    x_eff_pos   <- em_label + gap
    x_ci_pos    <- em_label + gap + em_eff + gap
    left_em     <- em_label + gap + em_eff + gap + em_ci + FOREST_LAYOUT$panel_right_pad

    x_padj_pos  <- em_padj / 2
    x_sig_pos   <- em_padj + gap + em_sig / 2
    right_em    <- em_padj + gap + em_sig + FOREST_LAYOUT$panel_right_pad

    forest_em <- (left_em + right_em) * FOREST_LAYOUT$forest_share_mult
    total_em  <- left_em + forest_em + right_em
    fig_width <- max(total_em * FOREST_LAYOUT$inches_per_em,
                    FOREST_LAYOUT$min_fig_width_in)

    df_s$y      <- n_rows:1
    header_y    <- n_rows + 0.55
    underline_y <- n_rows + 0.20
    y_lim       <- c(0.4, n_rows + 1.0)
    divider_y   <- (n_rows - divider_after_idx) + 0.5

    stratum_color <- if (s %in% names(STUDY_GROUP_COLORS)) STUDY_GROUP_COLORS[[s]] else "#7F7F7F"

    n_sig <- sum(df_s$is_sig)
    title_txt <- sprintf("Module scores bootRLM | %s | [%s] | %d/%d sig at BH-FDR",
                         CELL_TYPE, s, n_sig, nrow(df_s))

    p_left <- ggplot(df_s) +
        annotate("segment", x = 0, xend = left_em,
                y = divider_y, yend = divider_y,
                color = "#CCCCCC", linewidth = 0.4, linetype = "dashed") +
        geom_text(aes(x = x_label_pos, y = y, label = row_label,
                     color = row_color,
                     fontface = ifelse(is_sig, "bold", "plain")),
                 hjust = 0, size = 2.4) +
        geom_text(aes(x = x_eff_pos, y = y, label = eff_str,
                     fontface = ifelse(is_sig, "bold", "plain")),
                 hjust = 0, size = 2.2, family = "mono", color = "#222222") +
        geom_text(aes(x = x_ci_pos, y = y, label = ci_str),
                 hjust = 0, size = 2.0, family = "mono", color = "#666666") +
        annotate("text", x = x_label_pos, y = header_y, label = h_mod,
                fontface = "bold", hjust = 0, size = 2.4, color = "#222222") +
        annotate("text", x = x_eff_pos, y = header_y, label = h_eff,
                fontface = "bold", hjust = 0, size = 2.4, color = "#222222") +
        annotate("text", x = x_ci_pos, y = header_y, label = h_ci,
                fontface = "bold", hjust = 0, size = 2.2, color = "#222222") +
        annotate("segment", x = 0, xend = left_em,
                y = underline_y, yend = underline_y,
                color = "#333333", linewidth = 0.3) +
        scale_color_identity() +
        scale_x_continuous(limits = c(0, left_em), expand = c(0, 0)) +
        scale_y_continuous(limits = y_lim, expand = c(0, 0)) +
        labs(title = title_txt) +
        theme_void() +
        theme(
            plot.title = element_text(size = 9, face = "bold",
                                     color = stratum_color,
                                     hjust = 0,
                                     margin = margin(t = 2, b = 1)),
            plot.margin = margin(2, 2, 2, 4)
        )

    eff_vals    <- c(df_s$estimate, df_s$ci_low, df_s$ci_high)
    eff_finite  <- eff_vals[is.finite(eff_vals)]
    if (length(eff_finite) == 0) eff_finite <- c(-0.001, 0.001)

    data_range  <- diff(range(eff_finite))
    if (data_range == 0) data_range <- max(abs(eff_finite[1]), 1e-6) * 0.2

    pad         <- data_range * FOREST_LAYOUT$pad_x_axis_frac
    null_pad    <- data_range * FOREST_LAYOUT$pad_x_axis_null
    x_range     <- range(eff_finite) + c(-pad, pad)
    x_range[1]  <- min(x_range[1], cfg$null_value - null_pad)
    x_range[2]  <- max(x_range[2], cfg$null_value + null_pad)

    p_forest <- ggplot(df_s) +
        annotate("segment", x = -Inf, xend = Inf,
                y = divider_y, yend = divider_y,
                color = "#CCCCCC", linewidth = 0.4, linetype = "dashed") +
        geom_vline(xintercept = cfg$null_value,
                  linetype = "dashed", color = "#999999", linewidth = 0.4) +
        geom_errorbar(aes(y = y, xmin = ci_low, xmax = ci_high),
                     width = 0.18, linewidth = 0.4, color = "#4D4D4D") +
        geom_point(aes(x = estimate, y = y, fill = row_color,
                      size = ifelse(is_sig, 3.5, 2.5)),
                  shape = 23, color = "#222222", stroke = 0.4) +
        scale_fill_identity() +
        scale_size_identity() +
        scale_x_continuous(limits = x_range,
                          labels = cfg$axis_format,
                          breaks = scales::breaks_pretty(n = 4)) +
        scale_y_continuous(limits = y_lim, expand = c(0, 0)) +
        labs(x = cfg$x_label, y = NULL) +
        theme_classic(base_size = 7) +
        theme(
            axis.title.x = element_text(size = 6.5, margin = margin(t = 1)),
            axis.text.x  = element_text(size = 5.5),
            axis.text.y  = element_blank(),
            axis.ticks.y = element_blank(),
            axis.line.y  = element_blank(),
            axis.line.x  = element_line(color = "#444444", linewidth = 0.4),
            panel.grid   = element_blank(),
            plot.margin  = margin(1, 2, 1, 2)
        )

    p_right <- ggplot(df_s) +
        annotate("segment", x = 0, xend = right_em,
                y = divider_y, yend = divider_y,
                color = "#CCCCCC", linewidth = 0.4, linetype = "dashed") +
        geom_text(aes(x = x_padj_pos, y = y, label = p_str,
                     fontface = ifelse(is_sig, "bold", "plain"),
                     color    = ifelse(is_sig, "#222222", "#666666")),
                 hjust = 0.5, size = 2.2, family = "mono") +
        geom_text(aes(x = x_sig_pos, y = y, label = sig),
                 fontface = "bold", hjust = 0.5, size = 2.4,
                 family = "mono", color = "#222222") +
        annotate("text", x = x_padj_pos, y = header_y, label = h_padj,
                fontface = "bold", hjust = 0.5, size = 2.4, color = "#222222") +
        annotate("text", x = x_sig_pos, y = header_y, label = h_sig,
                fontface = "bold", hjust = 0.5, size = 2.4, color = "#222222") +
        annotate("segment", x = 0, xend = right_em,
                y = underline_y, yend = underline_y,
                color = "#333333", linewidth = 0.3) +
        scale_color_identity() +
        scale_x_continuous(limits = c(0, right_em), expand = c(0, 0)) +
        scale_y_continuous(limits = y_lim, expand = c(0, 0)) +
        theme_void() +
        theme(plot.margin = margin(2, 4, 2, 2))

    composed <- (p_left | p_forest | p_right) +
        plot_layout(widths = c(left_em, forest_em, right_em))

    fig_height <- max(FOREST_LAYOUT$min_fig_height_in,
                     FOREST_LAYOUT$base_height_in +
                         n_rows * FOREST_LAYOUT$row_height_in)

    slug <- sprintf("%s_modulescores_bootrlm_%s_forest", CELL_TYPE, s)

    save_figure(composed, slug = slug, width = fig_width, height = fig_height)

    options(repr.plot.width = fig_width + 1, repr.plot.height = fig_height + 0.5)
    tryCatch(
        print(composed),
        error = function(e) {
            cat(sprintf("    [inline preview unavailable: %s]\n",
                        conditionMessage(e)))
        }
    )

    cat(sprintf("    fig dims: %.2f x %.2f in (left_em=%.1f, forest_em=%.1f, right_em=%.1f)\n",
                fig_width, fig_height, left_em, forest_em, right_em))
}


# ─────────────────────────────────────────────────────────────────────────────
# §5.5.6 SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
n_sig_rows  <- sum(mod_boot_df$p_adj < STATISTICAL_PARAMS$fdr_threshold,
                  na.rm = TRUE)
mean_conv   <- mean(mod_boot_df$n_converged / mod_boot_df$n_iterations,
                   na.rm = TRUE) * 100

cat("\n", strrep("-", 72), "\n", sep = "")
cat(sprintf("  §5.5 SUMMARY  |  %s / %s\n", DATASET, CELL_TYPE))
cat(strrep("-", 72), "\n", sep = "")
cat(sprintf("  Method        : Cell-level balanced bootstrap + lmrob (KS2014)\n"))
cat(sprintf("  beta_scale    : score_units\n"))
cat(sprintf("  Adjustments   : Sex + Cohort + delta_log10_umi\n"))
cat(sprintf("  Iterations    : %d (seed=%d)\n",
            STATISTICAL_PARAMS$bootstrap_n_iter,
            STATISTICAL_PARAMS$bootstrap_seed))
cat(sprintf("  Mean conv.    : %.1f%%\n", mean_conv))
cat(sprintf("  Significant   : %d of %d at BH-FDR < %.2f\n",
            n_sig_rows, nrow(mod_boot_df), STATISTICAL_PARAMS$fdr_threshold))
cat(sprintf("  Total fit time: %s\n",
            fmt_elapsed(sum(mod_boot_df$fit_seconds, na.rm = TRUE))))

if (n_sig_rows > 0) {
    cat(sprintf("\n  Significant modules (beta > 0 = SnC enriched):\n"))
    sig_rows <- mod_boot_df %>%
        filter(p_adj < STATISTICAL_PARAMS$fdr_threshold) %>%
        arrange(stratum, desc(estimate))
    for (i in seq_len(nrow(sig_rows))) {
        r <- sig_rows[i, ]
        direction <- ifelse(r$estimate > 0, "+", "-")
        cat(sprintf("    %s %-22s [%s]  beta=%+.4f  p_adj=%.2e\n",
                    direction, r$outcome, r$stratum,
                    r$estimate, r$p_adj))
    }
}

cat(sprintf("\n  Per-stratum forests:\n"))
for (s in strata) {
    cat(sprintf("    %-22s -> %s_modulescores_bootrlm_%s_forest.{pdf,png,svg}\n",
                s, CELL_TYPE, s))
}
cat(strrep("-", 72), "\n", sep = "")
cat("\nv §5.5 module scores balanced RLM bootstrap complete\n")
cat("  Next: §5.6 -- Cross-model agreement (5-model comparison)\n")

---
## 19 · Module scores — cross-model agreement

**Why.** The synthesis for the hallmark panels. All five estimators are on the same score-unit scale here, so unlike section 13 the βs can be compared directly, not just their signs.

150 rows in, 30 out: one per stratum × module.

**Inputs.** `mod_wilcox_df`, `mod_rlm_df`, `mod_ols_df`, `mod_lmm_df`, `mod_boot_df`.  
**Scale.** All five on `score_units` — direct β comparison.  
**Reports.** direction agreement, `n_models_sig`, `all_sig`, consensus β.  

**Display.** Transposed heatmap — rows = 10 modules in canonical order, columns = 5 models, faceted by stratum, cell text = β with `*` for significant.

In [ ]:
# MODULE 05 -- Senescence Enrichment & Cell Cycle Analysis
# §5.6 -- MODULE SCORES: CROSS-MODEL AGREEMENT
# Synthesizes results across the 5 module-score models from §5.1-§5.5:
#   §5.1 wilcoxon         (donor-level paired Wilcoxon)
#   §5.2 rlm              (donor-level RLM, lmrob KS2014)
#   §5.3 ols              (donor-level OLS)
#   §5.4 lmm              (cell-level Gaussian LMM, lmer + Satterthwaite)
#   §5.5 rlm_bootstrap    (cell-level balanced bootstrap + lmrob)
#
# All 5 use beta_scale = "score_units" (continuous module score difference).
# Direct β comparison possible across models.
#
# Outputs:
#   - Long-form combined table (150 rows: 5 models x 10 modules x 3 strata)
#   - Wide summary table (30 rows: one per stratum x module)
#   - Heatmap figure (TRANSPOSED layout):
#       Rows = 10 modules in canonical order (hallmark/composite divider)
#       Cols = 5 models with short labels (Wilcox, RLM, OLS, BootRLM, LMM)
#       Faceted by stratum (3 panels horizontally side-by-side)
#       Cell text = beta value with * suffix for significant cells

cat("=", strrep("=", 71), "\n", sep = "")
cat(sprintf("§5.6 -- MODULE SCORES CROSS-MODEL AGREEMENT  |  %s / %s\n",
            DATASET, CELL_TYPE))
cat("=", strrep("=", 71), "\n", sep = "")


# ─────────────────────────────────────────────────────────────────────────────
# §5.6.1 PRECONDITION CHECK
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> PRECONDITION\n")

required_dfs <- list(
    wilcoxon       = "mod_wilcox_df",
    rlm            = "mod_rlm_df",
    ols            = "mod_ols_df",
    lmm            = "mod_lmm_df",
    rlm_bootstrap  = "mod_boot_df"
)

missing <- character(0)
for (model_name in names(required_dfs)) {
    df_name <- required_dfs[[model_name]]
    if (!exists(df_name) || is.null(get(df_name)) || nrow(get(df_name)) == 0) {
        missing <- c(missing, df_name)
    } else {
        cat(sprintf("  %-15s -> %s (%d rows)\n",
                    model_name, df_name, nrow(get(df_name))))
    }
}

if (length(missing) > 0) {
    stop(sprintf("✗ Missing model dataframes: %s\n  Re-run §5.1-§5.5 before §5.6.",
                 paste(missing, collapse = ", ")))
}


# ─────────────────────────────────────────────────────────────────────────────
# §5.6.2 BUILD LONG-FORM COMBINED TABLE
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> BUILD COMBINED TABLE\n")

keep_cols <- c("stratum", "outcome", "model",
              "n_donors", "estimate", "se", "ci_low", "ci_high",
              "p_value", "p_adj", "sig", "beta_scale")

extract_cols <- function(df, model_label) {
    out <- df %>% select(any_of(keep_cols))
    out$model <- model_label
    if (!"beta_scale" %in% colnames(out)) out$beta_scale <- NA_character_
    out
}

mod_crossmodel_long <- bind_rows(
    extract_cols(mod_wilcox_df, "wilcoxon"),
    extract_cols(mod_rlm_df,    "rlm"),
    extract_cols(mod_ols_df,    "ols"),
    extract_cols(mod_lmm_df,    "lmm"),
    extract_cols(mod_boot_df,   "rlm_bootstrap")
)

# Short display labels for models (column headers in heatmap)
MODEL_DISPLAY_LABELS <- c(
    "wilcoxon"      = "Wilcox",
    "rlm"           = "RLM",
    "ols"           = "OLS",
    "rlm_bootstrap" = "BootRLM",
    "lmm"           = "LMM"
)
mod_crossmodel_long$model_display <- MODEL_DISPLAY_LABELS[mod_crossmodel_long$model]
mod_crossmodel_long$model_display <- factor(
    mod_crossmodel_long$model_display,
    levels = c("Wilcox", "RLM", "OLS", "BootRLM", "LMM")
)

mod_crossmodel_long$model <- factor(
    mod_crossmodel_long$model,
    levels = c("wilcoxon", "rlm", "ols", "rlm_bootstrap", "lmm")
)
mod_crossmodel_long$stratum <- factor(mod_crossmodel_long$stratum, levels = strata)
mod_crossmodel_long$outcome <- factor(mod_crossmodel_long$outcome, levels = SLOAN_HALLMARK_NAMES)

# Direction (all 5 models on score_units scale)
mod_crossmodel_long$direction <- sign(mod_crossmodel_long$estimate)

# Sign-x-sig category for heatmap fill
mod_crossmodel_long$direction_label <- with(mod_crossmodel_long, {
    sig_flag <- !is.na(p_adj) & p_adj < STATISTICAL_PARAMS$fdr_threshold
    case_when(
        is.na(direction) | is.na(p_adj)   ~ "ns",
        direction > 0 &  sig_flag          ~ "Up (sig)",
        direction > 0 & !sig_flag          ~ "Up (ns)",
        direction < 0 &  sig_flag          ~ "Down (sig)",
        direction < 0 & !sig_flag          ~ "Down (ns)",
        TRUE                                ~ "ns"
    )
})
mod_crossmodel_long$direction_label <- factor(
    mod_crossmodel_long$direction_label,
    levels = c("Up (sig)", "Up (ns)", "ns", "Down (ns)", "Down (sig)")
)

cat(sprintf("  Long table: %d rows (5 models x %d modules x %d strata)\n",
            nrow(mod_crossmodel_long), length(SLOAN_HALLMARK_NAMES), length(strata)))

save_table(mod_crossmodel_long, paste0(CELL_TYPE, "_modulescores_crossmodel_long"))


# ─────────────────────────────────────────────────────────────────────────────
# §5.6.3 BUILD WIDE SUMMARY (one row per stratum x module)
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> BUILD SUMMARY TABLE\n")

mod_crossmodel_wide <- mod_crossmodel_long %>%
    group_by(stratum, outcome) %>%
    summarise(
        n_donors           = first(n_donors),
        n_models           = n(),
        n_pos              = sum(direction > 0, na.rm = TRUE),
        n_neg              = sum(direction < 0, na.rm = TRUE),
        n_zero             = sum(direction == 0 | is.na(direction)),
        majority_direction = case_when(
            n_pos > n_neg ~ "+",
            n_neg > n_pos ~ "-",
            TRUE          ~ "0"
        ),
        n_majority         = pmax(n_pos, n_neg),
        n_models_sig       = sum(!is.na(p_adj) & p_adj < STATISTICAL_PARAMS$fdr_threshold,
                                na.rm = TRUE),
        any_sig            = n_models_sig > 0,
        all_sig            = n_models_sig == n_models,
        median_beta        = median(estimate, na.rm = TRUE),
        mean_beta          = mean(estimate, na.rm = TRUE),
        beta_lmm           = estimate[model == "lmm"][1],
        beta_rlm           = estimate[model == "rlm"][1],
        beta_boot          = estimate[model == "rlm_bootstrap"][1],
        .groups = "drop"
    )

cat(sprintf("  Wide summary: %d rows (%d strata x %d modules)\n",
            nrow(mod_crossmodel_wide),
            length(strata), length(SLOAN_HALLMARK_NAMES)))

save_table(mod_crossmodel_wide, paste0(CELL_TYPE, "_modulescores_crossmodel_summary"))


# ─────────────────────────────────────────────────────────────────────────────
# §5.6.4 RESULTS TEXT
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> RESULTS\n")

cat(sprintf("\n  %-22s %-22s %5s %4s %+9s %+9s %+9s %+9s %5s %5s\n",
            "Stratum", "Module", "N",
            "Maj.",
            "med_beta", "mean_beta", "beta_LMM", "beta_RLM",
            "n_sig", "all_sig"))
cat("  ", strrep("-", 110), "\n", sep = "")

for (s in strata) {
    df_s <- mod_crossmodel_wide %>% filter(stratum == s)
    df_s <- df_s[match(SLOAN_HALLMARK_NAMES, df_s$outcome), ]
    df_s <- df_s[!is.na(df_s$stratum), ]

    for (i in seq_len(nrow(df_s))) {
        r <- df_s[i, ]
        cat(sprintf("  %-22s %-22s %5d  %s%d  %+8.4f %+8.4f %+8.4f %+8.4f  %2d/%d  %5s\n",
                    substr(as.character(r$stratum), 1, 22),
                    substr(as.character(r$outcome), 1, 22),
                    r$n_donors,
                    r$majority_direction, r$n_majority,
                    r$median_beta, r$mean_beta, r$beta_lmm, r$beta_rlm,
                    r$n_models_sig, r$n_models,
                    ifelse(r$all_sig, "yes", "no")))
    }
}

n_unanimous <- sum(mod_crossmodel_wide$n_majority == mod_crossmodel_wide$n_models,
                  na.rm = TRUE)
n_any_sig   <- sum(mod_crossmodel_wide$any_sig)
n_all_sig   <- sum(mod_crossmodel_wide$all_sig)

cat(sprintf("\n  Unanimous direction across all 5 models: %d / %d (stratum x module)\n",
            n_unanimous, nrow(mod_crossmodel_wide)))
cat(sprintf("  Any-model significance:    %d / %d\n",
            n_any_sig, nrow(mod_crossmodel_wide)))
cat(sprintf("  All-model significance:    %d / %d\n",
            n_all_sig, nrow(mod_crossmodel_wide)))


# ─────────────────────────────────────────────────────────────────────────────
# §5.6.5 FIGURE -- transposed heatmap (modules on rows, models on cols)
#
# Rows: 10 modules in canonical order (hallmarks then composites,
#       with horizontal divider line between row 8 SD_TMC and row 9 SenMayo).
#       Multi-hallmark composite rows shown in muted purple text via y-axis
#       custom theming via SLOAN_HALLMARK_COLORS.
# Cols: 5 short model labels (Wilcox, RLM, OLS, BootRLM, LMM), horizontal,
#       centered. No rotation needed.
# Facet: stratum across columns (3 panels side-by-side).
# Cell text: beta value with * for sig at BH-FDR < threshold.
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> FIGURE\n")

# Cell label: beta value, * suffix if sig
mod_crossmodel_long$cell_label <- sprintf("%+.3f", mod_crossmodel_long$estimate)
sig_flag <- !is.na(mod_crossmodel_long$p_adj) &
            mod_crossmodel_long$p_adj < STATISTICAL_PARAMS$fdr_threshold
mod_crossmodel_long$cell_label[sig_flag] <-
    paste0(mod_crossmodel_long$cell_label[sig_flag], "*")

n_strata  <- length(strata)
n_modules <- nlevels(mod_crossmodel_long$outcome)
n_models  <- nlevels(mod_crossmodel_long$model_display)

# Horizontal divider position between hallmarks (rows 1-8) and composites (9-10)
# In display, modules are reversed via scale_y_discrete(limits = rev(...))
# so divider sits between visual row 8 (SD_TMC) and row 9 (SenMayo).
divider_after_idx <- sum(SLOAN_LIST_TYPES == "Individual hallmark")  # = 8
# y-axis position: top is row 1 (p53_Targets), bottom is row 10 (Fridman_Up)
# divider line at y = 10 - 8 + 0.5 = 2.5 (between SenMayo and SD_TMC visually)
divider_y <- (n_modules - divider_after_idx) + 0.5

# Custom y-axis label colors: hallmarks dark, composites purple
y_axis_colors <- SLOAN_HALLMARK_COLORS[SLOAN_HALLMARK_NAMES]

p_heatmap <- ggplot(mod_crossmodel_long,
                   aes(x = model_display, y = outcome, fill = direction_label)) +
    geom_tile(color = "white", linewidth = 0.4) +
    geom_text(aes(label = cell_label),
             size = 2.2, family = "mono", color = "#222222") +
    geom_hline(yintercept = divider_y, linetype = "dashed",
              color = "#666666", linewidth = 0.4) +
    facet_wrap(~ stratum, nrow = 1, strip.position = "top") +
    scale_fill_manual(values = MODEL_AGREEMENT_COLORS,
                     drop = FALSE,
                     name = "Direction x sig") +
    scale_x_discrete(position = "bottom") +
    scale_y_discrete(limits = rev(SLOAN_HALLMARK_NAMES)) +
    labs(title    = sprintf("Module scores cross-model agreement | %s",
                            CELL_TYPE),
         subtitle = sprintf("5 models x %d modules x %d strata | %d/%d unanimous direction | %d/%d any-model sig (BH-FDR<%.2f)",
                            n_modules, n_strata,
                            n_unanimous, nrow(mod_crossmodel_wide),
                            n_any_sig, nrow(mod_crossmodel_wide),
                            STATISTICAL_PARAMS$fdr_threshold),
         x        = NULL,
         y        = "Module") +
    theme_minimal(base_size = 8) +
    theme(
        plot.title       = element_text(size = 10, face = "bold",
                                       color = "#222222",
                                       margin = margin(b = 1)),
        plot.subtitle    = element_text(size = 7, color = "#444444",
                                       margin = margin(b = 4)),
        axis.title.y     = element_text(size = 7),
        axis.text.x      = element_text(size = 7.5, face = "bold",
                                       angle = 0, hjust = 0.5),
        axis.text.y      = element_text(size = 7,
                                       color = rev(unname(y_axis_colors))),
        strip.text       = element_text(size = 8, face = "bold"),
        strip.background = element_rect(fill = "#F0F0F0", color = "#222222"),
        legend.position  = "bottom",
        legend.title     = element_text(size = 7),
        legend.text      = element_text(size = 6),
        legend.key.size  = unit(0.3, "cm"),
        panel.grid       = element_blank(),
        panel.spacing.x  = unit(0.4, "lines")
    )

# Adaptive figure dimensions: 3 strata side-by-side x 5 model cols
fig_width  <- max(9, 1.5 + n_strata * 2.5)
fig_height <- max(4.5, 0.9 + n_modules * 0.32)
slug <- paste0(CELL_TYPE, "_modulescores_crossmodel_heatmap")

save_figure(p_heatmap, slug = slug, width = fig_width, height = fig_height)

options(repr.plot.width = fig_width + 1, repr.plot.height = fig_height + 0.5)
tryCatch(
    print(p_heatmap),
    error = function(e) {
        cat(sprintf("    [inline preview unavailable: %s]\n",
                    conditionMessage(e)))
    }
)

cat(sprintf("    fig dims: %.2f x %.2f in\n", fig_width, fig_height))


# ─────────────────────────────────────────────────────────────────────────────
# §5.6.6 SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
cat("\n", strrep("-", 72), "\n", sep = "")
cat(sprintf("  §5.6 SUMMARY  |  %s / %s\n", DATASET, CELL_TYPE))
cat(strrep("-", 72), "\n", sep = "")
cat(sprintf("  Models compared    : %d (wilcoxon, rlm, ols, lmm, rlm_bootstrap)\n",
            n_models))
cat(sprintf("  Stratum x module   : %d\n", nrow(mod_crossmodel_wide)))
cat(sprintf("  Unanimous direction: %d / %d\n",
            n_unanimous, nrow(mod_crossmodel_wide)))
cat(sprintf("  Any-model sig      : %d / %d (BH-FDR < %.2f)\n",
            n_any_sig, nrow(mod_crossmodel_wide),
            STATISTICAL_PARAMS$fdr_threshold))
cat(sprintf("  All-model sig      : %d / %d\n",
            n_all_sig, nrow(mod_crossmodel_wide)))

# Highlight robustly significant findings (sig in >= 3 of 5 models)
robust_sig <- mod_crossmodel_wide %>%
    filter(n_models_sig >= 3) %>%
    arrange(desc(n_models_sig), stratum, desc(abs(median_beta)))

if (nrow(robust_sig) > 0) {
    cat(sprintf("\n  Robust findings (sig in >=3 of 5 models):\n"))
    for (i in seq_len(nrow(robust_sig))) {
        r <- robust_sig[i, ]
        direction <- ifelse(r$median_beta > 0, "+", "-")
        cat(sprintf("    %s %-22s [%s]  median beta=%+.4f  sig in %d/%d models\n",
                    direction, r$outcome, r$stratum,
                    r$median_beta, r$n_models_sig, r$n_models))
    }
}

cat(sprintf("\n  Output:\n"))
cat(sprintf("    results/%s_modulescores_crossmodel_long.csv\n", CELL_TYPE))
cat(sprintf("    results/%s_modulescores_crossmodel_summary.csv\n", CELL_TYPE))
cat(sprintf("    figures/%s.{pdf,png,svg}\n", slug))
cat(strrep("-", 72), "\n", sep = "")
cat("\nv §5.6 module scores cross-model agreement complete\n")
cat("  Next: §7 -- proliferation markers (8 genes, donor Wilcoxon + RLM)\n")
cat("  (§6 is cross-cell-type viz placeholder; do after OPC + Microglia runs)\n")

---
## 20 · Proliferation markers — Wilcoxon (donor-paired)

**Why.** Marker-level arrest test, read directly off log-normalized expression rather than through a score. The source calls this the rigorous arrest test for post-mitotic cells, and section 06's H1 verdict is settled here rather than in sections 08-13.

The panel spans both directions: mitotic markers (MKI67, TOP2A, HMGB2, CENPF, BIRC5, CCNB1, CCNB2, UBE2C) are expected **down** under arrest, while the CDK inhibitors (CDKN2A/1A/1B/2B) and the p53 arm (TP53, GADD45A/B/G) are expected **up**. Genes are tested individually, never pooled into one score.

**Test.** Paired Wilcoxon signed-rank.  
**Outcome.** Per-donor mean log-normalized expression, per gene.  
**β.** Mean paired difference, `SnC − Non-SnC`.  
**CI.** Hodges-Lehmann.  
**FDR.** BH within stratum, across the markers found.  
**Layer.** `GetAssayData(obj_ct, assay = "RNA", layer = "data")` — log-normalized, not residuals.  

**Display.** Box + violin grid, stratum × gene. `Expr_<gene>` columns are added to `md` here and reused by section 21.

In [ ]:
# MODULE 05 -- Senescence Enrichment & Cell Cycle Analysis
# §7.1 -- PROLIFERATION MARKERS: WILCOXON (donor-level paired)
# Stats:  Paired Wilcoxon signed-rank on per-donor mean log-normalized
#         expression of 8 canonical proliferation markers.
#         beta = mean(SnC - Non-SnC) per gene.
#         Hodges-Lehmann CI from wilcox.test(conf.int=TRUE).
# Scale:  score_units (log-normalized expression difference; raw decimal)
# FDR:    BH within (stratum) across 8 markers
#
# Biology: If SnC cells are truly arrested, they should have LOWER
# expression of MKI67, TOP2A, HMGB2, CENPF, BIRC5, CCNB1, CCNB2, UBE2C
# than Non-SnC cells. This is the rigorous arrest test for post-mitotic
# astrocytes (§3.5 H1 verdict relies on §7 confirming arrest direction).
#
# Markers in fixed order from PROLIFERATION_MARKERS (§0).
# Figure: single box+violin grid (3 strata x 8 genes), matching §5.1 pattern.

cat("=", strrep("=", 71), "\n", sep = "")
cat(sprintf("§7.1 -- PROLIFERATION MARKERS: WILCOXON  |  %s / %s\n",
            DATASET, CELL_TYPE))
cat("=", strrep("=", 71), "\n", sep = "")


# ─────────────────────────────────────────────────────────────────────────────
# §7.1.1 STATS
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> STATS\n")
cat("  Test     : Paired Wilcoxon signed-rank\n")
cat("  Outcome  : per-donor mean log-normalized expression (SnC vs Non-SnC)\n")
cat("  beta     : mean paired difference (SnC - Non-SnC)\n")
cat("  CI       : Hodges-Lehmann 95% from wilcox.test(conf.int=TRUE)\n")
cat("  Scale    : score_units (log-normalized expression units)\n")
cat(sprintf("  Markers  : %s\n", paste(PROLIFERATION_MARKERS, collapse = ", ")))
cat(sprintf("  Strata   : %s\n", paste(strata, collapse = ", ")))
cat(sprintf("  FDR      : %s within (stratum) across %d markers\n",
            STATISTICAL_PARAMS$fdr_method, length(PROLIFERATION_MARKERS)))


# ─────────────────────────────────────────────────────────────────────────────
# §7.1.2 EXTRACT MARKER EXPRESSION (one-time; reused by §7.2)
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> EXTRACT EXPRESSION\n")

all_genes <- rownames(obj_ct)
markers_found   <- PROLIFERATION_MARKERS[PROLIFERATION_MARKERS %in% all_genes]
markers_missing <- PROLIFERATION_MARKERS[!PROLIFERATION_MARKERS %in% all_genes]

cat(sprintf("  Requested : %d markers\n", length(PROLIFERATION_MARKERS)))
cat(sprintf("  Found     : %d -> %s\n",
            length(markers_found), paste(markers_found, collapse = ", ")))
if (length(markers_missing) > 0) {
    cat(sprintf("  Missing   : %d -> %s\n",
                length(markers_missing), paste(markers_missing, collapse = ", ")))
}

if (length(markers_found) == 0) {
    stop("✗ No proliferation markers found in obj_ct. Check gene symbol convention.")
}

# Skip extraction if columns already present (e.g. re-running §7.1)
prol_expr_cols <- paste0("Expr_", markers_found)
if (all(prol_expr_cols %in% colnames(md))) {
    cat(sprintf("  Expr_<gene> columns already in md; skipping re-extraction.\n"))
} else {
    expr_mat       <- GetAssayData(obj_ct, assay = "RNA", layer = "data")[markers_found, , drop = FALSE]
    expr_mat_dense <- as.matrix(expr_mat)
    cat(sprintf("  Expression: %d markers x %s cells (log-normalized)\n",
                nrow(expr_mat_dense), fmt_n(ncol(expr_mat_dense))))

    stopifnot(identical(colnames(expr_mat_dense), rownames(md)))

    for (i in seq_along(markers_found)) {
        md[[prol_expr_cols[i]]] <- expr_mat_dense[i, ]
    }

    cat(sprintf("  Added %d columns to md: %s\n",
                length(prol_expr_cols), paste(prol_expr_cols, collapse = ", ")))

    rm(expr_mat, expr_mat_dense); gc(verbose = FALSE)
}


# ─────────────────────────────────────────────────────────────────────────────
# §7.1.3 RUN
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> RUN\n")

prol_wilcox_rows <- list()

for (stratum in strata) {
    md_s        <- filter_to_stratum(md, stratum, STUDY_GROUP_COL)
    arms_s      <- build_donor_arms(md_s, DONOR_COL, SEN_LABEL_COL,
                                    STATISTICAL_PARAMS$min_cells_per_group)
    paired_ids  <- unique(arms_s$donor[arms_s$paired])
    n_paired    <- length(paired_ids)

    if (n_paired < 5) {
        cat(sprintf("  [%s] only %d paired donors -- skipping\n",
                    stratum, n_paired))
        next
    }

    md_pair <- md_s[md_s[[DONOR_COL]] %in% paired_ids, , drop = FALSE]
    md_pair$is_snc_arm <- as.character(md_pair[[SEN_LABEL_COL]]) %in%
                          c("1", "TRUE", "True", "Senescent")

    cell_cnt_snc <- sum(arms_s$n_cells[arms_s$donor %in% paired_ids &
                                       arms_s$arm %in% c("1", "TRUE", "True", "Senescent")])
    cell_cnt_non <- sum(arms_s$n_cells[arms_s$donor %in% paired_ids &
                                       !(arms_s$arm %in% c("1", "TRUE", "True", "Senescent"))])

    for (gene in markers_found) {
        expr_col <- paste0("Expr_", gene)

        per_donor <- md_pair %>%
            group_by(donor   = .data[[DONOR_COL]],
                     is_snc  = is_snc_arm) %>%
            summarise(mean_expr = mean(.data[[expr_col]], na.rm = TRUE),
                     .groups = "drop")

        wide_df <- per_donor %>%
            tidyr::pivot_wider(names_from   = is_snc,
                              values_from  = mean_expr,
                              names_prefix = "expr_")

        if (!"expr_TRUE"  %in% names(wide_df)) next
        if (!"expr_FALSE" %in% names(wide_df)) next
        wide_df <- wide_df[!is.na(wide_df$expr_TRUE) &
                           !is.na(wide_df$expr_FALSE), , drop = FALSE]
        if (nrow(wide_df) < 5) next

        snc_expr <- wide_df$expr_TRUE
        non_expr <- wide_df$expr_FALSE
        diffs    <- snc_expr - non_expr

        wt <- tryCatch({
            wilcox.test(snc_expr, non_expr, paired = TRUE,
                        exact = FALSE, conf.int = TRUE)
        }, error = function(e) NULL)

        if (is.null(wt)) next

        n_obs           <- sum(diffs != 0)
        W_stat          <- as.numeric(wt$statistic)
        W_max           <- if (n_obs > 0) n_obs * (n_obs + 1) / 2 else NA_real_
        rank_biserial_r <- if (!is.na(W_max) && W_max > 0) (2 * W_stat / W_max) - 1 else NA_real_
        cohens_d        <- if (sd(diffs) > 0) mean(diffs) / sd(diffs) else NA_real_
        pct_higher      <- 100 * mean(diffs > 0)

        is_snc_cells     <- md_pair$is_snc_arm
        snc_detect_pct   <- 100 * mean(md_pair[is_snc_cells, expr_col]  > 0, na.rm = TRUE)
        non_detect_pct   <- 100 * mean(md_pair[!is_snc_cells, expr_col] > 0, na.rm = TRUE)

        row <- tidy_model_results(
            stratum      = stratum,
            outcome      = gene,
            model        = "wilcoxon",
            n_donors     = nrow(wide_df),
            n_cells_test = cell_cnt_snc,
            n_cells_ref  = cell_cnt_non,
            estimate     = mean(diffs),
            se           = sd(diffs) / sqrt(nrow(wide_df)),
            ci_low       = if (!is.null(wt$conf.int)) wt$conf.int[1] else NA_real_,
            ci_high      = if (!is.null(wt$conf.int)) wt$conf.int[2] else NA_real_,
            statistic    = W_stat,
            p_value      = wt$p.value,
            extra        = list(
                beta_scale         = "score_units",
                expr_col           = expr_col,
                hl_pseudomedian    = if (!is.null(wt$estimate)) as.numeric(wt$estimate) else NA_real_,
                median_diff        = median(diffs),
                mean_snc_expr      = mean(snc_expr),
                mean_nonsnc_expr   = mean(non_expr),
                snc_detection_pct  = snc_detect_pct,
                non_detection_pct  = non_detect_pct,
                rank_biserial_r    = rank_biserial_r,
                cohens_d           = cohens_d,
                pct_donors_higher  = pct_higher
            )
        )
        prol_wilcox_rows[[paste(stratum, gene, sep = "|")]] <- row
    }
}

prol_wilcox_df <- bind_rows(prol_wilcox_rows)


# ─────────────────────────────────────────────────────────────────────────────
# §7.1.4 BH-FDR within (stratum)
# ─────────────────────────────────────────────────────────────────────────────
prol_wilcox_df <- prol_wilcox_df %>%
    group_by(stratum) %>%
    mutate(p_adj = p.adjust(p_value, method = STATISTICAL_PARAMS$fdr_method)) %>%
    ungroup() %>%
    mutate(sig = sig_stars(p_adj))


# ─────────────────────────────────────────────────────────────────────────────
# §7.1.5 RESULTS
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> RESULTS  (beta = mean paired difference, scale = log-norm expression)\n")

cat(sprintf("\n  %-22s %-8s %5s %+10s %+10s %+10s %7s %7s %8s %8s %4s\n",
            "Stratum", "Gene", "N",
            "beta", "CI_low", "CI_high",
            "SnC%", "Non%",
            "p_raw", "p_adj", "Sig"))
cat("  ", strrep("-", 115), "\n", sep = "")

for (s in strata) {
    df_s <- prol_wilcox_df %>% filter(stratum == s)
    df_s <- df_s[match(markers_found, df_s$outcome), ]
    df_s <- df_s[!is.na(df_s$stratum), ]

    for (i in seq_len(nrow(df_s))) {
        r <- df_s[i, ]
        cat(sprintf("  %-22s %-8s %5d %+9.4f %+9.4f %+9.4f %6.1f%% %6.1f%% %8.1e %8.1e %4s\n",
                    substr(r$stratum, 1, 22), substr(r$outcome, 1, 8),
                    r$n_donors,
                    r$estimate, r$ci_low, r$ci_high,
                    r$snc_detection_pct, r$non_detection_pct,
                    r$p_value, r$p_adj, r$sig))
    }
}

save_table(prol_wilcox_df, paste0(CELL_TYPE, "_proliferation_wilcoxon"))


# ─────────────────────────────────────────────────────────────────────────────
# §7.1.6 FIGURE -- single box+violin grid (mirrors §5.1 pattern)
#
# Layout: facet_grid2(stratum ~ gene). 3 stratum rows x 8 gene cols.
# Gene top-strips: uniform dark gray (all canonical proliferation markers).
# Stratum side-strips: STUDY_GROUP_COLORS.
# Each panel: x = Non-SnC vs SnC, y = per-donor mean log-norm expression.
#             violin (alpha 0.5) + boxplot (white fill) + jittered points.
# Annotations above each panel: BH-FDR sig stars + "beta=+0.0045".
# scales = "free_y" so each gene's natural baseline is visible.
# Genes in fixed canonical order from PROLIFERATION_MARKERS.
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> FIGURE\n")

if (!requireNamespace("ggh4x", quietly = TRUE)) {
    cat("  Installing ggh4x...\n")
    install.packages("ggh4x", repos = "https://cloud.r-project.org")
}
suppressPackageStartupMessages(library(ggh4x))

# Build per-donor plot data (one mean expression per donor x arm x gene)
plot_rows <- list()
for (stratum in strata) {
    md_s   <- filter_to_stratum(md, stratum, STUDY_GROUP_COL)
    arms_s <- build_donor_arms(md_s, DONOR_COL, SEN_LABEL_COL,
                               STATISTICAL_PARAMS$min_cells_per_group)
    paired_ids <- unique(arms_s$donor[arms_s$paired])
    if (length(paired_ids) < 5) next

    md_pair <- md_s[md_s[[DONOR_COL]] %in% paired_ids, , drop = FALSE]
    md_pair$is_snc_arm <- as.character(md_pair[[SEN_LABEL_COL]]) %in%
                          c("1", "TRUE", "True", "Senescent")
    md_pair$arm_label  <- ifelse(md_pair$is_snc_arm, "SnC", "Non-SnC")

    for (gene in markers_found) {
        expr_col <- paste0("Expr_", gene)

        donor_means <- md_pair %>%
            group_by(donor     = .data[[DONOR_COL]],
                     arm_label = arm_label) %>%
            summarise(mean_expr = mean(.data[[expr_col]], na.rm = TRUE),
                     .groups = "drop") %>%
            mutate(stratum = stratum,
                  gene    = gene)

        plot_rows[[paste(stratum, gene, sep = "|")]] <- donor_means
    }
}
plot_df <- bind_rows(plot_rows)
plot_df$stratum   <- factor(plot_df$stratum,   levels = strata)
plot_df$arm_label <- factor(plot_df$arm_label, levels = c("Non-SnC", "SnC"))
plot_df$gene      <- factor(plot_df$gene,      levels = markers_found)

# Significance annotations: positioned above each panel's max value
sig_annot <- prol_wilcox_df %>%
    select(stratum, gene = outcome, sig, p_adj, estimate) %>%
    mutate(stratum = factor(stratum, levels = strata),
          gene    = factor(gene,    levels = markers_found))

annot_y <- plot_df %>%
    group_by(stratum, gene) %>%
    summarise(panel_max = max(mean_expr, na.rm = TRUE),
             panel_min = min(mean_expr, na.rm = TRUE),
             panel_range = panel_max - panel_min,
             .groups = "drop") %>%
    mutate(panel_range = ifelse(panel_range == 0, max(abs(panel_max), 1e-6) * 0.2, panel_range),
          y_sig  = panel_max + panel_range * 0.18,
          y_beta = panel_max + panel_range * 0.07)

sig_annot <- sig_annot %>% left_join(annot_y, by = c("stratum", "gene"))

# Strip palettes
gene_strip_palette    <- rep("#444444", length(markers_found))
names(gene_strip_palette) <- markers_found

stratum_strip_palette <- sapply(strata, function(s) {
    if (s %in% names(STUDY_GROUP_COLORS)) STUDY_GROUP_COLORS[[s]] else "#7F7F7F"
})
names(stratum_strip_palette) <- strata

gene_text_colors    <- text_color_for_bg(gene_strip_palette)
stratum_text_colors <- text_color_for_bg(stratum_strip_palette)

p_prol_wilcox <- ggplot(plot_df,
                       aes(x = arm_label, y = mean_expr, fill = arm_label)) +
    geom_violin(alpha = 0.5, linewidth = 0.3, scale = "width") +
    geom_boxplot(width = 0.18, outlier.shape = NA, alpha = 0.7,
                fill = "white", color = "black", linewidth = 0.3) +
    geom_jitter(width = 0.12, size = 0.4, alpha = 0.4, color = "grey30") +
    geom_text(data = sig_annot,
             aes(x = 1.5, y = y_sig, label = sig),
             inherit.aes = FALSE,
             size = 3.5, fontface = "bold") +
    geom_text(data = sig_annot,
             aes(x = 1.5, y = y_beta,
                 label = sprintf("beta=%+.3f", estimate)),
             inherit.aes = FALSE,
             size = 2.2, color = "grey30") +
    scale_fill_manual(values = c("Non-SnC" = "#D3D3D3", "SnC" = "#C44E52")) +
    facet_grid2(stratum ~ gene,
               scales = "free_y",
               strip = strip_themed(
                   background_x = elem_list_rect(
                       fill  = unname(gene_strip_palette),
                       color = "black"
                   ),
                   text_x = elem_list_text(
                       color = unname(gene_text_colors),
                       face  = "bold",
                       size  = 7
                   ),
                   background_y = elem_list_rect(
                       fill  = unname(stratum_strip_palette),
                       color = "black"
                   ),
                   text_y = elem_list_text(
                       color = unname(stratum_text_colors),
                       face  = "bold",
                       size  = 8
                   )
               )) +
    labs(title    = sprintf("Proliferation markers -- Wilcoxon (paired, donor-level) -- %s",
                            CELL_TYPE),
         subtitle = sprintf("Top strips = Gene. Right strips = Stratum. SnC red / Non-SnC grey. *,**,*** = BH-FDR < %.2f. beta < 0 supports arrest.",
                            STATISTICAL_PARAMS$fdr_threshold),
         x        = NULL,
         y        = "Per-donor mean log-norm expression",
         fill     = NULL) +
    theme_clean(base_size = 8) +
    theme(legend.position = "top",
         strip.text.x    = element_text(size = 6, face = "bold"),
         axis.text.x     = element_text(size = 6),
         axis.text.y     = element_text(size = 6),
         panel.spacing.x = unit(0.15, "lines"),
         panel.spacing.y = unit(0.30, "lines"))

# Figure dims: ~1.3"/gene x 1.7"/stratum (matches §5.1 scaling)
fig_width  <- 2 + length(markers_found) * 1.3
fig_height <- 2 + length(strata) * 1.7

save_figure(p_prol_wilcox,
           slug   = paste0(CELL_TYPE, "_proliferation_wilcoxon"),
           width  = fig_width,
           height = fig_height)

options(repr.plot.width = fig_width + 1, repr.plot.height = fig_height + 0.5)
tryCatch(
    print(p_prol_wilcox),
    error = function(e) {
        cat(sprintf("    [inline preview unavailable: %s]\n",
                    conditionMessage(e)))
    }
)


# ─────────────────────────────────────────────────────────────────────────────
# §7.1.7 SUMMARY + biological interpretation
# ─────────────────────────────────────────────────────────────────────────────
n_sig_rows <- sum(prol_wilcox_df$p_adj < STATISTICAL_PARAMS$fdr_threshold,
                 na.rm = TRUE)
n_neg_sig  <- sum(prol_wilcox_df$p_adj < STATISTICAL_PARAMS$fdr_threshold &
                  prol_wilcox_df$estimate < 0, na.rm = TRUE)
n_pos_sig  <- sum(prol_wilcox_df$p_adj < STATISTICAL_PARAMS$fdr_threshold &
                  prol_wilcox_df$estimate > 0, na.rm = TRUE)

cat("\n", strrep("-", 72), "\n", sep = "")
cat(sprintf("  §7.1 SUMMARY  |  %s / %s\n", DATASET, CELL_TYPE))
cat(strrep("-", 72), "\n", sep = "")
cat(sprintf("  Test          : Paired Wilcoxon signed-rank\n"))
cat(sprintf("  beta_scale    : score_units (log-norm expression)\n"))
cat(sprintf("  Markers       : %d (canonical proliferation)\n", length(markers_found)))
cat(sprintf("  Significant   : %d of %d at BH-FDR < %.2f\n",
            n_sig_rows, nrow(prol_wilcox_df), STATISTICAL_PARAMS$fdr_threshold))
cat(sprintf("    SnC LOWER  (consistent with arrest)        : %d\n", n_neg_sig))
cat(sprintf("    SnC HIGHER (contradicts arrest hypothesis) : %d\n", n_pos_sig))

if (n_sig_rows > 0) {
    cat(sprintf("\n  Significant rows (sorted by stratum, magnitude):\n"))
    sig_rows <- prol_wilcox_df %>%
        filter(p_adj < STATISTICAL_PARAMS$fdr_threshold) %>%
        arrange(stratum, desc(abs(estimate)))
    for (i in seq_len(nrow(sig_rows))) {
        r <- sig_rows[i, ]
        direction <- ifelse(r$estimate < 0, "v (arrest)", "^ (proliferating)")
        cat(sprintf("    %-18s %-8s [%s]  beta=%+.4f  p_adj=%.2e\n",
                    direction, r$outcome, r$stratum,
                    r$estimate, r$p_adj))
    }
}

cat(sprintf("\n  Detection rates (cell-level, %% of cells with expression > 0):\n"))
detect_summary <- prol_wilcox_df %>%
    filter(stratum == "All") %>%
    select(outcome, snc_detection_pct, non_detection_pct)
detect_summary <- detect_summary[match(markers_found, detect_summary$outcome), ]
for (i in seq_len(nrow(detect_summary))) {
    r <- detect_summary[i, ]
    cat(sprintf("    %-8s : SnC %5.1f%%  Non-SnC %5.1f%%\n",
                r$outcome, r$snc_detection_pct, r$non_detection_pct))
}

cat(sprintf("\n  Figure: %s_proliferation_wilcoxon.{pdf,png,svg}  (%.1f x %.1f in)\n",
            CELL_TYPE, fig_width, fig_height))
cat(strrep("-", 72), "\n", sep = "")
cat("\nv §7.1 proliferation Wilcoxon complete\n")
cat("  Next: §7.2 -- RLM (covariate-adjusted donor-level)\n")

---
## 21 · Proliferation markers — RLM on paired differences

**Why.** Covariate-adjusted marker-level arrest test. Reuses the `Expr_<gene>` columns from section 20 rather than re-extracting the matrix.

**Test.** Robust LM, `lmrob` MM-estimator, `setting = "KS2014"`.  
**Outcome.** Donor-level paired difference in mean log-normalized expression.  
**Formula.** `diff ~ Sex + Cohort + delta_log10_umi`  
**β.** Intercept.  
**FDR.** BH within stratum, across the markers found.  

**Display.** Per-stratum forest, adaptive layout.

In [ ]:
# MODULE 05 -- Senescence Enrichment & Cell Cycle Analysis
# §7.2 -- PROLIFERATION MARKERS: RLM (paired diffs, covariate-adjusted)
# Stats:
#   Test:     Robust LM via lmrob (MM-estimator, KS2014)
#   Outcome:  donor-level paired difference (mean_SnC - mean_NonSnC)
#             of log-normalized expression
#   Formula:  diff ~ Sex + Cohort + delta_log10_umi
#   beta:     intercept (= adjusted mean diff after covariate adjustment)
#   Scale:    score_units (log-norm expression units)
#   FDR:      BH within (stratum) across 8 markers
#
# Markers in fixed order from PROLIFERATION_MARKERS (§0).
# Figure: per-stratum forest plot, fully adaptive layout.
# Reuses Expr_<gene> columns added to md by §7.1.2.

cat("=", strrep("=", 71), "\n", sep = "")
cat(sprintf("§7.2 -- PROLIFERATION MARKERS: RLM (paired diffs)  |  %s / %s\n",
            DATASET, CELL_TYPE))
cat("=", strrep("=", 71), "\n", sep = "")


# ─────────────────────────────────────────────────────────────────────────────
# §7.2.1 STATS
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> STATS\n")
cat("  Test     : Robust LM via lmrob (MM-estimator, KS2014)\n")
cat("  Outcome  : donor-level paired difference (mean_SnC - mean_NonSnC)\n")
cat("  Formula  : diff ~ Sex + Cohort + delta_log10_umi\n")
cat("  beta     : intercept (= adjusted mean diff)\n")
cat("  Scale    : score_units (log-norm expression units)\n")
cat(sprintf("  Strata   : %s\n", paste(strata, collapse = ", ")))
cat(sprintf("  FDR      : %s within (stratum) across %d markers\n",
            STATISTICAL_PARAMS$fdr_method,
            length(PROLIFERATION_MARKERS)))


# ─────────────────────────────────────────────────────────────────────────────
# §7.2.2 RUN
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> RUN\n")

# Resolve markers found in obj_ct (assumes §7.1 has run; rebuild defensively)
if (!exists("markers_found")) {
    all_genes <- rownames(obj_ct)
    markers_found <- PROLIFERATION_MARKERS[PROLIFERATION_MARKERS %in% all_genes]
    if (length(markers_found) == 0) stop("✗ No markers in obj_ct.")
    cat(sprintf("  Re-resolved markers_found: %d\n", length(markers_found)))
}

# Verify Expr_<gene> columns exist in md (extracted by §7.1.2)
prol_expr_cols <- paste0("Expr_", markers_found)
missing_cols <- setdiff(prol_expr_cols, colnames(md))
if (length(missing_cols) > 0) {
    stop(sprintf("✗ Missing expression columns in md: %s. Re-run §7.1 first.",
                 paste(missing_cols, collapse = ", ")))
}

mod_prol_rlm_rows <- list()

for (stratum in strata) {
    md_s        <- filter_to_stratum(md, stratum, STUDY_GROUP_COL)
    arms_s      <- build_donor_arms(md_s, DONOR_COL, SEN_LABEL_COL,
                                    STATISTICAL_PARAMS$min_cells_per_group)
    paired_ids  <- unique(arms_s$donor[arms_s$paired])
    n_paired    <- length(paired_ids)

    if (n_paired < 5) {
        cat(sprintf("  [%s] only %d paired donors -- skipping\n",
                    stratum, n_paired))
        next
    }

    donor_meta_s <- build_donor_meta(
        md_s, paired_ids,
        DONOR_COL, SEN_LABEL_COL,
        SEX_COL, AGE_COL, COHORT_COL,
        has_cohort = HAS_COHORT
    )

    md_pair <- md_s[md_s[[DONOR_COL]] %in% paired_ids, , drop = FALSE]
    md_pair$is_snc_arm <- as.character(md_pair[[SEN_LABEL_COL]]) %in%
                          c("1", "TRUE", "True", "Senescent")

    cell_cnt_snc <- sum(arms_s$n_cells[arms_s$donor %in% paired_ids &
                                       arms_s$arm %in% c("1", "TRUE", "True", "Senescent")])
    cell_cnt_non <- sum(arms_s$n_cells[arms_s$donor %in% paired_ids &
                                       !(arms_s$arm %in% c("1", "TRUE", "True", "Senescent"))])

    for (gene in markers_found) {
        expr_col <- paste0("Expr_", gene)

        per_donor <- md_pair %>%
            group_by(donor   = .data[[DONOR_COL]],
                     is_snc  = is_snc_arm) %>%
            summarise(mean_expr = mean(.data[[expr_col]], na.rm = TRUE),
                     .groups = "drop")

        wide_df <- per_donor %>%
            tidyr::pivot_wider(names_from   = is_snc,
                              values_from  = mean_expr,
                              names_prefix = "expr_")

        if (!"expr_TRUE"  %in% names(wide_df)) next
        if (!"expr_FALSE" %in% names(wide_df)) next
        wide_df <- wide_df[!is.na(wide_df$expr_TRUE) &
                           !is.na(wide_df$expr_FALSE), , drop = FALSE]
        if (nrow(wide_df) < 5) next

        wide_df$diff <- wide_df$expr_TRUE - wide_df$expr_FALSE

        reg_df <- wide_df %>% left_join(donor_meta_s, by = "donor")

        sex_levels    <- length(unique(reg_df[[SEX_COL]]))
        cohort_levels <- if (HAS_COHORT) length(unique(reg_df[[COHORT_COL]])) else 0

        rhs <- character(0)
        if (sex_levels    >= 2) rhs <- c(rhs, SEX_COL)
        if (cohort_levels >= 2) rhs <- c(rhs, COHORT_COL)
        rhs <- c(rhs, "delta_log10_umi")

        formula_str <- paste("diff ~", paste(rhs, collapse = " + "))
        formula_obj <- as.formula(formula_str)

        fit <- tryCatch({
            robustbase::lmrob(formula_obj, data = reg_df, setting = "KS2014")
        }, error = function(e) {
            cat(sprintf("  [%s | %s] lmrob ERROR: %s\n",
                        stratum, gene, e$message))
            NULL
        }, warning = function(w) {
            tryCatch(robustbase::lmrob(formula_obj, data = reg_df,
                                       setting = "KS2014"),
                    error = function(e) NULL)
        })

        if (is.null(fit)) next

        co <- summary(fit)$coefficients
        if (!"(Intercept)" %in% rownames(co)) next

        beta      <- co["(Intercept)", "Estimate"]
        se        <- co["(Intercept)", "Std. Error"]
        statistic <- co["(Intercept)", "t value"]
        p_value   <- co["(Intercept)", "Pr(>|t|)"]

        z <- qnorm(1 - (1 - STATISTICAL_PARAMS$confidence_level) / 2)
        ci_low  <- beta - z * se
        ci_high <- beta + z * se

        # Cell-level detection rates (carried over from Wilcoxon for context)
        is_snc_cells   <- md_pair$is_snc_arm
        snc_detect_pct <- 100 * mean(md_pair[is_snc_cells, expr_col]  > 0, na.rm = TRUE)
        non_detect_pct <- 100 * mean(md_pair[!is_snc_cells, expr_col] > 0, na.rm = TRUE)

        row <- tidy_model_results(
            stratum      = stratum,
            outcome      = gene,
            model        = "rlm",
            n_donors     = nrow(wide_df),
            n_cells_test = cell_cnt_snc,
            n_cells_ref  = cell_cnt_non,
            estimate     = beta,
            se           = se,
            ci_low       = ci_low,
            ci_high      = ci_high,
            statistic    = statistic,
            p_value      = p_value,
            extra        = list(
                beta_scale         = "score_units",
                expr_col           = expr_col,
                formula            = formula_str,
                cohort_in_model    = cohort_levels >= 2,
                sex_in_model       = sex_levels >= 2,
                delta_umi_in_model = TRUE,
                converged          = isTRUE(fit$converged),
                n_covariate_terms  = nrow(co) - 1,
                snc_detection_pct  = snc_detect_pct,
                non_detection_pct  = non_detect_pct
            )
        )
        mod_prol_rlm_rows[[paste(stratum, gene, sep = "|")]] <- row
    }
}

prol_rlm_df <- bind_rows(mod_prol_rlm_rows)


# ─────────────────────────────────────────────────────────────────────────────
# §7.2.3 BH-FDR within (stratum)
# ─────────────────────────────────────────────────────────────────────────────
prol_rlm_df <- prol_rlm_df %>%
    group_by(stratum) %>%
    mutate(p_adj = p.adjust(p_value, method = STATISTICAL_PARAMS$fdr_method)) %>%
    ungroup() %>%
    mutate(sig = sig_stars(p_adj))


# ─────────────────────────────────────────────────────────────────────────────
# §7.2.4 RESULTS
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> RESULTS  (beta = adjusted intercept, scale = log-norm expression)\n")

cat(sprintf("\n  %-22s %-8s %5s %+10s %+10s %+10s %7s %7s %8s %8s %4s\n",
            "Stratum", "Gene", "N",
            "beta", "CI_low", "CI_high",
            "SnC%", "Non%",
            "p_raw", "p_adj", "Sig"))
cat("  ", strrep("-", 115), "\n", sep = "")

for (s in strata) {
    df_s <- prol_rlm_df %>% filter(stratum == s)
    df_s <- df_s[match(markers_found, df_s$outcome), ]
    df_s <- df_s[!is.na(df_s$stratum), ]

    for (i in seq_len(nrow(df_s))) {
        r <- df_s[i, ]
        cat(sprintf("  %-22s %-8s %5d %+9.4f %+9.4f %+9.4f %6.1f%% %6.1f%% %8.1e %8.1e %4s\n",
                    substr(r$stratum, 1, 22), substr(r$outcome, 1, 8),
                    r$n_donors,
                    r$estimate, r$ci_low, r$ci_high,
                    r$snc_detection_pct, r$non_detection_pct,
                    r$p_value, r$p_adj, r$sig))
    }
}

save_table(prol_rlm_df, paste0(CELL_TYPE, "_proliferation_rlm"))


# ─────────────────────────────────────────────────────────────────────────────
# §7.2.5 FIGURE -- per-stratum forest plot, FULLY ADAPTIVE layout
#
# beta < 0 = SnC has LOWER expression after covariate adjustment (= arrest)
# beta > 0 = SnC has HIGHER expression (= contradicts arrest)
# Markers in fixed PROLIFERATION_MARKERS order (no sub-group divider).
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> FIGURE\n")

if (!exists("FOREST_LAYOUT")) {
    FOREST_LAYOUT <- list(
        em_per_char_mono   = 0.62,
        em_per_char_prop   = 0.55,
        inter_col_gap      = 1.5,
        panel_right_pad    = 0.8,
        forest_share_mult  = 0.7,
        inches_per_em      = 0.020,
        min_fig_width_in   = 6.0,
        min_fig_height_in  = 2.5,
        row_height_in      = 0.30,
        base_height_in     = 0.9,
        pad_x_axis_frac    = 0.10,
        pad_x_axis_null    = 0.02
    )
}

# Override x-axis label for proliferation expression context
cfg <- EFFECT_CONFIGS$score_units
cfg$x_label <- "beta (log-norm expression, SnC - Non-SnC)"

for (s in strata) {
    df_s <- prol_rlm_df %>%
        filter(stratum == s) %>%
        mutate(outcome = factor(outcome, levels = markers_found)) %>%
        arrange(outcome)

    if (nrow(df_s) == 0) {
        cat(sprintf("  [%s] no rows -- skipping figure\n", s))
        next
    }

    df_s$row_label  <- as.character(df_s$outcome)
    df_s$eff_str    <- sapply(df_s$estimate, cfg$fmt_effect)
    df_s$ci_str     <- mapply(cfg$fmt_ci, df_s$ci_low, df_s$ci_high)
    df_s$p_str      <- sapply(df_s$p_adj, fmt_p_short)
    df_s$row_color  <- "#222222"
    df_s$is_sig     <- df_s$sig != "ns"

    n_rows <- nrow(df_s)

    h_gene <- "Gene"
    h_eff  <- cfg$eff_h_label
    h_ci   <- cfg$ci_h_label
    h_padj <- "p(adj)"
    h_sig  <- "Sig"

    nchar_label <- max(nchar(c(df_s$row_label, h_gene)), na.rm = TRUE)
    nchar_eff   <- max(nchar(c(df_s$eff_str,   h_eff)),  na.rm = TRUE)
    nchar_ci    <- max(nchar(c(df_s$ci_str,    h_ci)),   na.rm = TRUE)
    nchar_padj  <- max(nchar(c(df_s$p_str,     h_padj)), na.rm = TRUE)
    nchar_sig   <- max(nchar(c(df_s$sig,       h_sig)),  na.rm = TRUE)

    em_label <- nchar_label * FOREST_LAYOUT$em_per_char_prop
    em_eff   <- nchar_eff   * FOREST_LAYOUT$em_per_char_mono
    em_ci    <- nchar_ci    * FOREST_LAYOUT$em_per_char_mono
    em_padj  <- nchar_padj  * FOREST_LAYOUT$em_per_char_mono
    em_sig   <- nchar_sig   * FOREST_LAYOUT$em_per_char_mono

    gap <- FOREST_LAYOUT$inter_col_gap

    x_label_pos <- 0
    x_eff_pos   <- em_label + gap
    x_ci_pos    <- em_label + gap + em_eff + gap
    left_em     <- em_label + gap + em_eff + gap + em_ci + FOREST_LAYOUT$panel_right_pad

    x_padj_pos  <- em_padj / 2
    x_sig_pos   <- em_padj + gap + em_sig / 2
    right_em    <- em_padj + gap + em_sig + FOREST_LAYOUT$panel_right_pad

    forest_em <- (left_em + right_em) * FOREST_LAYOUT$forest_share_mult
    total_em  <- left_em + forest_em + right_em
    fig_width <- max(total_em * FOREST_LAYOUT$inches_per_em,
                    FOREST_LAYOUT$min_fig_width_in)

    df_s$y      <- n_rows:1
    header_y    <- n_rows + 0.55
    underline_y <- n_rows + 0.20
    y_lim       <- c(0.4, n_rows + 1.0)

    stratum_color <- if (s %in% names(STUDY_GROUP_COLORS)) STUDY_GROUP_COLORS[[s]] else "#7F7F7F"

    n_sig <- sum(df_s$is_sig)
    title_txt <- sprintf("Proliferation RLM | %s | [%s] | %d/%d sig at BH-FDR",
                         CELL_TYPE, s, n_sig, nrow(df_s))

    p_left <- ggplot(df_s) +
        geom_text(aes(x = x_label_pos, y = y, label = row_label,
                     color = row_color,
                     fontface = ifelse(is_sig, "bold", "plain")),
                 hjust = 0, size = 2.4, family = "mono") +
        geom_text(aes(x = x_eff_pos, y = y, label = eff_str,
                     fontface = ifelse(is_sig, "bold", "plain")),
                 hjust = 0, size = 2.2, family = "mono", color = "#222222") +
        geom_text(aes(x = x_ci_pos, y = y, label = ci_str),
                 hjust = 0, size = 2.0, family = "mono", color = "#666666") +
        annotate("text", x = x_label_pos, y = header_y, label = h_gene,
                fontface = "bold", hjust = 0, size = 2.4, color = "#222222") +
        annotate("text", x = x_eff_pos, y = header_y, label = h_eff,
                fontface = "bold", hjust = 0, size = 2.4, color = "#222222") +
        annotate("text", x = x_ci_pos, y = header_y, label = h_ci,
                fontface = "bold", hjust = 0, size = 2.2, color = "#222222") +
        annotate("segment", x = 0, xend = left_em,
                y = underline_y, yend = underline_y,
                color = "#333333", linewidth = 0.3) +
        scale_color_identity() +
        scale_x_continuous(limits = c(0, left_em), expand = c(0, 0)) +
        scale_y_continuous(limits = y_lim, expand = c(0, 0)) +
        labs(title = title_txt) +
        theme_void() +
        theme(
            plot.title = element_text(size = 9, face = "bold",
                                     color = stratum_color,
                                     hjust = 0,
                                     margin = margin(t = 2, b = 1)),
            plot.margin = margin(2, 2, 2, 4)
        )

    eff_vals    <- c(df_s$estimate, df_s$ci_low, df_s$ci_high)
    eff_finite  <- eff_vals[is.finite(eff_vals)]
    if (length(eff_finite) == 0) eff_finite <- c(-0.001, 0.001)

    data_range  <- diff(range(eff_finite))
    if (data_range == 0) data_range <- max(abs(eff_finite[1]), 1e-6) * 0.2

    pad         <- data_range * FOREST_LAYOUT$pad_x_axis_frac
    null_pad    <- data_range * FOREST_LAYOUT$pad_x_axis_null
    x_range     <- range(eff_finite) + c(-pad, pad)
    x_range[1]  <- min(x_range[1], cfg$null_value - null_pad)
    x_range[2]  <- max(x_range[2], cfg$null_value + null_pad)

    p_forest <- ggplot(df_s) +
        geom_vline(xintercept = cfg$null_value,
                  linetype = "dashed", color = "#999999", linewidth = 0.4) +
        geom_errorbar(aes(y = y, xmin = ci_low, xmax = ci_high),
                     width = 0.18, linewidth = 0.4, color = "#4D4D4D") +
        geom_point(aes(x = estimate, y = y, fill = row_color,
                      size = ifelse(is_sig, 3.5, 2.5)),
                  shape = 23, color = "#222222", stroke = 0.4) +
        scale_fill_identity() +
        scale_size_identity() +
        scale_x_continuous(limits = x_range,
                          labels = cfg$axis_format,
                          breaks = scales::breaks_pretty(n = 4)) +
        scale_y_continuous(limits = y_lim, expand = c(0, 0)) +
        labs(x = cfg$x_label, y = NULL) +
        theme_classic(base_size = 7) +
        theme(
            axis.title.x = element_text(size = 6.5, margin = margin(t = 1)),
            axis.text.x  = element_text(size = 5.5),
            axis.text.y  = element_blank(),
            axis.ticks.y = element_blank(),
            axis.line.y  = element_blank(),
            axis.line.x  = element_line(color = "#444444", linewidth = 0.4),
            panel.grid   = element_blank(),
            plot.margin  = margin(1, 2, 1, 2)
        )

    p_right <- ggplot(df_s) +
        geom_text(aes(x = x_padj_pos, y = y, label = p_str,
                     fontface = ifelse(is_sig, "bold", "plain"),
                     color    = ifelse(is_sig, "#222222", "#666666")),
                 hjust = 0.5, size = 2.2, family = "mono") +
        geom_text(aes(x = x_sig_pos, y = y, label = sig),
                 fontface = "bold", hjust = 0.5, size = 2.4,
                 family = "mono", color = "#222222") +
        annotate("text", x = x_padj_pos, y = header_y, label = h_padj,
                fontface = "bold", hjust = 0.5, size = 2.4, color = "#222222") +
        annotate("text", x = x_sig_pos, y = header_y, label = h_sig,
                fontface = "bold", hjust = 0.5, size = 2.4, color = "#222222") +
        annotate("segment", x = 0, xend = right_em,
                y = underline_y, yend = underline_y,
                color = "#333333", linewidth = 0.3) +
        scale_color_identity() +
        scale_x_continuous(limits = c(0, right_em), expand = c(0, 0)) +
        scale_y_continuous(limits = y_lim, expand = c(0, 0)) +
        theme_void() +
        theme(plot.margin = margin(2, 4, 2, 2))

    composed <- (p_left | p_forest | p_right) +
        plot_layout(widths = c(left_em, forest_em, right_em))

    fig_height <- max(FOREST_LAYOUT$min_fig_height_in,
                     FOREST_LAYOUT$base_height_in +
                         n_rows * FOREST_LAYOUT$row_height_in)

    slug <- sprintf("%s_proliferation_rlm_%s_forest", CELL_TYPE, s)

    save_figure(composed, slug = slug, width = fig_width, height = fig_height)

    options(repr.plot.width = fig_width + 1, repr.plot.height = fig_height + 0.5)
    tryCatch(
        print(composed),
        error = function(e) {
            cat(sprintf("    [inline preview unavailable: %s]\n",
                        conditionMessage(e)))
        }
    )

    cat(sprintf("    fig dims: %.2f x %.2f in (left_em=%.1f, forest_em=%.1f, right_em=%.1f)\n",
                fig_width, fig_height, left_em, forest_em, right_em))
}


# ─────────────────────────────────────────────────────────────────────────────
# §7.2.6 WILCOXON vs RLM direction agreement (TEXT TABLE ONLY)
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> AGREEMENT WITH §7.1 WILCOXON (text only)\n")

prol_agreement <- prol_wilcox_df %>%
    select(stratum, outcome, beta_wilcox = estimate, p_wilcox = p_adj) %>%
    inner_join(
        prol_rlm_df %>% select(stratum, outcome, beta_rlm = estimate, p_rlm = p_adj),
        by = c("stratum", "outcome")
    ) %>%
    mutate(
        same_sign     = sign(beta_wilcox) == sign(beta_rlm),
        wilcox_sig    = !is.na(p_wilcox) & p_wilcox < STATISTICAL_PARAMS$fdr_threshold,
        rlm_sig       = !is.na(p_rlm)    & p_rlm    < STATISTICAL_PARAMS$fdr_threshold,
        both_sig      = wilcox_sig & rlm_sig,
        either_sig    = wilcox_sig | rlm_sig,
        abs_diff      = abs(beta_wilcox - beta_rlm)
    )

cat(sprintf("\n  %-22s %-8s %+10s %+10s %5s %5s %5s\n",
            "Stratum", "Gene",
            "beta_wilcox", "beta_rlm",
            "Same", "WlxSig", "RLMSig"))
cat("  ", strrep("-", 80), "\n", sep = "")

for (s in strata) {
    df_s <- prol_agreement %>% filter(stratum == s)
    df_s <- df_s[match(markers_found, df_s$outcome), ]
    df_s <- df_s[!is.na(df_s$stratum), ]

    for (i in seq_len(nrow(df_s))) {
        r <- df_s[i, ]
        cat(sprintf("  %-22s %-8s %+9.4f %+9.4f %5s %5s %5s\n",
                    substr(r$stratum, 1, 22), substr(r$outcome, 1, 8),
                    r$beta_wilcox, r$beta_rlm,
                    ifelse(r$same_sign, "yes", "FLIP"),
                    ifelse(r$wilcox_sig, "yes", "no"),
                    ifelse(r$rlm_sig,    "yes", "no")))
    }
}

n_sign_disagree <- sum(!prol_agreement$same_sign)
n_both_sig      <- sum(prol_agreement$both_sig)
n_either_sig    <- sum(prol_agreement$either_sig)

if (n_sign_disagree == 0) {
    cat(sprintf("\n  v All %d (stratum x gene) results agree in sign between Wilcoxon and RLM\n",
                nrow(prol_agreement)))
} else {
    cat(sprintf("\n  ! %d / %d (stratum x gene) results FLIP sign between Wilcoxon and RLM\n",
                n_sign_disagree, nrow(prol_agreement)))
}
cat(sprintf("  Both models sig (robust):  %d / %d\n",
            n_both_sig, nrow(prol_agreement)))
cat(sprintf("  Either model sig:          %d / %d\n",
            n_either_sig, nrow(prol_agreement)))


# ─────────────────────────────────────────────────────────────────────────────
# §7.2.7 SUMMARY + biological interpretation
# ─────────────────────────────────────────────────────────────────────────────
n_sig_rows <- sum(prol_rlm_df$p_adj < STATISTICAL_PARAMS$fdr_threshold,
                 na.rm = TRUE)
n_neg_sig  <- sum(prol_rlm_df$p_adj < STATISTICAL_PARAMS$fdr_threshold &
                  prol_rlm_df$estimate < 0, na.rm = TRUE)
n_pos_sig  <- sum(prol_rlm_df$p_adj < STATISTICAL_PARAMS$fdr_threshold &
                  prol_rlm_df$estimate > 0, na.rm = TRUE)

cat("\n", strrep("-", 72), "\n", sep = "")
cat(sprintf("  §7.2 SUMMARY  |  %s / %s\n", DATASET, CELL_TYPE))
cat(strrep("-", 72), "\n", sep = "")
cat(sprintf("  Test          : Robust LM (lmrob, MM-estimator, KS2014)\n"))
cat(sprintf("  beta_scale    : score_units (log-norm expression)\n"))
cat(sprintf("  Adjustments   : Sex + Cohort + delta_log10_umi\n"))
cat(sprintf("  Markers       : %d (canonical proliferation)\n", length(markers_found)))
cat(sprintf("  Significant   : %d of %d at BH-FDR < %.2f\n",
            n_sig_rows, nrow(prol_rlm_df), STATISTICAL_PARAMS$fdr_threshold))
cat(sprintf("    SnC LOWER  (consistent with arrest)        : %d\n", n_neg_sig))
cat(sprintf("    SnC HIGHER (contradicts arrest hypothesis) : %d\n", n_pos_sig))
cat(sprintf("  Sign agreement (vs Wilcoxon): %d / %d\n",
            sum(prol_agreement$same_sign), nrow(prol_agreement)))
cat(sprintf("  Robust hits (sig in both Wilcoxon AND RLM): %d / %d\n",
            n_both_sig, nrow(prol_agreement)))

if (n_sig_rows > 0) {
    cat(sprintf("\n  Significant rows (sorted by stratum, magnitude):\n"))
    sig_rows <- prol_rlm_df %>%
        filter(p_adj < STATISTICAL_PARAMS$fdr_threshold) %>%
        arrange(stratum, desc(abs(estimate)))
    for (i in seq_len(nrow(sig_rows))) {
        r <- sig_rows[i, ]
        direction <- ifelse(r$estimate < 0, "v (arrest)", "^ (proliferating)")
        cat(sprintf("    %-18s %-8s [%s]  beta=%+.4f  p_adj=%.2e\n",
                    direction, r$outcome, r$stratum,
                    r$estimate, r$p_adj))
    }
}

cat(sprintf("\n  Per-stratum forests:\n"))
for (s in strata) {
    cat(sprintf("    %-22s -> %s_proliferation_rlm_%s_forest.{pdf,png,svg}\n",
                s, CELL_TYPE, s))
}
cat(strrep("-", 72), "\n", sep = "")
cat("\nv §7.2 proliferation RLM complete\n")
cat("  Next: §8 -- Save scored .qs + write m05_manifest.json\n")

---
## 22 · Manifest and gate

**Why.** The module has to declare what it produced and under what settings,
so a downstream module — or a reviewer — can tell which run a table came from
without re-reading the notebook. The manifest is the record; the gate is the
check that nothing silently skipped.

**Records.** dataset, cell type, condition tag, strata, `STATISTICAL_PARAMS`,
paired-donor counts per stratum, gene-list sizes after intersection, package
versions from `R_PKG_VERSIONS`, and every table and figure written.

> **Source not in this rebuild.** The code lives in `05_validation_v2.ipynb`,
> cell 21 (`§8 — WRITE M05 MANIFEST + CLOSE`). Paste it in here.

In [ ]:
# =============================================================================
# MODULE 05 -- Senescence Enrichment & Cell Cycle Analysis
# §8 -- WRITE M05 MANIFEST + CLOSE
# =============================================================================
# Closes M05 for one (CELL_TYPE, DATASET) run. Writes m05_manifest.json with:
#   - schema_version (M05 manifest format)
#   - parent_manifest pointer (M04 manifest)
#   - run params (TISSUE, STUDY_TYPE, DISEASE, DATASET, CELL_TYPE)
#   - statistical params and effect configs
#   - input gene lists (Sloan, SenMayo, Fridman) with file sizes
#   - results inventory (CSVs with row counts)
#   - figures inventory (per file size + format)
#   - scored .qs reference (parent: §3 output)
#   - run summary statistics (sig counts per analysis)
#   - timestamp + R/package versions
#
# After this cell, M05 for this CELL_TYPE is complete. Re-run §0 with
# CELL_TYPE = "OPC" to start the next cell type.
# =============================================================================

cat("=", strrep("=", 71), "\n", sep = "")
cat(sprintf("§8 -- WRITE M05 MANIFEST  |  %s / %s\n", DATASET, CELL_TYPE))
cat("=", strrep("=", 71), "\n", sep = "")


# ─────────────────────────────────────────────────────────────────────────────
# §8.1 INVENTORY -- enumerate all output artifacts
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> INVENTORY\n")

# Results CSVs
result_files <- list.files(PATHS$results, pattern = "\\.csv$", full.names = TRUE)
results_inventory <- list()
for (f in result_files) {
    nm    <- basename(f)
    sz    <- file.size(f)
    nrows <- tryCatch({
        # Count rows quickly without loading full df
        length(readLines(f, warn = FALSE)) - 1L
    }, error = function(e) NA_integer_)
    results_inventory[[nm]] <- list(
        filename   = nm,
        bytes      = bytes_str(sz),
        size_human = fmt_size(f),
        n_rows     = nrows
    )
}
cat(sprintf("  Results: %d CSV files\n", length(results_inventory)))
for (nm in sort(names(results_inventory))) {
    r <- results_inventory[[nm]]
    cat(sprintf("    %-50s  %5s rows  (%s)\n",
                nm,
                ifelse(is.na(r$n_rows), "?", as.character(r$n_rows)),
                r$size_human))
}

# Figures (count by slug, list formats)
figure_files <- list.files(PATHS$figures, pattern = "\\.(pdf|png|svg)$",
                          full.names = TRUE)
fig_slugs    <- unique(tools::file_path_sans_ext(basename(figure_files)))

figures_inventory <- list()
for (slug in fig_slugs) {
    matched <- figure_files[tools::file_path_sans_ext(basename(figure_files)) == slug]
    formats <- tools::file_ext(basename(matched))
    sizes   <- vapply(matched, fmt_size, character(1))
    figures_inventory[[slug]] <- list(
        slug    = slug,
        formats = formats,
        sizes   = setNames(sizes, formats)
    )
}
cat(sprintf("\n  Figures: %d unique slugs (%d total files across formats)\n",
            length(figures_inventory), length(figure_files)))
for (slug in sort(names(figures_inventory))) {
    fmts <- figures_inventory[[slug]]$formats
    cat(sprintf("    %-58s  [%s]\n",
                slug, paste(sort(fmts), collapse = ", ")))
}


# ─────────────────────────────────────────────────────────────────────────────
# §8.2 SUMMARY STATISTICS -- key result counts per analysis
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> SUMMARY STATISTICS\n")

count_sig <- function(df_name) {
    if (!exists(df_name)) return(list(rows = NA, sig = NA, neg_sig = NA, pos_sig = NA))
    df <- get(df_name)
    if (nrow(df) == 0) return(list(rows = 0, sig = 0, neg_sig = 0, pos_sig = 0))
    sig_flag <- !is.na(df$p_adj) & df$p_adj < STATISTICAL_PARAMS$fdr_threshold
    list(
        rows    = nrow(df),
        sig     = sum(sig_flag),
        neg_sig = sum(sig_flag & df$estimate < 0, na.rm = TRUE),
        pos_sig = sum(sig_flag & df$estimate > 0, na.rm = TRUE)
    )
}

analyses <- list(
    cellcycle_wilcoxon       = "cc_wilcox_df",
    cellcycle_rlm            = "cc_rlm_df",
    cellcycle_ols            = "cc_ols_df",
    cellcycle_lmm            = "cc_lmm_df",
    cellcycle_bootrlm        = "cc_boot_df",
    modulescores_wilcoxon    = "mod_wilcox_df",
    modulescores_rlm         = "mod_rlm_df",
    modulescores_ols         = "mod_ols_df",
    modulescores_lmm         = "mod_lmm_df",
    modulescores_bootrlm     = "mod_boot_df",
    proliferation_wilcoxon   = "prol_wilcox_df",
    proliferation_rlm        = "prol_rlm_df"
)

cat(sprintf("\n  %-30s %5s %5s %5s %5s\n",
            "Analysis", "rows", "sig", "neg", "pos"))
cat("  ", strrep("-", 60), "\n", sep = "")

analysis_summary <- list()
for (nm in names(analyses)) {
    cs <- count_sig(analyses[[nm]])
    analysis_summary[[nm]] <- cs
    cat(sprintf("  %-30s %5s %5s %5s %5s\n",
                nm,
                ifelse(is.na(cs$rows),    "--", cs$rows),
                ifelse(is.na(cs$sig),     "--", cs$sig),
                ifelse(is.na(cs$neg_sig), "--", cs$neg_sig),
                ifelse(is.na(cs$pos_sig), "--", cs$pos_sig)))
}


# ─────────────────────────────────────────────────────────────────────────────
# §8.3 BUILD MANIFEST OBJECT
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> BUILD MANIFEST\n")

# Read M04 parent manifest if available (for parent-pointer)
parent_manifest_data <- NULL
if (file.exists(PATHS$m04_manifest)) {
    parent_manifest_data <- tryCatch(
        fromJSON(PATHS$m04_manifest),
        error = function(e) NULL
    )
}

# Stratification info
stratification_info <- list(
    enabled               = STRATIFY_BY_GROUP,
    stratification_groups = if (STRATIFY_BY_GROUP) STRATIFICATION_GROUPS else NULL,
    strata_run            = strata,
    n_strata              = length(strata)
)

# Gene-list provenance
gene_lists_info <- list(
    sloan_xlsx   = list(
        path       = PATHS$sloan_xlsx,
        size_human = fmt_size(PATHS$sloan_xlsx),
        n_lists    = length(SLOAN_HALLMARK_NAMES),
        list_names = SLOAN_HALLMARK_NAMES,
        list_types = SLOAN_LIST_TYPES
    ),
    senmayo_xlsx = list(
        path       = PATHS$senmayo_xlsx,
        size_human = fmt_size(PATHS$senmayo_xlsx)
    ),
    fridman_gmt  = list(
        path       = PATHS$fridman_gmt,
        size_human = fmt_size(PATHS$fridman_gmt)
    )
)

# Proliferation markers
proliferation_info <- list(
    requested = PROLIFERATION_MARKERS,
    found     = if (exists("markers_found")) markers_found else NA,
    missing   = if (exists("markers_missing")) markers_missing else NA
)

# Scored .qs reference
scored_info <- list(
    path       = PATHS$scored_qs,
    exists     = file.exists(PATHS$scored_qs),
    size_human = if (file.exists(PATHS$scored_qs)) fmt_size(PATHS$scored_qs) else NA
)

# Cell counts (if md is in scope)
cell_counts <- if (exists("md")) {
    list(
        n_cells_total  = nrow(md),
        n_donors_total = length(unique(md[[DONOR_COL]]))
    )
} else {
    list(n_cells_total = NA, n_donors_total = NA)
}

# Strata-specific paired donor counts
paired_donor_counts <- list()
if (exists("md")) {
    for (s in strata) {
        md_s   <- filter_to_stratum(md, s, STUDY_GROUP_COL)
        arms_s <- build_donor_arms(md_s, DONOR_COL, SEN_LABEL_COL,
                                   STATISTICAL_PARAMS$min_cells_per_group)
        paired_donor_counts[[s]] <- list(
            n_paired_donors = length(unique(arms_s$donor[arms_s$paired])),
            n_cells_in_stratum = nrow(md_s)
        )
    }
}

manifest <- list(
    schema_version  = "m05.1.0",
    module          = "M05",
    module_name     = "Senescence Enrichment & Cell Cycle Analysis",
    timestamp       = now_iso(),

    run_params = list(
        tissue        = TISSUE,
        study_type    = STUDY_TYPE,
        disease       = if (IS_DISEASE) DISEASE else NA,
        dataset       = DATASET,
        cell_type     = CELL_TYPE,
        condition_tag = CONDITION_TAG
    ),

    parent_manifest = list(
        path        = PATHS$m04_manifest,
        schema_version = if (!is.null(parent_manifest_data) &&
                            !is.null(parent_manifest_data$schema_version))
                        parent_manifest_data$schema_version else NA,
        cell_type   = if (!is.null(parent_manifest_data) &&
                         !is.null(parent_manifest_data$cell_type))
                     parent_manifest_data$cell_type else NA
    ),

    stratification    = stratification_info,
    cell_counts       = cell_counts,
    paired_donor_counts = paired_donor_counts,
    statistical_params = STATISTICAL_PARAMS,

    effect_configs = list(
        names_used = names(EFFECT_CONFIGS),
        labels     = sapply(EFFECT_CONFIGS, function(c) c$x_label)
    ),

    gene_lists           = gene_lists_info,
    proliferation_markers = proliferation_info,
    scored_qs            = scored_info,

    analyses_run    = names(analyses),
    analysis_summary = analysis_summary,

    results_inventory = results_inventory,
    figures_inventory = lapply(figures_inventory, function(x) {
        list(slug = x$slug, formats = x$formats)
    }),

    paths = list(
        results_dir = PATHS$results,
        figures_dir = PATHS$figures,
        data_dir    = PATHS$data,
        logs_dir    = PATHS$logs
    ),

    package_versions = R_PKG_VERSIONS
)

cat(sprintf("  Manifest object built: %d top-level keys\n", length(manifest)))


# ─────────────────────────────────────────────────────────────────────────────
# §8.4 WRITE MANIFEST
# ─────────────────────────────────────────────────────────────────────────────
cat("\n>> WRITE MANIFEST\n")

manifest_json <- toJSON(manifest, pretty = TRUE, auto_unbox = TRUE,
                       null = "null", na = "string")
writeLines(manifest_json, PATHS$manifest)

cat(sprintf("  ✓ Wrote %s  (%s)\n",
            basename(PATHS$manifest), fmt_size(PATHS$manifest)))
cat(sprintf("    Path: %s\n", PATHS$manifest))


# ─────────────────────────────────────────────────────────────────────────────
# §8.5 FINAL SUMMARY BANNER
# ─────────────────────────────────────────────────────────────────────────────
cat("\n", strrep("=", 72), "\n", sep = "")
cat(sprintf("  M05 COMPLETE  |  %s / %s\n", DATASET, CELL_TYPE))
cat(strrep("=", 72), "\n", sep = "")
cat(sprintf("  Output root        : %s\n", PATHS$output_root))
cat(sprintf("  Cells x donors     : %s x %d\n",
            ifelse(is.na(cell_counts$n_cells_total), "--",
                   fmt_n(cell_counts$n_cells_total)),
            ifelse(is.na(cell_counts$n_donors_total), -1L,
                   cell_counts$n_donors_total)))
cat(sprintf("  Strata             : %d (%s)\n",
            length(strata), paste(strata, collapse = ", ")))
cat(sprintf("  Analyses run       : %d\n", length(analyses)))
cat(sprintf("  Result CSVs        : %d\n", length(results_inventory)))
cat(sprintf("  Figure slugs       : %d (%d files total)\n",
            length(figures_inventory), length(figure_files)))
cat(sprintf("  Scored .qs         : %s\n",
            ifelse(scored_info$exists, scored_info$size_human, "MISSING")))
cat(sprintf("  Manifest           : %s\n", basename(PATHS$manifest)))
cat(sprintf("  Timestamp          : %s\n", manifest$timestamp))

# Headline biology summary -- the key narrative for manuscript / next steps
cat(sprintf("\n  Headline findings (BH-FDR < %.2f):\n",
            STATISTICAL_PARAMS$fdr_threshold))
total_cc_sig    <- sum(sapply(analysis_summary[grep("^cellcycle_",       names(analysis_summary))],
                              function(x) ifelse(is.na(x$sig), 0, x$sig)))
total_mod_sig   <- sum(sapply(analysis_summary[grep("^modulescores_",    names(analysis_summary))],
                              function(x) ifelse(is.na(x$sig), 0, x$sig)))
total_prol_sig  <- sum(sapply(analysis_summary[grep("^proliferation_",   names(analysis_summary))],
                              function(x) ifelse(is.na(x$sig), 0, x$sig)))

cat(sprintf("    Cell cycle (5 models x 9 outcomes = 45)         : %d sig\n",
            total_cc_sig))
cat(sprintf("    Module scores (5 models x 30 outcomes = 150)    : %d sig\n",
            total_mod_sig))
cat(sprintf("    Proliferation (2 models x 24 outcomes = 48)     : %d sig\n",
            total_prol_sig))

cat(strrep("=", 72), "\n", sep = "")
cat("\nv §8 manifest complete. M05 closed for this cell type.\n")
cat("\n  Next steps:\n")
cat("    1. Re-run §0 with CELL_TYPE = \"OPC\"          (~30-60 min)\n")
cat("    2. Re-run §0 with CELL_TYPE = \"Microglia\"    (~30-60 min)\n")
cat("    3. Return to §6 for cross-cell-type viz\n")
cat("\n  Or: switch to other manuscript work (M06 DGE update, ERAS,\n")
cat("       Step 3 prep, etc.) -- Astrocyte M05 is fully self-contained.\n")